# Ablation_HDFS_Full_Monty — prepare + train / eval

Self-contained notebook for campaign `hdfs-full-monty-v1`. It inlines Family A **prepare** (Drain, Deepseek, MiniLM, collapsed graphs) and **train/eval** (AttributeAwareGAE). No `git clone`, no `run_ablation.py`, no `src.modules` imports.

Protocol (identical across arms):

1. Fit Drain on the **full** unlabeled HDFS log, then TF-IDF on **all** templates (`fit_on=all`).
2. Group lines by `block_id`, then take a **stratified** 70/15/15 split (`seed=42`).
3. Clean-train the same GAE (`y==0` only, GINE sum, concat fusion, α=β=γ=1, 25 epochs, HDFS lr **0.01**).

Upload `HDFS_full.log` and `anomaly_label.csv` to Drive `hybrid-log-analyzer-artifacts/data/raw/hdfs/`. Colab secrets: `AZURE_OPENAI_API_KEY`, `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO`.

On Colab, Drain/Deepseek artefacts under `campaigns/hdfs-full-monty-v1/_prepare/` are **pulled from Drive if they already exist** and **pushed back as soon as they are written**. Each arm's graphs and each seed's `outputs/hdfs/...` pack (metrics, confusion matrix, learning curve, PR curve, score distribution with the validation threshold) are copied to Drive when that variant finishes — do not wait for the final cell. Train still uses `/content/workspace`, not the Drive mount.

Set `RUN_PREPARE=False` if `campaigns/hdfs-full-monty-v1/` already exists. Do **not** mix a local ST 2.2.2 prepare with a Colab ST 5.x prepare. `SMOKE=True` caps prepare/train at 5000 graphs and 1 epoch — do not publish those metrics. Rank by **test PR-AUC** vs `hybrid_llm` (chance ≈ HDFS positive rate, ~0.03).


In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")


In [ ]:
from pathlib import Path
import os
import shutil
import sys

CAMPAIGN_ID = "hdfs-full-monty-v1"
DATASET = "hdfs"
RUN_PREPARE = True
RUN_TRAIN = True
SMOKE = True  # pipeline check; set False for the published 40 GAE + 5 IF runs
SEEDS = [13, 29, 42, 71, 101]
SPLIT_SEED = 42
SMOKE_GRAPH_CAP = 5000
EXPECTED_SPLIT_LOCK_ID = None  # optional pin after the first published prepare
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/hybrid-log-analyzer-artifacts")
AZURE_SECRET_KEYS = (
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO",
)
PREPARE_DRIVE_FILES = (
    "templates.json",
    "drain_parser.bin",
    "drain.ini",
    "annotated.parquet",
    "sequences.parquet",
    "graph_structure.pkl",
    "enrichment_provenance.json",
)


def find_local_workspace() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "notebooks").exists() and (candidate / "run_ablation.py").exists():
            return candidate
        if (candidate / "campaigns" / CAMPAIGN_ID / "manifest.json").exists():
            return candidate
    return here


WORKSPACE_ROOT = Path("/content/workspace") if IN_COLAB else find_local_workspace()
CAMPAIGN_DIR = WORKSPACE_ROOT / "campaigns" / CAMPAIGN_ID


def _copy_tree(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_file():
        shutil.copy2(source, destination)
        return
    shutil.copytree(source, destination, dirs_exist_ok=True)


def local_prepare_dir() -> Path:
    return Path(CAMPAIGN_DIR) / "_prepare"


def drive_prepare_dir() -> Path:
    return DRIVE_ARTIFACT_ROOT / "campaigns" / CAMPAIGN_ID / "_prepare"


def _copy_file_if_needed(source: Path, dest: Path) -> bool:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size == source.stat().st_size and dest.stat().st_mtime >= source.stat().st_mtime:
        return False
    shutil.copy2(source, dest)
    return True


def push_to_drive(local: Path) -> Path | None:
    """Copy a workspace file or directory onto Drive. No-op off Colab."""
    local = Path(local)
    if not IN_COLAB or not local.exists():
        return None
    try:
        relative = local.resolve().relative_to(Path(WORKSPACE_ROOT).resolve())
    except ValueError:
        print(f"[DRIVE] skip (outside workspace): {local}")
        return None
    dest = DRIVE_ARTIFACT_ROOT / relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    copied = 0
    if local.is_file():
        copied += int(_copy_file_if_needed(local, dest))
    else:
        dest.mkdir(parents=True, exist_ok=True)
        for path in local.rglob("*"):
            if path.is_file():
                copied += int(_copy_file_if_needed(path, dest / path.relative_to(local)))
    print(f"[DRIVE] saved {relative} → {dest} ({copied} file(s) copied)")
    return dest


def stage_drive_prepare_artifacts() -> None:
    """Reuse Drain + Deepseek files from Drive when the local copies are missing."""
    if not IN_COLAB:
        return
    dest_dir = local_prepare_dir()
    dest_dir.mkdir(parents=True, exist_ok=True)
    source_dir = drive_prepare_dir()
    if not source_dir.exists():
        print(f"[DRIVE] no parser/enrichment at {source_dir} (will build on first prepare)")
        return
    for name in PREPARE_DRIVE_FILES:
        source = source_dir / name
        dest = dest_dir / name
        if source.exists() and not dest.exists():
            shutil.copy2(source, dest)
            print(f"[DRIVE] reuse {name}")
        elif dest.exists():
            print(f"[DRIVE] local {name} already present")
        else:
            print(f"[DRIVE] missing {name} (will build if needed)")
    templates = dest_dir / "templates.json"
    if templates.exists():
        try:
            import json as _json

            records = _json.loads(templates.read_text())
            real = [item for item in records if int(item.get("cluster_id", 0)) >= 0]
            enriched = sum(1 for item in real if isinstance(item.get("enriched_large"), dict))
            print(f"[DRIVE] templates.json: {len(real)} clusters, {enriched} with enriched_large")
        except Exception as exc:
            print(f"[DRIVE] could not inspect templates.json ({exc})")


def push_prepare_artifacts(*, reason: str) -> None:
    if not IN_COLAB:
        return
    local = local_prepare_dir()
    if not local.exists():
        return
    dest = drive_prepare_dir()
    dest.mkdir(parents=True, exist_ok=True)
    for path in sorted(local.iterdir()):
        if path.is_file():
            shutil.copy2(path, dest / path.name)
    print(f"[DRIVE] saved _prepare ({reason}) → {dest}")


def push_campaign_arm(arm: str) -> None:
    push_to_drive(Path(CAMPAIGN_DIR) / "graphs" / arm)
    for name in ("manifest.json", "split_lock.npz"):
        push_to_drive(Path(CAMPAIGN_DIR) / name)


def push_run_outputs(output_dir: Path) -> None:
    push_to_drive(Path(output_dir))


def stage_colab_workspace() -> None:
    """Copy raw logs, frozen parser/enrichment, graphs, and prior outputs from Drive."""
    if not IN_COLAB:
        return
    drive_raw = DRIVE_ARTIFACT_ROOT / "data" / "raw" / "hdfs"
    local_raw = WORKSPACE_ROOT / "data" / "raw" / "hdfs"
    if drive_raw.exists():
        print(f"[DRIVE] staging raw HDFS {drive_raw} → {local_raw}")
        _copy_tree(drive_raw, local_raw)
    else:
        print(f"[DRIVE] raw HDFS not found at {drive_raw}")
    stage_drive_prepare_artifacts()
    drive_campaign = DRIVE_ARTIFACT_ROOT / "campaigns" / CAMPAIGN_ID
    Path(CAMPAIGN_DIR).mkdir(parents=True, exist_ok=True)
    for name in ("split_lock.npz", "manifest.json"):
        source = drive_campaign / name
        dest = Path(CAMPAIGN_DIR) / name
        if source.exists() and not dest.exists():
            shutil.copy2(source, dest)
            print(f"[DRIVE] reuse {name}")
    drive_graphs = drive_campaign / "graphs"
    if drive_graphs.exists():
        for arm_dir in sorted(p for p in drive_graphs.iterdir() if p.is_dir()):
            dest = Path(CAMPAIGN_DIR) / "graphs" / arm_dir.name
            marker = dest / "graph_dataset.pt.gz"
            if marker.exists():
                print(f"[DRIVE] local graphs/{arm_dir.name} already present")
                continue
            print(f"[DRIVE] staging graphs/{arm_dir.name}")
            _copy_tree(arm_dir, dest)
    drive_outputs = DRIVE_ARTIFACT_ROOT / "outputs" / DATASET
    local_outputs = WORKSPACE_ROOT / "outputs" / DATASET
    if drive_outputs.exists():
        print(f"[DRIVE] staging prior outputs {drive_outputs} → {local_outputs}")
        _copy_tree(drive_outputs, local_outputs)


print(f"WORKSPACE_ROOT = {WORKSPACE_ROOT}")
print(f"CAMPAIGN_DIR   = {CAMPAIGN_DIR}")
print(f"RUN_PREPARE    = {RUN_PREPARE}  RUN_TRAIN={RUN_TRAIN}  SMOKE={SMOKE}")
print(f"SEEDS          = {SEEDS}  SPLIT_SEED={SPLIT_SEED}")


## Colab environment

Install **torch-geometric** against Colab's CUDA PyTorch, plus Drain / MiniLM / Azure deps needed to prepare graphs. Local runs use the existing `.venv` (sentence-transformers 2.2.2). Never mix those two MiniLM versions inside one campaign id.

The setup cell mounts Drive and **reuses** `_prepare/templates.json`, `drain_parser.bin`, and `enriched_large` when they are already on Drive. Missing parser/enrichment files are built once, then copied to Drive immediately.


In [ ]:
def pyg_wheel_url() -> str:
    import torch

    torch_version = torch.__version__.split("+", maxsplit=1)[0]
    cuda_version = torch.version.cuda
    platform = f"cu{str(cuda_version).replace('.', '')}" if cuda_version else "cpu"
    return f"https://data.pyg.org/whl/torch-{torch_version}+{platform}.html"


def _copy_tree(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_file():
        shutil.copy2(source, destination)
        return
    shutil.copytree(source, destination, dirs_exist_ok=True)


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)

    import torch

    print(f"Colab PyTorch {torch.__version__}  cuda={torch.cuda.is_available()}")
    wheel_url = pyg_wheel_url()
    print(f"PyG wheel index: {wheel_url}")
    get_ipython().run_line_magic(
        "pip",
        "install -q drain3 networkx langchain-openai langchain-core pydantic python-dotenv "
        "pyarrow fastparquet pandas scikit-learn sentence-transformers tqdm pyyaml matplotlib",
    )
    get_ipython().run_line_magic("pip", f"install -q torch-geometric -f {wheel_url}")
    try:
        get_ipython().run_line_magic(
            "pip",
            f"install -q --only-binary=:all: pyg_lib torch_scatter torch_sparse -f {wheel_url}",
        )
    except Exception as exc:
        print(f"Optional PyG extension wheels unavailable ({exc}); using PyTorch fallbacks.")

    import torch_geometric

    print(f"torch-geometric {torch_geometric.__version__}")
    stage_colab_workspace()
else:
    import torch
    import torch_geometric

    mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    print(f"Local PyTorch {torch.__version__}  pyg={torch_geometric.__version__}")
    print(f"cuda={torch.cuda.is_available()}  mps={mps}")
    local_raw = WORKSPACE_ROOT / "data" / "raw" / "hdfs"
    print(f"Raw HDFS dir: {local_raw}  exists={local_raw.exists()}")


## Inlined prepare library

DrainParser, Deepseek enricher, HDFS block sequencer, collapsed-graph structures, TF-IDF (fit on all templates), MiniLM, split lock, and per-arm splice. Current implementations pasted in; no repo imports.


In [ ]:
"""Inlined Family A prepare stack (Drain, enrich, sequence, graphs)."""

from __future__ import annotations



import gzip

import hashlib

import json

import logging

import os

import pickle

import re

import shutil

import time

from collections import Counter, defaultdict

from collections.abc import Callable, Mapping, Sequence

from datetime import datetime, timezone

from importlib.util import find_spec

from pathlib import Path

from typing import Any, Literal, Optional, Dict, List



import numpy as np

import pandas as pd

import networkx as nx

from pydantic import BaseModel, Field, field_validator

from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate

from langchain_openai import ChatOpenAI



logger = logging.getLogger(__name__)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')



DRAIN_INI_TEXT = '[PROFILING]\nenabled = False\nreport_sec = 60\n\n[SNAPSHOT]\nsnapshot_interval_minutes = 5\ncompress_state = False\n\n[DRAIN]\nsim_th = 0.4\ndepth = 4\nmax_children = 100\nmax_clusters = 1000\nextra_delimiters = []\n\n[MASKING]\nmasking = [\n  {"regex_pattern": "blk_-?\\\\d+", "mask_with": "BLK"},\n  {"regex_pattern": "(?:\\\\d{1,3}\\\\.){3}\\\\d{1,3}(?::\\\\d+)?", "mask_with": "IP"},\n  {"regex_pattern": "[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}", "mask_with": "UUID"},\n  {"regex_pattern": "\\\\b0x[0-9a-fA-F]+\\\\b", "mask_with": "HEX"}\n  ]\nmask_prefix = <\nmask_suffix = >\n'

PROMPT_FILE_SHA256 = {'__init__.py': '3b023de2df5bb51171f410424b07f68bde66158d3226bdd17e10281a46b88e8e', 'bgl_prompts.py': '9ff0422ca9a7b9310c29ea9f055db860b6449d093e5b8f19e3203ba326bd4633', 'generic_prompts.py': '9f92e69124895aac0971ce2cecc9d375d3ec715e13927b484a3d7768b3817157', 'hdfs_prompts.py': '7b48beda854a3d9ec7cab1b8e8e087979fa2f655c4e7f10e6567a12835318cd9', 'system.py': 'd39d74dc6a7a35ff651c48d04164c719b5ce526bc3c8e0b28c1deb3a7b33727f'}



from typing import List, Dict, Any, Optional, Callable
from collections import defaultdict
from pathlib import Path
import json
import re

from drain3 import TemplateMiner
from drain3.file_persistence import FilePersistence
from drain3.template_miner_config import TemplateMinerConfig


class UnmatchedLogLine(ValueError):
  """Raised when strict annotation encounters a line with no frozen template."""

  def __init__(self, line_number: int) -> None:
    self.line_number = line_number
    super().__init__(f"Log line {line_number} did not match a frozen template.")


OOV_CLUSTER_ID = -1
OOV_TEMPLATE = "<*>"


class DrainParser:
  """
  Wrapper around Drain3's TemplateMiner for:
    - training on a log file
    - exporting learned templates
    - validating template quality heuristically

  Default implementation targets the HDFS log format. Dataset-specific
  subclasses (e.g. :class:`BGLParser`) can override:
    - the class attribute ``_HEADER_TOKENS`` (how many prefix tokens to strip)
    - the hook method :meth:`_extract_row` (what columns to emit per line)
  """

  # Number of leading whitespace-separated tokens to strip before passing
  # the line to Drain.
  # HDFS format: <DDMMYY> <HHMMSS> <thread_id> <LEVEL> <component>: <message>
  # Stripping the first 3 (date, time, thread) removes always-variable header
  # fields; LEVEL and component are kept because they help Drain cluster lines.
  _HEADER_TOKENS = 3
  _HDFS_BLOCK_ID_RE = re.compile(r"blk_-?\d+")

  @staticmethod
  def _preprocess_line(line: str, strip_tokens: int = 3) -> str:
    """Strip the first `strip_tokens` whitespace-separated tokens from *line*.

    This removes the variable header (e.g. date/time/thread for HDFS,
    label/timestamp/node for BGL) so Drain only sees the stable
    LEVEL + COMPONENT + MESSAGE portion.
    Returns the original line unchanged if it has fewer tokens than requested.
    """
    parts = line.split(None, strip_tokens)
    return parts[strip_tokens] if len(parts) > strip_tokens else line

  @classmethod
  def extract_hdfs_block_id(cls, line: str) -> str | None:
    """Return the first HDFS block ID using the parser's sequence identity rule."""
    block_match = cls._HDFS_BLOCK_ID_RE.search(line)
    return block_match.group(0) if block_match else None

  def __init__(self, config_path: str | None = None, persistence_path: Optional[str] = None):
    cfg = TemplateMinerConfig()
    if config_path:
      cfg.load(config_path)

    persistence = None
    if persistence_path:
      Path(persistence_path).parent.mkdir(parents=True, exist_ok=True)
      persistence = FilePersistence(persistence_path)

    self.miner = TemplateMiner(persistence_handler=persistence, config=cfg)
    self._persistence_path = persistence_path
    self._config_path = config_path

    # cluster_id is stable throughout the online parsing process;
    # template strings evolve as tokens are replaced with <*>, so we
    # key example lines by ID to avoid orphaned entries.
    self.cluster_id_to_lines: Dict[int, List[str]] = defaultdict(list)
    self.line_template_ids: List[str] = []


  def fit_file(
    self,
    log_path: str,
    max_lines: int | None = None,
    *,
    include_line: Callable[[str], bool] | None = None,
  ) -> None:
    log_file = Path(log_path)
    print(f"[INFO] Training Drain3 on: {log_file}")
    n_skipped = 0

    with log_file.open("r", errors="replace") as f:
      for i, raw in enumerate(f, start=1):
        line = raw.rstrip("\n")
        if not line:
          continue
        if include_line is not None and not include_line(line):
          n_skipped += 1
          if max_lines is not None and i >= max_lines:
            print(f"[INFO] Stopped early at {max_lines} lines")
            break
          continue

        content = self._preprocess_line(line, self._HEADER_TOKENS)
        result = self.miner.add_log_message(content) # Process the log line through Drain3
        cluster_id = result.get("cluster_id")

        if cluster_id is None:
          print(f"[WARN] No cluster for line {i}: {line[:120]}...")
          continue

        # Store for validation — keyed by stable cluster_id, not by the
        # template string which may still evolve after this point.
        self.line_template_ids.append(cluster_id)
        self.cluster_id_to_lines[cluster_id].append(line)

        # Early stopping
        if max_lines is not None and i >= max_lines:
          print(f"[INFO] Stopped early at {max_lines} lines")
          break

        if i % 100000 == 0:
          print(f"[INFO] Processed {i} lines...")

      print(f"[INFO] Parsed {len(self.line_template_ids)} log lines total.")
      if n_skipped:
        print(f"[INFO] Skipped {n_skipped} lines (held-out split; not used to grow Drain).")
      print(f"[INFO] Learned {len(self.miner.drain.id_to_cluster)} distinct templates (final).")


  def _extract_row(
    self,
    line: str,
    cluster_id: int,
    template_str: str,
    params: List[Any],
  ) -> Dict[str, Any]:
    """Hook: build one output row from a matched log line.

    Default implementation produces HDFS-specific columns:
      - date, time, thread   : raw header fields
      - timestamp            : datetime parsed from date+time
      - raw                  : full original log line
      - cluster_id           : stable Drain cluster id
      - template             : final (fully generalised) template string
      - parameters           : list of values extracted for each <*> token
      - block_id             : first blk_XXX value found in the line (or None)

    Subclasses targeting other log formats (e.g. :class:`BGLParser`)
    should override this method to return an alternative column dict.
    """
    from datetime import datetime

    parts = line.split(None, 3)  # date time thread rest
    date_s, time_s, thread_s = (parts + ["", "", ""])[:3]

    try:
      ts = datetime.strptime(f"{date_s} {time_s}", "%d%m%y %H%M%S")
    except ValueError:
      ts = None

    block_id = self.extract_hdfs_block_id(line)

    return {
      "date":       date_s,
      "time":       time_s,
      "thread":     thread_s,
      "timestamp":  ts,
      "raw":        line,
      "cluster_id": cluster_id,
      "template":   template_str,
      "parameters": list(params) if params else [],
      "block_id":   block_id,
    }


  def annotate_file(
    self,
    log_path: str,
    max_lines: int | None = None,
    *,
    unmatched: str = "skip",
  ):
    """Second pass over the log file using the final learned templates.

    Returns a pandas DataFrame with one row per log line. Column schema is
    determined by :meth:`_extract_row` (overridable by subclasses).
    ``unmatched="fail"`` raises :class:`UnmatchedLogLine`.
    ``unmatched="oov"`` assigns :data:`OOV_CLUSTER_ID` (never seen in a
    train-only Drain fit).
    """
    if unmatched not in {"skip", "fail", "oov"}:
      raise ValueError("unmatched must be 'skip', 'fail', or 'oov'.")
    import pandas as pd

    rows = []
    log_file = Path(log_path)
    n_oov = 0

    with log_file.open("r", errors="replace") as f:
      for i, raw in enumerate(f, start=1):
        line = raw.rstrip("\n")
        if not line:
          continue

        # Strip the same header tokens that were removed during fit
        content = self._preprocess_line(line, self._HEADER_TOKENS)

        # Match against final templates (does not mutate clusters)
        match = self.miner.match(content)
        if match is None:
          if unmatched == "fail":
            raise UnmatchedLogLine(i)
          if unmatched == "skip":
            continue
          row = self._extract_row(line, OOV_CLUSTER_ID, OOV_TEMPLATE, [])
          row["line_number"] = i
          rows.append(row)
          n_oov += 1
          if max_lines is not None and i >= max_lines:
            break
          continue

        template_tokens = match.get_template()  # list[str]
        template_str    = " ".join(template_tokens)
        params          = self.miner.get_parameter_list(template_tokens, content)

        row = self._extract_row(line, match.cluster_id, template_str, list(params) if params else [])
        row["line_number"] = i
        rows.append(row)

        if max_lines is not None and i >= max_lines:
          break

        if i % 500_000 == 0:
          print(f"[INFO] Annotated {i} lines...")

    print(f"[INFO] Annotated {len(rows)} lines.")
    if n_oov:
      print(f"[INFO] Assigned OOV cluster {OOV_CLUSTER_ID} to {n_oov} unmatched lines.")
    frame = pd.DataFrame(rows)
    frame.attrs["n_oov"] = n_oov
    return frame


  def export_templates(self, out_path: str) -> None:
    """Export learned templates to a file."""
    records = []
    for cluster in self.miner.drain.id_to_cluster.values():
      tmpl = " ".join(cluster.log_template_tokens)
      lines = self.cluster_id_to_lines.get(cluster.cluster_id, [])
      examples = [self._template_example(line) for line in lines[:5]]
      records.append({
          "cluster_id": cluster.cluster_id, # Stable ID for this template cluster
          "template": tmpl,                 # The template string learned by Drain3
          "count": cluster.size,            # authoritative count from Drain3
          "examples": [example for example in examples if example]
      })

    output_path = Path(out_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w") as f:
      json.dump(records, f, indent=2)

    print(f"[INFO] Exported {len(records)} templates → {output_path}")


  @staticmethod
  def _template_example(line: str) -> str:
    """Return an example safe to expose as parser/template context.

    Dataset-specific parsers may remove benchmark-only target fields before
    examples are written to the template artifact and later passed to an LLM.
    """
    return line


  def save(self, path: Optional[str] = None) -> None:
    """Manually trigger a drain3 snapshot save.

    If *path* is given and differs from the current persistence path, a new
    FilePersistence pointing at that path is used for this one-off save.
    Otherwise the persistence handler configured at construction time is used.
    """
    target_path = path or self._persistence_path
    if target_path is None:
      raise ValueError("No persistence_path configured and no path argument given.")

    out = Path(target_path)
    out.parent.mkdir(parents=True, exist_ok=True)

    if target_path != self._persistence_path:
      # One-off save to a different file — swap handler temporarily
      original = self.miner.persistence_handler
      self.miner.persistence_handler = FilePersistence(target_path)
      self.miner.save_state("manual_save")
      self.miner.persistence_handler = original
    else:
      self.miner.save_state("manual_save")

    print(f"[INFO] DrainParser snapshot saved → {out}  ({out.stat().st_size / 1_048_576:.2f} MB)")

  @classmethod
  def load(cls, path: str, config_path: Optional[str] = None) -> "DrainParser":
    """Restore a previously saved DrainParser from a drain3 FilePersistence snapshot.

    Passing the same *config_path* as during training ensures masking rules and
    Drain hyperparameters are identical — only cluster state is loaded from the
    snapshot file (as per drain3 design).

    Drain3 FilePersistence reconstructs objects with jsonpickle. Production inference
    may call this only on a checksum-bound ``drain_parser.bin`` from a Publisher-admitted
    preprocessing bundle. The public API and isolated validator must not deserialize it.
    """
    if not Path(path).exists():
      raise FileNotFoundError(f"Snapshot file not found: {path}")
    # Constructing with FilePersistence triggers load_state() automatically
    obj = cls(config_path=config_path, persistence_path=path)
    n_clusters = len(obj.miner.drain.id_to_cluster)
    print(f"[INFO] DrainParser loaded ← {path}  ({n_clusters} templates)")
    return obj

  def _final_clusters(self) -> List[tuple]:
    """Return list of (template_str, size) from Drain's authoritative cluster store."""
    return [
      (" ".join(c.log_template_tokens), c.size)
      for c in self.miner.drain.id_to_cluster.values()
    ]


  def _print_template_support_distribution(self) -> List[int]:
    support_counts = sorted(size for _, size in self._final_clusters())
    min_sup = support_counts[0]
    max_sup = support_counts[-1]
    avg_sup = sum(support_counts) / len(support_counts)
    print(
      f"[INFO] Template support (lines per template): "
      f"min={min_sup}, max={max_sup}, avg={avg_sup:.1f}"
    )
    return support_counts


  def _validate_singleton_templates(self, support_counts: List[int], n_templates: int) -> None:
    singletons = sum(1 for s in support_counts if s == 1)
    if singletons / n_templates > 0.3:
      print(
        f"[WARN] High fraction of singleton templates "
        f"({singletons}/{n_templates} ≈ {singletons/n_templates:.1%}). "
        f"Drain might be overfitting (simply memorizing lines)."
      )
    else:
      print("[OK] Singleton template fraction looks reasonable.")


  def _validate_overly_generic_templates(self) -> None:
    too_generic = []
    for tmpl, size in self._final_clusters():
      num_tokens = len(tmpl.split())
      num_wild = tmpl.count("<*>")
      if num_tokens > 0 and num_wild / num_tokens > 0.7:
        too_generic.append((tmpl, size))

    if too_generic:
      print(
        f"[WARN] Found {len(too_generic)} templates that look very generic "
        f"(>70% wildcard tokens). Examples:"
      )
      for tmpl, sup in too_generic[:5]:
        print(f"      support={sup:4d} | template='{tmpl}'")
      if len(too_generic) > 5:
        print("      ...")
    else:
      print("[OK] No overly generic templates detected by wildcard heuristic.")


  def _validate_overly_specific_templates(self) -> None:
    too_specific = []
    for tmpl, size in self._final_clusters():
      if size == 1 and "<*>" not in tmpl:
        too_specific.append(tmpl)

    if too_specific:
      print(
        f"[WARN] Found {len(too_specific)} templates that are "
        f"single-use with no wildcards (likely overfitting)."
      )
      for tmpl in too_specific[:5]:
        print(f"      '{tmpl}'")
      if len(too_specific) > 5:
        print("      ...")
    else:
      print("[OK] No obviously over-specific single-use templates found.")


  def validate(self) -> None:
    """Heuristic validation of learned templates."""
    print("\n[INFO] Running template validation...")

    total_lines = len(self.line_template_ids)
    n_templates = len(self.miner.drain.id_to_cluster)

    if total_lines == 0:
        print("[ERROR] No lines were parsed. Check your log path / format.")
        return

    print(f"[INFO] Total lines parsed   : {total_lines}")
    print(f"[INFO] Distinct templates   : {n_templates}")

    assigned = len(self.line_template_ids)
    if assigned != total_lines:
        print(f"[WARN] Coverage mismatch: {assigned}/{total_lines} lines have a cluster_id.")
    else:
        print("[OK] 100% of lines received a template assignment.")

    support_counts = self._print_template_support_distribution()
    self._validate_singleton_templates(support_counts, n_templates)
    self._validate_overly_generic_templates()
    self._validate_overly_specific_templates()


from typing import Any, Literal, Mapping, Sequence

from pydantic import BaseModel, Field, field_validator


MetadataConfidence = Literal["high", "medium", "low", "unknown"]
MetadataSource = Literal[
    "template",
    "examples",
    "corpus_relation",
    "documentation",
    "unknown",
]
DiagnosticRole = Literal[
    "informational",
    "lifecycle_transition",
    "warning_or_error",
    "context_dependent",
    "unknown",
]
RelationType = Literal[
    "commonly_precedes",
    "commonly_follows",
    "co_occurs_in_trace",
    "same_component_lifecycle",
    "potential_rca_context",
]


class TemplateContext(BaseModel):
    """Format-agnostic evidence supplied to one enrichment call per template."""

    template_id: str
    template: str
    occurrence_count: int = Field(ge=0)
    examples: list[str] = Field(default_factory=list, max_length=5)
    candidate_relations: list[dict[str, Any]] = Field(default_factory=list)
    retrieved_docs: list[dict[str, Any]] = Field(default_factory=list)
    dataset_context: str | None = None

    @field_validator("examples")
    @classmethod
    def remove_empty_examples(cls, examples: list[str]) -> list[str]:
        return [example.strip() for example in examples if example.strip()]

    @classmethod
    def from_template_record(
        cls,
        record: Mapping[str, Any],
        *,
        candidate_relations: Sequence[Mapping[str, Any]] | None = None,
        retrieved_docs: Sequence[Mapping[str, Any]] | None = None,
        dataset_context: str | None = None,
    ) -> "TemplateContext":
        """Create context from a mined-template record without format-specific rules."""
        examples = record.get("examples", [])
        return cls(
            template_id=str(record["cluster_id"]),
            template=str(record["template"]),
            occurrence_count=int(record.get("count", 0)),
            examples=[str(example) for example in examples[:5]],
            candidate_relations=[dict(item) for item in candidate_relations or ()],
            retrieved_docs=[dict(item) for item in retrieved_docs or ()],
            dataset_context=dataset_context,
        )


class TemplateField(BaseModel):
    placeholder: str
    semantic_role: str
    source: MetadataSource = "unknown"
    confidence: MetadataConfidence = "unknown"

    @field_validator("placeholder", "semantic_role", mode="before")
    @classmethod
    def _coerce_required_str(cls, value: Any) -> str:
        if value is None:
            return "unknown"
        text = str(value).strip()
        return text or "unknown"


class TemplateRelation(BaseModel):
    template_id: str
    relation: RelationType
    support: str
    source: MetadataSource = "corpus_relation"


class FailureSignal(BaseModel):
    name: str
    manifestation: str
    trigger_scope: Literal[
        "explicit_in_template",
        "requires_sequence_context",
        "requires_external_metric",
    ]
    source: MetadataSource = "unknown"
    confidence: MetadataConfidence = "unknown"


def coerce_string_list(value: Any) -> list[str]:
    """Accept a JSON array or a single string the model dumped instead of a list."""
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        return [text] if text else []
    if isinstance(value, list):
        items: list[str] = []
        for item in value:
            if item is None:
                continue
            text = str(item).strip()
            if text:
                items.append(text)
        return items
    text = str(value).strip()
    return [text] if text else []


class EnrichedTemplate(BaseModel):
    """Observed template metadata and semantic enrichment from one LLM response."""

    component: str
    log_level: str
    operation: str
    fields: list[TemplateField] = Field(default_factory=list)
    explicit_conditions: list[str] = Field(default_factory=list)
    metadata_confidence: MetadataConfidence
    component_role: str
    event_semantics: str
    diagnostic_role: DiagnosticRole
    failure_signals: list[FailureSignal] = Field(default_factory=list)
    sequence_context: list[TemplateRelation] = Field(default_factory=list)
    dataset_label_caveat: str
    embedding_text: str
    unsupported_inferences: list[str] = Field(default_factory=list)

    @field_validator("explicit_conditions", "unsupported_inferences", mode="before")
    @classmethod
    def _coerce_string_lists(cls, value: Any) -> list[str]:
        return coerce_string_list(value)


from langchain_core.prompts import SystemMessagePromptTemplate


def get_system_prompt() -> SystemMessagePromptTemplate:
    return SystemMessagePromptTemplate.from_template(
        """
You enrich mined log templates from previously unseen log formats. The input is a JSON
record containing one mined template, representative raw examples, and optional corpus
or dataset context.

Perform both steps internally and return one flat JSON object:
1. Infer observed metadata: component/emitter, log level or priority, operation,
   placeholder roles, and explicit conditions. Use the template first; use examples only
   when they consistently support the value. Set the corresponding confidence to low or
   unknown when examples conflict or evidence is absent.
2. Use that metadata to write concise semantic enrichment and embedding text.

GROUNDING
Use only the supplied JSON record. Do not assume a known log format, product, timestamp
layout, severity convention, component taxonomy, or placeholder meaning. Record claims
that cannot be supported in unsupported_inferences. A value of "unknown" is valid only
when evidence is insufficient; do not replace facts visible in the template or examples
with "unknown".

EVENTS ARE NOT LABELS
Do not classify a template or event as normal or anomalous unless the supplied dataset
context explicitly defines an event-level label. Explain any supplied label granularity
in dataset_label_caveat. If none is supplied, state that no event-level label information
was supplied.

RELATION RULES
- Only return a sequence_context item for a candidate relation supplied in the input.
- Copy its template_id and relation exactly. Never invent transition or lifecycle edges.
- Add a failure signal only when the template, examples, documentation, or supplied
  corpus relation directly supports it. Emit one when FATAL, panic, fail, ERROR, or
  uncorrectable appears in the template or examples. Do not invent follow-on templates
  (for example uncorrectable variants, heartbeats, or reboot events) that are not in
  the evidence.

PLACEHOLDER-HEAVY TEMPLATES
If the template contains two or more <*> tokens, treat it as generic. When the examples
agree on a payload, describe that payload (for example "generating core.<id>"). Put the
generic-template note in event_semantics and embedding_text, not in explicit_conditions.
Do not enumerate other event types that the placeholders could match.

DIAGNOSTIC ROLE
warning_or_error only when the template or examples contain an explicit fault token
such as FATAL, ERROR, fail, panic, uncorrectable, or terminated. INFO plus a recovered
or maintenance action (corrected, bit sparing, detected and corrected) is informational.
lifecycle_transition is for start/stop/mount/init without a fault token.
context_dependent when examples disagree or the template is too generic to choose.
unknown when evidence is insufficient.

OUTPUT
Return only one JSON object, without Markdown fences or a wrapper key. Its top-level keys
must be exactly: component, log_level, operation, fields, explicit_conditions,
metadata_confidence, component_role, event_semantics, diagnostic_role, failure_signals,
sequence_context, dataset_label_caveat, embedding_text, unsupported_inferences.

Each fields item has: placeholder, semantic_role, source, confidence.
Each failure_signals item has: name, manifestation, trigger_scope, source, confidence.
Each sequence_context item has: template_id, relation, support, source.

ENUMERATION RULES
- explicit_conditions is a JSON array of short strings copied from the template or
  examples (for example ["ddr errors detected and corrected"]). Use [] when none are
  explicit. Never return a paragraph or a bare string.
- metadata_confidence is exactly one string: "high", "medium", "low", or "unknown".
  Never return a per-field object there; describe per-field confidence only in fields items.
- diagnostic_role is exactly one string: "informational", "lifecycle_transition",
  "warning_or_error", "context_dependent", or "unknown". Put an explanation in
  event_semantics, not in diagnostic_role.
- fields and failure_signals confidence values use the same four confidence strings.
- failure_signals trigger_scope is exactly one string: "explicit_in_template",
  "requires_sequence_context", or "requires_external_metric". Use
  "explicit_in_template" for a condition observable in one template or example;
  use "requires_sequence_context" only when the signal depends on event order or
  trace context; use "requires_external_metric" only for an external measurement.
- source is exactly one of: "template", "examples", "corpus_relation",
  "documentation", or "unknown".

embedding_text must be two to four short factual sentences by default.
- Reuse distinctive tokens from the template: severity, component, and the specific
  fault or action (for example "instruction cache parity error corrected").
- Do not paraphrase those tokens into generic reliability-subsystem boilerplate.
- Forbidden phrases: "informational message from the reliability subsystem", invented
  product taxonomy, dataset-label language, and sibling templates that are not in
  the evidence.
- Exclude template IDs, source IDs, and unsupported causal explanations.

BGL EXTENDED PERFORMANCE PROFILE
When dataset_context ends with "Enrichment profile: bgl_extended_v1.", write
embedding_text as 6 to 9 information-dense sentences. Preserve all explicit
template facts, then add clearly qualified operational diagnostic context:
plausible failure mechanism, likely immediate trigger, affected hardware or
software scope, potential downstream consequence, and checks an operator could
perform. Use wording such as "may indicate", "can be associated with", or
"a possible cause is" for anything beyond the supplied template/examples.
Do not call those hypotheses observed facts. Retain distinctive message tokens
and avoid benchmark labels, template IDs, and unsupported product-specific
details. Put uncertain causal claims in unsupported_inferences as well.
"""
    )


from langchain_core.prompts import ChatPromptTemplate



TEMPLATE_PROMPT = ChatPromptTemplate.from_messages(
    [
        get_system_prompt(),
        (
            "human",
            """
# Template evidence
{context_json}
""",
        ),
    ]
)


HDFS_DATASET_CONTEXT = (
    "HDFS-v1 labels apply to complete traces grouped by block ID, not to an "
    "individual log event or template."
)

# Backward-compatible prompt export. Dataset facts are supplied through
# TemplateContext rather than hard-coded into the generic prompt.
HDFS_PROMPT_CORPUS = TEMPLATE_PROMPT



class LLM:
    """Azure ChatOpenAI wrapper; credentials are read at construction time."""

    def __init__(self, model: str | None = None) -> None:
        api_key = os.getenv("AZURE_OPENAI_API_KEY")
        endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
        default_model = os.getenv("AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO")
        if not api_key or not endpoint or not (model or default_model):
            raise EnvironmentError(
                "Azure OpenAI credentials missing. Set AZURE_OPENAI_API_KEY, "
                "AZURE_OPENAI_ENDPOINT, and AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO."
            )
        self.client = ChatOpenAI(
            api_key=api_key,
            base_url=endpoint,
            model=model or default_model,
            temperature=0,
            max_tokens=None,
            timeout=None,
            max_retries=2,
        )

    def get_llm(self):
        return self.client


from collections.abc import Mapping, Sequence
import json
from typing import Any



class Enricher:
    def __init__(self, model: str | None = None) -> None:
        llm = LLM(model=model).get_llm()
        # Parse JSON manually because some Azure-hosted models wrap the object
        # in an extra key before Pydantic sees it.
        self.llm = llm

    def enrich_template(self, context: TemplateContext) -> EnrichedTemplate:
        """Infer metadata and semantic enrichment in one LLM call.

        ``TemplateContext`` contains only evidence produced by template mining
        and optional corpus-level observations. It does not encode a log-format
        parser or domain-specific field assumptions.
        """
        chain = TEMPLATE_PROMPT | self.llm
        response = chain.invoke(
            {
                "context_json": json.dumps(
                    context.model_dump(exclude_none=True),
                    ensure_ascii=False,
                    indent=2,
                )
            }
        )
        return self._parse_response(response)

    def enrich_corpus_hdfs(
        self,
        template: str,
        *,
        template_id: str = "unknown",
        examples: Sequence[str] | None = None,
        occurrence_count: int = 0,
        retrieved_docs: Sequence[Mapping[str, Any]] | None = None,
        candidate_relations: Sequence[dict[str, Any]] | None = None,
    ) -> EnrichedTemplate:
        """Backward-compatible HDFS wrapper around :meth:`enrich_template`."""
        return self.enrich_template(
            TemplateContext(
                template_id=template_id,
                template=template,
                occurrence_count=occurrence_count,
                examples=list(examples or []),
                retrieved_docs=[dict(item) for item in retrieved_docs or ()],
                candidate_relations=list(candidate_relations or []),
                dataset_context=HDFS_DATASET_CONTEXT,
            )
        )

    def enrich_corpus_bgl(
        self,
        template: str,
        *,
        template_id: str = "unknown",
        examples: Sequence[str] | None = None,
        occurrence_count: int = 0,
        retrieved_docs: Sequence[Mapping[str, Any]] | None = None,
        candidate_relations: Sequence[dict[str, Any]] | None = None,
    ) -> EnrichedTemplate:
        """Backward-compatible BGL wrapper around :meth:`enrich_template`."""
        return self.enrich_template(
            TemplateContext(
                template_id=template_id,
                template=template,
                occurrence_count=occurrence_count,
                examples=list(examples or []),
                retrieved_docs=[dict(item) for item in retrieved_docs or ()],
                candidate_relations=list(candidate_relations or []),
                dataset_context=BGL_DATASET_CONTEXT,
            )
        )

    @staticmethod
    def _parse_response(response: Any) -> EnrichedTemplate:
        """Parse and conservatively repair provider-specific JSON responses."""
        content = getattr(response, "content", response)
        if isinstance(content, list):
            content = "".join(
                part.get("text", "") if isinstance(part, dict) else str(part)
                for part in content
            )
        if not isinstance(content, str):
            raise TypeError(f"Expected text JSON response, got {type(content).__name__}")

        content = content.strip()
        if content.startswith("```"):
            content = content.strip("`")
            if content.startswith("json"):
                content = content[4:].lstrip()

        payload = json.loads(content)
        if not isinstance(payload, dict):
            raise ValueError("The LLM response must be a JSON object.")

        has_enrichment_wrapper = isinstance(payload.get("enrichment"), dict)
        payload = Enricher._flatten_sections(payload)
        payload = Enricher._normalise_root_aliases(payload)
        payload["failure_signals"], failure_signal_notes = (
            Enricher._normalise_failure_signals(payload.get("failure_signals", []))
        )
        payload["fields"] = Enricher._normalise_fields(payload.get("fields", []))
        payload["sequence_context"] = Enricher._normalise_relations(
            payload.get("sequence_context", [])
        )
        payload["explicit_conditions"] = coerce_string_list(
            payload.get("explicit_conditions")
        )
        payload["unsupported_inferences"] = coerce_string_list(
            payload.get("unsupported_inferences")
        )
        payload["unsupported_inferences"].extend(failure_signal_notes)
        unsupported = payload["unsupported_inferences"]
        payload["metadata_confidence"], confidence_note = (
            Enricher._normalise_confidence(payload.get("metadata_confidence"))
        )
        if confidence_note:
            unsupported.append(confidence_note)
        payload["diagnostic_role"], role_note = Enricher._normalise_diagnostic_role(
            payload.get("diagnostic_role")
        )
        if role_note:
            unsupported.append(role_note)
        if has_enrichment_wrapper:
            unsupported.append("Provider returned a nested enrichment object; it was unwrapped.")
        if not str(payload.get("dataset_label_caveat") or "").strip():
            payload["dataset_label_caveat"] = (
                "No event-level label information was supplied."
            )
            unsupported.append(
                "dataset_label_caveat was omitted by the model; filled from the prompt fallback."
            )

        return EnrichedTemplate.model_validate(payload)

    @staticmethod
    def _flatten_sections(payload: dict[str, Any]) -> dict[str, Any]:
        """Accept sectioned responses without making sectioning part of the API."""
        if not isinstance(payload.get("metadata"), dict) and not isinstance(
            payload.get("enrichment"), dict
        ):
            return payload

        result: dict[str, Any] = {}
        metadata = payload.get("metadata", {})
        enrichment = payload.get("enrichment", {})
        if isinstance(metadata, dict):
            result.update(metadata)
        if isinstance(enrichment, dict):
            result.update(enrichment)
        for key, value in payload.items():
            if key not in {"metadata", "enrichment"}:
                result[key] = value
        return result

    @staticmethod
    def _normalise_root_aliases(payload: dict[str, Any]) -> dict[str, Any]:
        """Repair common naming variants without inventing missing claims."""
        result = dict(payload)
        aliases = {
            "emitter_or_component": "component",
            "severity_or_priority": "log_level",
            "parameters": "fields",
            "conditions": "explicit_conditions",
            "confidence": "metadata_confidence",
        }
        for source, target in aliases.items():
            if target not in result and source in result:
                result[target] = result[source]
        return result

    @staticmethod
    def _normalise_fields(value: Any) -> list[dict[str, Any]]:
        if not isinstance(value, list):
            return []
        fields: list[dict[str, Any]] = []
        for item in value:
            if not isinstance(item, dict):
                continue
            field = dict(item)
            placeholder = field.get("placeholder")
            if placeholder is None:
                placeholder = field.pop("name", None)
            semantic_role = field.get("semantic_role")
            if semantic_role is None:
                semantic_role = field.pop("role", field.pop("description", None))
            placeholder = "" if placeholder is None else str(placeholder).strip()
            semantic_role = "" if semantic_role is None else str(semantic_role).strip()
            if not placeholder and not semantic_role:
                continue
            field["placeholder"] = placeholder or "unknown"
            field["semantic_role"] = semantic_role or "unknown"
            field.setdefault("source", field.pop("evidence", "unknown"))
            if field["source"] not in {
                "template",
                "examples",
                "corpus_relation",
                "documentation",
            }:
                field["source"] = "unknown"
            field["confidence"], _ = Enricher._normalise_confidence(
                field.get("confidence")
            )
            fields.append(field)
        return fields

    @staticmethod
    def _normalise_relations(value: Any) -> list[dict[str, Any]]:
        if not isinstance(value, list):
            return []
        relations: list[dict[str, Any]] = []
        for item in value:
            if not isinstance(item, dict):
                continue
            relation = dict(item)
            relation.setdefault("template_id", "unknown")
            relation.setdefault("relation", "potential_rca_context")
            relation.setdefault("support", "unknown")
            if relation["relation"] == "co-occurrence":
                relation["relation"] = "co_occurs_in_trace"
            relation.setdefault("source", "corpus_relation")
            if relation["source"] not in {
                "template",
                "examples",
                "corpus_relation",
                "documentation",
            }:
                relation["source"] = "unknown"
            relations.append(relation)
        return relations

    @staticmethod
    def _normalise_failure_signals(
        value: Any,
    ) -> tuple[list[dict[str, Any]], list[str]]:
        """Map common abbreviated provider output to the strict signal schema."""
        if value is None:
            return [], []
        if not isinstance(value, list):
            value = [value]

        signals: list[dict[str, Any]] = []
        notes: list[str] = []
        for item in value:
            if not isinstance(item, dict):
                continue
            signal = dict(item)
            signal.setdefault("name", signal.pop("signal", "unknown"))
            signal.setdefault(
                "manifestation",
                signal.pop("description", signal.pop("observable_signal", "unknown")),
            )
            signal["trigger_scope"], scope_note = Enricher._normalise_trigger_scope(
                signal.get("trigger_scope")
            )
            if scope_note:
                notes.append(
                    f"Failure signal {signal['name']!r}: {scope_note}"
                )
            signal.setdefault("source", signal.pop("evidence", "unknown"))
            if signal["source"] not in {
                "template",
                "examples",
                "corpus_relation",
                "documentation",
            }:
                signal["source"] = "unknown"
            signal["confidence"], _ = Enricher._normalise_confidence(
                signal.get("confidence")
            )
            signals.append(signal)
        return signals, notes

    @staticmethod
    def _normalise_trigger_scope(value: Any) -> tuple[str, str | None]:
        """Translate clear scope variants without treating ambiguous signals as local."""
        valid = {
            "explicit_in_template",
            "requires_sequence_context",
            "requires_external_metric",
        }
        if isinstance(value, str):
            normalised = value.strip().lower().replace("-", "_").replace(" ", "_")
            aliases = {
                "single_block_operation": "explicit_in_template",
                "single_event": "explicit_in_template",
                "event_level": "explicit_in_template",
                "template": "explicit_in_template",
                "sequence": "requires_sequence_context",
                "trace": "requires_sequence_context",
                "lifecycle": "requires_sequence_context",
                "external_metric": "requires_external_metric",
                "metric": "requires_external_metric",
            }
            scope = aliases.get(normalised, normalised)
            if scope in valid:
                return scope, None

        return "requires_sequence_context", (
            f"Provider returned unsupported trigger_scope {value!r}; "
            "recorded as 'requires_sequence_context' to avoid treating the signal "
            "as independently diagnostic."
        )

    @staticmethod
    def _normalise_confidence(value: Any) -> tuple[str, str | None]:
        """Return the schema's confidence enum without overstating evidence."""
        valid = ("high", "medium", "low", "unknown")
        if isinstance(value, str):
            normalised = value.strip().lower().replace(" ", "_")
            if normalised in valid:
                return normalised, None
            for confidence in valid:
                if confidence in normalised:
                    return (
                        confidence,
                        f"Provider used non-canonical confidence value {value!r}; "
                        f"normalised to {confidence!r}.",
                    )
        elif isinstance(value, dict):
            reported = [
                item.strip().lower()
                for item in value.values()
                if isinstance(item, str) and item.strip().lower() in valid
            ]
            if reported:
                # A single root confidence must not be stronger than any
                # field-level confidence the provider supplied.
                confidence = max(reported, key=valid.index)
                return (
                    confidence,
                    "Provider returned per-field confidence instead of one "
                    f"metadata_confidence value; conservatively used {confidence!r}.",
                )

        return "unknown", (
            f"Provider returned unsupported confidence value {value!r}; "
            "recorded as 'unknown'."
        )

    @staticmethod
    def _normalise_diagnostic_role(value: Any) -> tuple[str, str | None]:
        """Map only clear diagnostic-role variants; preserve ambiguity as unknown."""
        valid = {
            "informational",
            "lifecycle_transition",
            "warning_or_error",
            "context_dependent",
            "unknown",
        }
        if isinstance(value, str):
            normalised = value.strip().lower().replace("-", "_").replace(" ", "_")
            aliases = {
                "info": "informational",
                "information": "informational",
                "lifecycle": "lifecycle_transition",
                "transition": "lifecycle_transition",
                "warning": "warning_or_error",
                "error": "warning_or_error",
                "warning/error": "warning_or_error",
                "warning_or_error": "warning_or_error",
                "contextual": "context_dependent",
            }
            role = aliases.get(normalised, normalised)
            if role in valid:
                return role, None

        return "unknown", (
            f"Provider returned non-categorical diagnostic_role {value!r}; "
            "recorded as 'unknown'."
        )


"""Stage 2 — LLM template semantic enrichment.

Wraps the existing :class:`src.enricher.Enricher` and adds
batch-level helpers consumed by :mod:`run_ablation`.
"""


import json
import logging
import os
import hashlib
from pathlib import Path
from typing import Any

logger = logging.getLogger(__name__)

# Bumped when the enricher system prompt changes. Stage-2 cache keys omit git SHA,
# so this constant is what forces a re-enrich instead of reusing bland templates.
ENRICHMENT_PROMPT_VERSION = "bgl_extended_v1"


class IncompleteEnrichmentError(ValueError):
    """Raised when a known template lacks a validated frozen LLM response."""


def _canonical_digest(value: Any) -> str:
    """Return a stable SHA-256 digest for JSON-serialisable provenance."""
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def enrichment_provenance(
    templates_data: list[dict],
    *,
    dataset: str,
    enabled: bool,
    deployment: str | None = None,
) -> dict[str, Any]:
    """Describe frozen LLM inputs and outputs without storing credentials.

    The context digest covers exactly the template records passed to the
    enricher, including sanitized examples. The response digest covers the
    validated `enriched_large` records. Both are checked before an LLM graph
    can be built from a cached stage-2 result.
    """
    prompt_sources = dict(PROMPT_FILE_SHA256)
    contexts = [
        {
            "cluster_id": int(entry.get("cluster_id", index)),
            "template": entry.get("template", ""),
            "examples": entry.get("examples", []),
            "candidate_relations": entry.get("candidate_relations", []),
            "retrieved_docs": entry.get("retrieved_docs", []),
        }
        for index, entry in enumerate(templates_data)
        if int(entry.get("cluster_id", index)) >= 0
    ]
    responses = [
        {
            "cluster_id": int(entry.get("cluster_id", index)),
            "enriched_large": entry.get("enriched_large"),
        }
        for index, entry in enumerate(templates_data)
        if int(entry.get("cluster_id", index)) >= 0
    ]
    return {
        "schema_version": 1,
        "dataset": dataset.lower(),
        "enabled": bool(enabled),
        "prompt_version": ENRICHMENT_PROMPT_VERSION,
        "prompt_sources_sha256": prompt_sources,
        "prompt_sha256": _canonical_digest(prompt_sources),
        "deployment": deployment if enabled else None,
        "decoding": {"temperature": 0, "max_tokens": None, "max_retries": 2},
        "n_known_templates": len(contexts),
        "context_sha256": _canonical_digest(contexts),
        "response_sha256": _canonical_digest(responses) if enabled else None,
    }


def require_valid_enrichment_provenance(
    templates_data: list[dict],
    provenance: dict[str, Any],
    *,
    dataset: str,
) -> None:
    """Fail closed if frozen enrichment provenance is absent or inconsistent."""
    if not provenance.get("enabled"):
        raise IncompleteEnrichmentError("LLM enrichment provenance is not marked enabled.")
    if provenance.get("dataset") != dataset.lower():
        raise IncompleteEnrichmentError("LLM enrichment provenance dataset does not match.")
    if not provenance.get("deployment"):
        raise IncompleteEnrichmentError("LLM enrichment provenance lacks a deployment identifier.")
    require_complete_enrichment(templates_data)
    expected = enrichment_provenance(
        templates_data,
        dataset=dataset,
        enabled=True,
        deployment=str(provenance["deployment"]),
    )
    for key in ("prompt_version", "prompt_sha256", "context_sha256", "response_sha256"):
        if provenance.get(key) != expected[key]:
            raise IncompleteEnrichmentError(f"LLM enrichment provenance mismatch for {key}.")


def require_complete_enrichment(
    templates_data: list[dict],
    *,
    field: str = "enriched_large",
) -> None:
    """Fail closed when a non-OOV template lacks usable enrichment data."""
    missing = [
        int(entry.get("cluster_id", index))
        for index, entry in enumerate(templates_data)
        if int(entry.get("cluster_id", 0)) >= 0
        and not isinstance(entry.get(field), dict)
    ]
    if missing:
        raise IncompleteEnrichmentError(
            f"Missing validated {field} data for {len(missing)} known template(s): "
            f"cluster_ids={missing[:20]}"
        )


def enrich_templates(
    templates_data: list[dict],
    dataset: str,
    model_size: str = "large",
    enrichment_profile: str = "grounded",
) -> list[dict]:
    """Enrich a list of template dicts with LLM semantic annotations.

    Parameters
    ----------
    templates_data:
        List of dicts with at least a ``"template"`` key (output of the parser
        stage).  Adds ``"enriched_large"`` (Deepseek v4 Pro) in-place.
    dataset:
        ``"hdfs"`` or ``"bgl"`` — selects the enrichment prompt.
    model_size:
        Kept for cache/identity compatibility. Ablation enrichment always uses
        Azure ``AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO`` and stores the result
        in ``enriched_large``. ``small`` / ``both`` are not valid arms.

    Returns
    -------
    list[dict]
        The enriched templates list (same object, modified in-place).
    """

    if model_size in {"small", "both"}:
        raise ValueError(
            "Ablation enrichment uses Deepseek v4 Pro only; "
            f"model_size={model_size!r} is not supported."
        )
    deployment_env = "AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO"
    deployment = os.getenv(deployment_env)
    if not deployment:
        raise EnvironmentError(
            f"Environment variable {deployment_env!r} is not set. "
            "Cannot run LLM enrichment."
        )

    enricher = Enricher(deployment)
    field = "enriched_large"

    for i, entry in enumerate(templates_data):
        if int(entry.get("cluster_id", 0)) < 0:
            logger.info("Skipping OOV template cluster_id=%s", entry.get("cluster_id"))
            continue
        template = entry["template"]
        try:
            context = TemplateContext.from_template_record(
                entry,
                candidate_relations=entry.get("candidate_relations", []),
                retrieved_docs=entry.get("retrieved_docs", []),
            )
            if dataset.lower() == "hdfs":
                context.dataset_context = (
                    "HDFS-v1 labels apply to complete traces grouped by block ID, not "
                    "to an individual log event or template."
                )
            elif dataset.lower() == "bgl":
                context.dataset_context = (
                    "BGL source log messages carry event-level labels. When transformed "
                    "into time windows, a window is anomalous when it contains an "
                    "anomalous event. Enrichment profile: "
                    f"{enrichment_profile}."
                )

            result = enricher.enrich_template(context)
            entry[field] = result.model_dump(mode="json")
            logger.debug("Enriched template %d/%d", i + 1, len(templates_data))
        except Exception as exc:
            logger.warning(
                "Failed to enrich template %d (%r): %s", i + 1, template[:60], exc
            )

    require_complete_enrichment(templates_data, field=field)

    return templates_data


def load_enriched_templates(
    path: str | Path,
    *,
    preferred_size: str | None = None,
) -> tuple[list[dict], dict[int, str], dict[int, Any]]:
    """Load an enriched (or plain) templates JSON file and return look-ups.

    Gracefully falls back when no enrichment fields are present, so callers
    can use this function regardless of whether Stage 2 ran.

    Parameters
    ----------
    preferred_size:
        ``"large"`` or ``"small"`` selects ``enriched_*`` first when both
        fields are stored in the same JSON (campaign prepare writes both).

    Returns
    -------
    templates_data : list[dict]
        Raw list as stored on disk.
    cluster_to_template : dict[int, str]
        ``cluster_id → template string``.
    cluster_to_enriched : dict[int, Any]
        ``cluster_id → parsed enrichment dict``.  Empty if enrichment was
        disabled or the file contains only raw templates.
    """
    path = Path(path)
    with open(path) as fh:
        templates_data = json.load(fh)

    cluster_to_template: dict[int, str] = {
        t["cluster_id"]: t["template"] for t in templates_data
    }
    fields: list[str] = []
    if preferred_size in {"large", "small"}:
        fields.append(f"enriched_{preferred_size}")
    for name in ("enriched_large", "enriched_small"):
        if name not in fields:
            fields.append(name)
    cluster_to_enriched: dict[int, Any] = {}
    for t in templates_data:
        for field in fields:
            if field not in t:
                continue
            value = t[field]
            if isinstance(value, dict):
                cluster_to_enriched[t["cluster_id"]] = value
            elif isinstance(value, str):
                try:
                    cluster_to_enriched[t["cluster_id"]] = json.loads(value)
                except json.JSONDecodeError:
                    pass
            break

    return templates_data, cluster_to_template, cluster_to_enriched


"""Stage 3 — Log-line sequencer.

Two strategies, toggled via ``dataset``:

* ``"hdfs"`` — Block-ID grouping: every log line that references a block
  is grouped with all other lines sharing the same ``block_id``, producing
  one sequence per HDFS block (one sequence = one anomaly-labelling unit).

* ``"bgl"`` — Sliding time-window: the log is partitioned into time windows.
  A window is labelled anomalous if at least one of its lines has
  ``is_anomaly == True``.

  ``split="time"`` cuts the event stream into train/val/test **before**
  window construction so a window cannot straddle the cut and overlap is
  confined to one split. ``split="stratified"`` windows the full stream
  (notebook-era behaviour; pair only with a later random graph split).

Public API
----------
    build_sequences(df, dataset, **kwargs)  → dict
    save_sequences(df_blocks, output_path)
    load_sequences(path, dataset)           → (df_blocks, sequences)
    event_count_slices(n, train_ratio, val_ratio)
    bgl_split_name(window_id)
"""


import json
import logging
import time
from importlib.util import find_spec
from pathlib import Path

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Namespace window_id by split so train/val/test unix starts cannot collide.
BGL_SPLIT_STRIDE = 1 << 40


def _parquet_engine() -> str:
    """Prefer fastparquet for legacy artifacts, with a PyArrow fallback."""
    return "fastparquet" if find_spec("fastparquet") else "pyarrow"


def event_count_slices(
    n: int,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> tuple[int, int, int]:
    """Return (n_train, n_val, n_test) covering *n* time-ordered events."""
    if n <= 0:
        return 0, 0, 0
    n_train = max(1, int(n * train_ratio))
    n_val = max(1, int(n * val_ratio))
    if n_train + n_val >= n:
        n_test = 1 if n >= 3 else 0
        leftover = n - n_test
        n_train = max(1, leftover // 2) if leftover >= 2 else leftover
        n_val = leftover - n_train
        return n_train, n_val, n - n_train - n_val
    n_test = n - n_train - n_val
    return n_train, n_val, n_test


def bgl_split_name(window_id: int) -> str:
    """Map a namespaced BGL window_id back to train/val/test."""
    value = int(window_id)
    if value >= 2 * BGL_SPLIT_STRIDE:
        return "test"
    if value >= BGL_SPLIT_STRIDE:
        return "val"
    return "train"


def split_indices_from_bgl_windows(sequence_ids: list) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Build idx_train/val/test from time-namespaced window ids (stable order)."""
    train, val, test = [], [], []
    for index, sequence_id in enumerate(sequence_ids):
        name = bgl_split_name(int(sequence_id))
        if name == "train":
            train.append(index)
        elif name == "val":
            val.append(index)
        else:
            test.append(index)
    return (
        np.asarray(train, dtype=np.int64),
        np.asarray(val, dtype=np.int64),
        np.asarray(test, dtype=np.int64),
    )


def build_sequences(
    df: pd.DataFrame,
    dataset: str,
    *,
    window_minutes: int = 20,
    step_minutes: int = 10,
    split: str = "stratified",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    embargo_minutes: int = 0,
) -> dict:
    """Route to the dataset-appropriate sequencer.

    Parameters
    ----------
    df : pd.DataFrame
        Annotated log DataFrame produced by the parser stage.
    dataset : str
        ``"hdfs"`` or ``"bgl"``.
    window_minutes, step_minutes : int
        BGL-only sliding-window parameters (ignored for HDFS).
    split : str
        BGL only: ``"time"`` (cut events, then window) or ``"stratified"``
        (window the full stream).
    train_ratio, val_ratio : float
        Event-count fractions used when ``split="time"``.
    embargo_minutes : int
        BGL time only: discard events within this many minutes on either
        side of each chronological partition boundary.

    Returns
    -------
    dict
        ``{sequence_id: group_DataFrame}`` mapping.  For HDFS the keys are
        ``block_id`` strings; for BGL they are integer window ids.
    """
    dataset = dataset.lower()
    if dataset == "hdfs":
        return _build_hdfs_sequences(df)
    if dataset == "bgl":
        if str(split).lower() == "time":
            return _build_bgl_sequences_time_split(
                df,
                window_minutes=window_minutes,
                step_minutes=step_minutes,
                train_ratio=train_ratio,
                val_ratio=val_ratio,
                embargo_minutes=embargo_minutes,
            )
        return _build_bgl_sequences(df, window_minutes=window_minutes, step_minutes=step_minutes)
    raise ValueError(f"Unknown dataset {dataset!r}. Choose 'hdfs' or 'bgl'.")


def save_sequences(df_blocks: pd.DataFrame, output_path: str | Path) -> None:
    """Persist the flat sequences DataFrame to Parquet."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df_save = df_blocks.copy()
    if "parameters" in df_save.columns:
        df_save["parameters"] = df_save["parameters"].apply(json.dumps)
    for col in df_save.select_dtypes(include=["object", "string"]).columns:
        df_save[col] = df_save[col].astype(object)
    df_save.to_parquet(output_path, index=False, engine=_parquet_engine())
    logger.info("Saved %d rows → %s", len(df_save), output_path)


def load_sequences(
    path: str | Path,
    dataset: str,
) -> tuple[pd.DataFrame, dict]:
    """Load a sequences Parquet file and reconstruct the ``{id → group}`` dict.

    Returns
    -------
    df_blocks : pd.DataFrame
    sequences : dict
    """
    path = Path(path)
    df_blocks = pd.read_parquet(path, engine=_parquet_engine())
    if "parameters" in df_blocks.columns:
        df_blocks["parameters"] = df_blocks["parameters"].apply(json.loads)

    id_col = "window_id" if dataset.lower() == "bgl" else "block_id"
    sequences = {sid: grp for sid, grp in df_blocks.groupby(id_col, sort=False)}
    return df_blocks, sequences


def _build_hdfs_sequences(df: pd.DataFrame) -> dict:
    """Group HDFS log lines by ``block_id`` (one sequence per HDFS block)."""
    t0 = time.time()
    df_blocks = df.loc[df["block_id"].notna()].sort_values(["block_id", "timestamp"])
    sequences = {
        block_id: group for block_id, group in df_blocks.groupby("block_id", sort=False)
    }
    n_dropped = len(df) - len(df_blocks)
    logger.info(
        "HDFS sequencer: %d blocks in %.2fs  (dropped %d lines without block_id)",
        len(sequences),
        time.time() - t0,
        n_dropped,
    )
    return sequences


def _window_one_span(
    df: pd.DataFrame,
    *,
    window_minutes: int,
    step_minutes: int,
    id_offset: int,
) -> dict:
    """Sliding windows over one contiguous time span (one split)."""
    if df.empty:
        return {}
    df = df.dropna(subset=["unix_ts"]).sort_values("unix_ts").reset_index(drop=True)
    df["unix_ts"] = df["unix_ts"].astype(np.int64)
    window_seconds = window_minutes * 60
    step_seconds = step_minutes * 60
    unix_arr = df["unix_ts"].values
    t_min = int(unix_arr[0])
    t_max = int(unix_arr[-1])
    if t_max < t_min + window_seconds:
        # Span shorter than one window: keep a single window of whatever is there.
        window_starts = np.array([t_min], dtype=np.int64)
    else:
        window_starts = np.arange(
            t_min, t_max - window_seconds + 1, step_seconds, dtype=np.int64
        )

    row_indices: list[np.ndarray] = []
    win_ids: list[np.ndarray] = []
    for w_start in window_starts:
        lo = int(np.searchsorted(unix_arr, w_start, side="left"))
        hi = int(np.searchsorted(unix_arr, w_start + window_seconds, side="left"))
        n = hi - lo
        if n == 0:
            continue
        wid = int(w_start) + int(id_offset)
        row_indices.append(np.arange(lo, hi, dtype=np.int64))
        win_ids.append(np.full(n, wid, dtype=np.int64))

    if not row_indices:
        return {}
    all_rows = np.concatenate(row_indices)
    all_wids = np.concatenate(win_ids)
    df_windows = df.iloc[all_rows].copy()
    df_windows["window_id"] = all_wids
    df_windows = df_windows.sort_values(["window_id", "unix_ts"]).reset_index(drop=True)
    return {wid: grp for wid, grp in df_windows.groupby("window_id", sort=False)}


def _build_bgl_sequences(
    df: pd.DataFrame,
    *,
    window_minutes: int = 20,
    step_minutes: int = 10,
) -> dict:
    """Partition BGL log lines into sliding time windows on the full stream."""
    t0 = time.time()
    sequences = _window_one_span(
        df, window_minutes=window_minutes, step_minutes=step_minutes, id_offset=0
    )
    if not sequences:
        logger.warning("BGL sequencer produced 0 non-empty windows. Check log timestamps.")
        return {}
    logger.info(
        "BGL sequencer: %d non-empty windows (W=%d min, step=%d min) in %.2fs",
        len(sequences),
        window_minutes,
        step_minutes,
        time.time() - t0,
    )
    return sequences


def _build_bgl_sequences_time_split(
    df: pd.DataFrame,
    *,
    window_minutes: int,
    step_minutes: int,
    train_ratio: float,
    val_ratio: float,
    embargo_minutes: int = 0,
) -> dict:
    """Cut the event stream by count, embargo boundaries, then window each split."""
    t0 = time.time()
    ordered = df.dropna(subset=["unix_ts"]).sort_values("unix_ts").reset_index(drop=True)
    n_train, n_val, n_test = event_count_slices(len(ordered), train_ratio, val_ratio)
    embargo_seconds = int(embargo_minutes) * 60
    if embargo_seconds < 0:
        raise ValueError("embargo_minutes must be non-negative.")
    train_cut = int(ordered["unix_ts"].iloc[n_train - 1])
    val_cut = int(ordered["unix_ts"].iloc[n_train + n_val - 1])
    train_span = ordered.iloc[:n_train]
    val_span = ordered.iloc[n_train : n_train + n_val]
    test_span = ordered.iloc[n_train + n_val :]
    if embargo_seconds:
        train_span = train_span.loc[train_span["unix_ts"] <= train_cut - embargo_seconds]
        val_span = val_span.loc[
            (val_span["unix_ts"] > train_cut + embargo_seconds)
            & (val_span["unix_ts"] <= val_cut - embargo_seconds)
        ]
        test_span = test_span.loc[test_span["unix_ts"] > val_cut + embargo_seconds]
    if train_span.empty or val_span.empty or test_span.empty:
        raise ValueError(
            "BGL time split has an empty partition after applying the "
            f"{embargo_minutes}-minute embargo."
        )
    slices = {
        "train": (train_span, 0),
        "val": (val_span, BGL_SPLIT_STRIDE),
        "test": (test_span, 2 * BGL_SPLIT_STRIDE),
    }
    sequences: dict = {}
    counts: dict[str, int] = {}
    for name, (span, offset) in slices.items():
        part = _window_one_span(
            span, window_minutes=window_minutes, step_minutes=step_minutes, id_offset=offset
        )
        counts[name] = len(part)
        sequences.update(part)
    logger.info(
        "BGL time-split sequencer: train=%d val=%d test=%d windows "
        "(events %d/%d/%d before embargo, embargo=%d min, W=%d min, step=%d min) in %.2fs",
        counts.get("train", 0),
        counts.get("val", 0),
        counts.get("test", 0),
        n_train,
        n_val,
        n_test,
        embargo_minutes,
        window_minutes,
        step_minutes,
        time.time() - t0,
    )
    return sequences


"""Stage 4 — Collapsed-template graph builder.

Converts a per-sequence DataFrame into a NetworkX DiGraph using the
**collapsed-template** representation:

* One node per unique ``cluster_id`` in the sequence.
* Node features: occurrence count, parameter statistics, and 5 positional dims.
* Edge features: transition count, 7 time-delta distribution dims, and 3
  positional dims (10 dims total).
* When ``use_edge_features=False`` edges carry only the transition count
  (``weight``), reducing edge dimensionality for ablation experiments.

Also provides sequence fingerprinting for deduplication: sequences sharing
the same ordered ``cluster_id`` list are structurally identical and can
reuse a single canonical graph for efficiency.

Public API
----------
    build_collapsed_graph(seq, cluster_to_template, *, use_edge_features) → nx.DiGraph
    sequence_fingerprint(seq)                                               → str
    select_unique_sequences(sequences, labels)                              → (dict, dict, dict)
"""


import hashlib
import logging
from collections import Counter, defaultdict
from typing import Any, cast

import networkx as nx
import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)


# ── Public API ────────────────────────────────────────────────────────────────


def build_collapsed_graph(
    seq: pd.DataFrame,
    cluster_to_template: dict,
    *,
    use_edge_features: bool = True,
) -> nx.DiGraph:
    """Build a collapsed-template directed graph from a sequence DataFrame.

    Parameters
    ----------
    seq : pd.DataFrame
        Single-sequence DataFrame (one block or one window).  Must have
        ``cluster_id`` and ``parameters`` columns plus either ``timestamp``
        (HDFS datetime objects) or ``unix_ts`` (BGL integer seconds).
    cluster_to_template : dict
        Mapping ``cluster_id → template string``.
    use_edge_features : bool
        When ``False``, edges carry only ``weight`` (transition count).
        All time-delta and positional edge features are omitted.

    Returns
    -------
    nx.DiGraph
        Node attributes::

            template, occurrence_count, param_count, param_num_mean,
            param_num_max, first_pos, last_pos, mean_pos, std_pos, pos_spread

        Edge attributes (when ``use_edge_features=True``)::

            weight, mean_src_pos, mean_dst_pos, mean_pos_delta,
            td_min, td_p25, td_median, td_p75, td_max, td_std
    """
    G = nx.DiGraph()

    # Store sequence identifier for traceability
    for id_col in ("window_id", "block_id"):
        if id_col in seq.columns:
            G.graph[id_col] = seq[id_col].iloc[0]
            break

    cids = seq["cluster_id"].tolist()
    params = seq["parameters"].tolist()
    ts = _extract_timestamps(seq)
    n = len(cids)

    # ── Per-node aggregation ──────────────────────────────────────────────────
    node_params: dict = defaultdict(list)
    node_count = Counter(cids)
    node_positions: dict = defaultdict(list)

    for i, (cid, p) in enumerate(zip(cids, params)):
        node_params[cid].extend(p)
        node_positions[cid].append(i / max(n - 1, 1))

    for cid, count in node_count.items():
        nums = [float(x) for x in node_params[cid] if _is_numeric(x)]
        positions = np.array(node_positions[cid])

        G.add_node(
            cid,
            template=cluster_to_template.get(cid, ""),
            occurrence_count=count,
            param_count=len(node_params[cid]),
            param_num_mean=float(np.mean(nums)) if nums else 0.0,
            param_num_max=float(np.max(nums)) if nums else 0.0,
            first_pos=float(positions.min()),
            last_pos=float(positions.max()),
            mean_pos=float(positions.mean()),
            std_pos=float(positions.std()) if len(positions) > 1 else 0.0,
            pos_spread=float(positions.max() - positions.min()),
        )

    # ── Per-edge aggregation ──────────────────────────────────────────────────
    edge_deltas: dict = defaultdict(list)
    edge_src_pos: dict = defaultdict(list)
    edge_dst_pos: dict = defaultdict(list)

    for i in range(n - 1):
        src, dst = cids[i], cids[i + 1]
        src_norm = i / max(n - 1, 1)
        dst_norm = (i + 1) / max(n - 1, 1)

        edge_src_pos[(src, dst)].append(src_norm)
        edge_dst_pos[(src, dst)].append(dst_norm)

        t_src, t_dst = ts[i], ts[i + 1]
        if t_src is not None and t_dst is not None:
            try:
                delta = t_dst - t_src
                if hasattr(delta, "total_seconds"):
                    delta = delta.total_seconds()
                edge_deltas[(src, dst)].append(float(delta))
            except TypeError:
                if (src, dst) not in edge_deltas:
                    edge_deltas[(src, dst)]
        elif (src, dst) not in edge_deltas:
            edge_deltas[(src, dst)]

    for (src, dst) in edge_src_pos:
        s_pos = np.array(edge_src_pos[(src, dst)])
        d_pos = np.array(edge_dst_pos[(src, dst)])
        deltas = edge_deltas.get((src, dst), [])

        edge_attrs: dict = {"weight": len(s_pos)}

        if use_edge_features:
            edge_attrs.update(
                mean_src_pos=float(s_pos.mean()),
                mean_dst_pos=float(d_pos.mean()),
                mean_pos_delta=float((d_pos - s_pos).mean()),
            )
            if deltas:
                arr = np.array(deltas)
                edge_attrs.update(
                    td_min=float(arr.min()),
                    td_p25=float(np.percentile(arr, 25)),
                    td_median=float(np.median(arr)),
                    td_p75=float(np.percentile(arr, 75)),
                    td_max=float(arr.max()),
                    td_std=float(arr.std()),
                )
            else:
                edge_attrs.update(
                    td_min=-1, td_p25=-1, td_median=-1,
                    td_p75=-1, td_max=-1, td_std=0,
                )

        G.add_edge(src, dst, **edge_attrs)

    return G


def sequence_fingerprint(seq: pd.DataFrame) -> str:
    """Hash the ordered ``cluster_id`` sequence to a compact hex fingerprint.

    Two sequences with the same fingerprint have identical template orderings
    (same structural graph topology), though timestamps and parameters may differ.
    """
    cids = seq["cluster_id"].tolist()
    key = "|".join(str(c) for c in cids)
    return hashlib.md5(key.encode()).hexdigest()  # noqa: S324 — not security-sensitive


def select_unique_sequences(
    sequences: dict,
    labels: dict,
    *,
    prefer_ids: set | None = None,
) -> tuple[dict, dict, dict[str, int]]:
    """Keep one sequence per ordered-template fingerprint (and label, if mixed).

    The representative for each ``(fingerprint, label)`` pair is the first
    sequence id in sorted string order, unless ``prefer_ids`` is given (train
    block ids under inductive_v1). Fingerprints that appear with both
    normal and anomalous labels keep one example of each class.

    Returns
    -------
    unique_sequences, unique_labels, stats
        ``stats`` has ``n_raw``, ``n_unique``, and ``n_mixed_label_fingerprints``.
    """
    grouped: dict[str, dict[int, list]] = defaultdict(lambda: defaultdict(list))
    missing = [sequence_id for sequence_id in sequences if sequence_id not in labels]
    if missing:
        preview = ", ".join(str(item) for item in missing[:5])
        raise KeyError(
            f"{len(missing)} sequence(s) have no label (e.g. {preview}). "
            "Refusing to treat unlabeled sequences as normal."
        )
    for sequence_id in sorted(sequences, key=lambda sid: str(sid)):
        fingerprint = sequence_fingerprint(sequences[sequence_id])
        label = int(labels[sequence_id])
        grouped[fingerprint][label].append(sequence_id)

    preferred = {str(item) for item in (prefer_ids or [])}
    selected_ids: list = []
    n_mixed = 0
    for by_label in grouped.values():
        if len(by_label) > 1:
            n_mixed += 1
        for label in sorted(by_label):
            candidates = by_label[label]
            chosen = next((sid for sid in candidates if str(sid) in preferred), candidates[0])
            selected_ids.append(chosen)

    selected_ids.sort(key=lambda sid: str(sid))
    unique_sequences = {sequence_id: sequences[sequence_id] for sequence_id in selected_ids}
    unique_labels = {
        sequence_id: int(labels[sequence_id]) for sequence_id in selected_ids
    }
    stats = {
        "n_raw": len(sequences),
        "n_unique": len(unique_sequences),
        "n_mixed_label_fingerprints": n_mixed,
    }
    logger.info(
        "Unique sequences: %d / %d raw  (%d mixed-label fingerprints)",
        stats["n_unique"],
        stats["n_raw"],
        stats["n_mixed_label_fingerprints"],
    )
    return unique_sequences, unique_labels, stats


# ── Internal helpers ──────────────────────────────────────────────────────────


def _extract_timestamps(seq: pd.DataFrame) -> list:
    """Return a list of timestamps compatible with delta computation.

    Prefers ``unix_ts`` (numeric seconds) over ``timestamp`` (datetime).
    """
    if "unix_ts" in seq.columns:
        return [float(t) if t is not None else None for t in seq["unix_ts"].tolist()]
    if "timestamp" in seq.columns:
        return seq["timestamp"].tolist()
    return [None] * len(seq)


def _is_numeric(x: object) -> bool:
    """Return True only for finite non-NaN non-Inf numeric values."""
    try:
        v = float(cast(Any, x))
        return np.isfinite(v)
    except (ValueError, TypeError):
        return False


"""Stage 5 — PyTorch Geometric dataset preparation.

Converts per-sequence DataFrames and pre-computed template embeddings into
:class:`torch_geometric.data.Data` objects ready for GNN training.

Embedding strategy is controlled by two boolean ablation flags:
    ``tfidf_enabled``  — include TF-IDF vectors (structural/token features)
    ``sbert_enabled``  — include Sentence-BERT vectors (semantic features)
At least one must be True.

Public API
----------
    compute_embeddings(templates_data, cluster_to_enriched, *, ...) → (ndarray, cids, vec)
    densify_tfidf_rows(matrix) → ndarray
    build_graph_structures(sequences, block_labels, *, ...) → (list[dict], stats)
    attach_cluster_embeddings(structures, cluster_embeddings, *, ...) → list[Data]
    build_pyg_dataset(sequences, block_labels, cluster_embeddings, *, ...) → list[Data]
    split_dataset(all_data, seed, *, ...) → (idx_train, idx_val, idx_test)
    save_graph_dataset(all_data, ..., path, *, ...)
    load_graph_splits(path) → (train, val, test, meta)
    GraphDirectoryDataset(directory) — lazy per-graph ``*.pt`` shards
    save_graph_structures(path, structures, meta)
    load_graph_structures(path) → (structures, meta)
"""


import json
import logging
import pickle
import time
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Literal, Mapping, cast

import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)
HdfsFeatureContract = Literal["notebook_raw_v1", "stabilized_v2"]
NODE_EXTRA_DIM = 9
STRUCTURE_EDGE_DIM = 10
STRUCTURE_CACHE_VERSION = 1
SBERT_EMBED_DIM = 384
SBERT_TEXT_EMBEDDING = "embedding_text"
SBERT_TEXT_GROUNDED = "grounded_v1"


class MissingClusterEmbedding(ValueError):
    """Raised when a sequence references a cluster ID absent from frozen embeddings."""

    def __init__(self, cluster_id: int) -> None:
        self.cluster_id = cluster_id
        super().__init__(f"Cluster {cluster_id} has no frozen embedding.")


class MissingSequenceLabel(KeyError):
    """Raised when a sequence has no ground-truth anomaly label."""

    def __init__(self, sequence_id: Any) -> None:
        self.sequence_id = sequence_id
        super().__init__(
            f"Sequence {sequence_id!r} has no label. "
            "Refusing to treat unlabeled sequences as normal."
        )


TFIDF_DENSE_MAX_FEATURES = 10_000


def compose_sbert_text(
    template: str = "",
    enriched: Mapping[str, Any] | None = None,
    *,
    mode: str = SBERT_TEXT_EMBEDDING,
) -> str:
    """Build the MiniLM input from LLM enrichment fields only.

    Drain templates and raw examples are never encoded. Distinctive tokens must
    come from the enricher. ``embedding_text`` uses the LLM paragraph and falls
    back to the Drain template only when enrichment is missing (historical
    llm-off recipe). ``grounded_v1`` concatenates enrichment metadata,
    ``embedding_text``, ``event_semantics``, and explicit failure signals.
    """
    if mode not in {SBERT_TEXT_EMBEDDING, SBERT_TEXT_GROUNDED}:
        raise ValueError(
            f"sbert_text must be {SBERT_TEXT_EMBEDDING!r} or {SBERT_TEXT_GROUNDED!r}, got {mode!r}"
        )
    if mode == SBERT_TEXT_EMBEDDING:
        if enriched:
            text = str(enriched.get("embedding_text") or "").strip()
            if text:
                return text
        return str(template or "").strip() or "unknown log template"

    if not enriched:
        return "unknown log template"
    parts: list[str] = []
    meta_bits = [
        str(enriched.get(key) or "").strip()
        for key in ("log_level", "diagnostic_role", "operation")
    ]
    meta_line = " ".join(bit for bit in meta_bits if bit)
    if meta_line:
        parts.append(meta_line)
    for key in ("embedding_text", "event_semantics"):
        value = str(enriched.get(key) or "").strip()
        if value:
            parts.append(value)
    for signal in enriched.get("failure_signals") or []:
        if not isinstance(signal, dict):
            continue
        if signal.get("trigger_scope") != "explicit_in_template":
            continue
        chunk = " ".join(
            str(signal.get(key) or "").strip()
            for key in ("name", "manifestation")
        ).strip()
        if chunk:
            parts.append(chunk)
    return " ".join(parts).strip() or "unknown log template"


def embedding_block_dims(
    embed_dim: int,
    *,
    tfidf_enabled: bool,
    sbert_enabled: bool,
    sbert_width: int = SBERT_EMBED_DIM,
) -> tuple[int, int]:
    """Return ``(tfidf_dim, sbert_dim)`` for a concatenated embedding block."""
    if tfidf_enabled and sbert_enabled:
        if embed_dim <= sbert_width:
            raise ValueError(
                f"Hybrid embed_dim={embed_dim} is too small for MiniLM width {sbert_width}."
            )
        return int(embed_dim - sbert_width), int(sbert_width)
    if sbert_enabled:
        return 0, int(embed_dim)
    return int(embed_dim), 0


def sbert_dim_from_meta(meta: Mapping[str, Any] | None, node_dim: int | None = None) -> int:
    """SBERT block width from dataset_meta, or MiniLM convention on old bundles."""
    if not meta:
        return 0
    if meta.get("sbert_dim") is not None:
        return int(meta["sbert_dim"])
    flags = meta.get("embedding_flags") or {}
    identity = meta.get("graph_identity") or {}
    sbert_enabled = bool(flags.get("sbert_enabled", identity.get("sbert_enabled", False)))
    tfidf_enabled = bool(flags.get("tfidf_enabled", identity.get("tfidf_enabled", True)))
    embed_dim = int(meta.get("embed_dim") or 0)
    if embed_dim <= 0:
        width = int(node_dim or meta.get("node_dim") or 0)
        embed_dim = max(width - NODE_EXTRA_DIM, 0)
    if embed_dim <= 0 or not sbert_enabled:
        return 0
    _, sbert_dim = embedding_block_dims(
        embed_dim, tfidf_enabled=tfidf_enabled, sbert_enabled=True
    )
    return sbert_dim


def node_recon_indices(
    node_dim: int,
    *,
    sbert_dim: int,
    extra_dim: int = NODE_EXTRA_DIM,
) -> np.ndarray:
    """Column indices to reconstruct when skipping the SBERT block.

    Layout is ``[tfidf | sbert | extras]``. ``sbert_dim=0`` reconstructs every
    column (identical to the historical full-vector decoder).
    """
    if sbert_dim <= 0:
        return np.arange(node_dim, dtype=np.int64)
    embed_dim = node_dim - extra_dim
    if embed_dim < sbert_dim:
        raise ValueError(
            f"sbert_dim={sbert_dim} exceeds embedding width {embed_dim} "
            f"(node_dim={node_dim}, extra_dim={extra_dim})."
        )
    tfidf_dim = embed_dim - sbert_dim
    lexical = np.arange(tfidf_dim, dtype=np.int64)
    extras = np.arange(embed_dim, node_dim, dtype=np.int64)
    return np.concatenate([lexical, extras])


def _require_torch_geometric() -> None:
    """Fail before the per-sequence loop if PyG is missing from this venv."""
    try:
        import torch_geometric  # noqa: F401
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "torch_geometric is required to build graph_dataset.pt. "
            "This repository venv needs the same PyG as the local Torch 2.2.2 "
            "baseline: pip install torch-geometric==2.3.0"
        ) from exc


# ── Embedding computation ─────────────────────────────────────────────────────


def compute_embeddings(
    templates_data: list[dict],
    cluster_to_enriched: dict,
    *,
    tfidf_enabled: bool = True,
    sbert_enabled: bool = True,
    tfidf_fit_texts: list[str] | None = None,
    sbert_text: str = SBERT_TEXT_EMBEDDING,
) -> tuple[np.ndarray, list[int], Any]:
    """Compute hybrid (TF-IDF + SBERT) template embedding matrix.

    Parameters
    ----------
    templates_data : list[dict]
        Template metadata list (from parser/enrichment stage).
    cluster_to_enriched : dict
        Mapping ``cluster_id → enrichment dict``.
    tfidf_enabled, sbert_enabled : bool
        Ablation toggles.  At least one must be True.
    tfidf_fit_texts : list[str] | None
        If given, the TF-IDF vocabulary and IDF are fitted on this subset
        (train templates) and every template is then ``transform``ed.
        ``None`` fits on all templates (transductive).

    Returns
    -------
    hybrid_embeddings : np.ndarray, shape (n_templates, embed_dim)
    all_cids : list[int]
        Ordered cluster IDs matching embedding rows.
    tfidf_vectorizer : TfidfVectorizer | None
        Fitted vectorizer (serialisable for reuse); None if TF-IDF disabled.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer

    if not tfidf_enabled and not sbert_enabled:
        raise ValueError("At least one of tfidf_enabled or sbert_enabled must be True.")

    all_cids: list[int] = [t["cluster_id"] for t in templates_data]
    all_templates: list[str] = [t["template"] for t in templates_data]
    cluster_to_template: dict[int, str] = dict(zip(all_cids, all_templates))

    parts: list[np.ndarray] = []
    tfidf_vectorizer: Any = None

    # ── TF-IDF (structural token features) ───────────────────────────────────
    if tfidf_enabled:
        fit_corpus = list(tfidf_fit_texts) if tfidf_fit_texts is not None else all_templates
        if not fit_corpus:
            raise ValueError("TF-IDF requires at least one training template.")
        tfidf_vectorizer = TfidfVectorizer(analyzer="word", token_pattern=r"[^\s]+")
        tfidf_vectorizer.fit(fit_corpus)
        # Densify the *template table* (tens–hundreds of rows). Safe for HDFS/BGL
        # (vocab ≪ 10k). A much larger vocabulary should stay sparse until each
        # graph materialises its node matrix; see densify_tfidf_rows().
        tfidf_matrix = tfidf_vectorizer.transform(all_templates)
        n_features = int(tfidf_matrix.shape[1])
        if n_features > TFIDF_DENSE_MAX_FEATURES:
            logger.warning(
                "TF-IDF vocabulary has %d features (> %d). Densifying the "
                "template table anyway because GINE still needs dense node "
                "rows; consider a hashed/sparse path for a larger corpus.",
                n_features,
                TFIDF_DENSE_MAX_FEATURES,
            )
        tfidf_dense = densify_tfidf_rows(tfidf_matrix)
        parts.append(tfidf_dense)
        logger.info(
            "TF-IDF: %d-dim vectors for %d templates (fitted on %d)",
            tfidf_dense.shape[1],
            len(all_cids),
            len(fit_corpus),
        )

    # ── Sentence-BERT (semantic features on enriched text) ───────────────────
    if sbert_enabled:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise ImportError(
                "sentence-transformers is not installed. "
                "Run: pip install sentence-transformers"
            ) from exc

        sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
        template_by_cid = {int(item["cluster_id"]): item for item in templates_data}
        enriched_texts: list[str] = []
        for cid in all_cids:
            record = template_by_cid.get(int(cid), {})
            enriched_texts.append(
                compose_sbert_text(
                    str(record.get("template") or cluster_to_template.get(cid, "")),
                    cluster_to_enriched.get(cid),
                    mode=sbert_text,
                )
            )

        sbert_emb = sbert_model.encode(
            enriched_texts, show_progress_bar=True, normalize_embeddings=True
        )
        sbert_emb = np.nan_to_num(
            sbert_emb.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0
        )
        parts.append(sbert_emb)
        logger.info(
            "SBERT: %d-dim vectors for %d templates", sbert_emb.shape[1], len(all_cids)
        )

    hybrid_embeddings = np.hstack(parts) if len(parts) > 1 else parts[0]
    logger.info("Hybrid embedding dim: %d", hybrid_embeddings.shape[1])
    return hybrid_embeddings, all_cids, tfidf_vectorizer


def densify_tfidf_rows(matrix: Any) -> np.ndarray:
    """Materialise TF-IDF rows as float32. Template tables stay small (n ≪ 1k).

    GINEConv expects dense node features, so each template is densified here
    rather than storing a (n_graphs × vocab) cube. Call this per template
    table, not per log line.
    """
    if hasattr(matrix, "toarray"):
        return np.asarray(matrix.toarray(), dtype=np.float32)
    return np.asarray(matrix, dtype=np.float32)


# ── Graph structure (embedding-agnostic) ──────────────────────────────────────


def build_graph_structures(
    sequences: dict,
    block_labels: dict,
    *,
    dataset: str = "bgl",
    on_graph_error: Literal["skip", "fail"] = "skip",
    hdfs_feature_contract: HdfsFeatureContract = "stabilized_v2",
) -> tuple[list[dict[str, Any]], dict[str, int]]:
    """Build collapsed topology + node/edge extras, without template embeddings.

    Family A arms that share ``dataset``, ``unique_sequences``, and
    ``feature_contract`` can reuse this list and splice a different embedding
    matrix. Full 10-d edges are always stored; ``use_edge_features=False`` is a
    later column slice, not a second construction pass.
    """
    structures: list[dict[str, Any]] = []
    skipped = 0
    t0 = time.time()
    from tqdm import tqdm

    iterator = tqdm(
        sequences.items(),
        total=len(sequences),
        desc="Graph structure",
        unit="seq",
        mininterval=2.0,
    )
    for wid, seq in iterator:
        if wid not in block_labels:
            raise MissingSequenceLabel(wid)
        label = int(block_labels[wid])
        try:
            structures.append(
                _seq_to_structure(
                    seq,
                    label,
                    dataset=dataset,
                    hdfs_feature_contract=hdfs_feature_contract,
                )
            )
        except Exception as exc:
            if on_graph_error == "fail":
                raise
            skipped += 1
            if skipped <= 5:
                logger.warning("Skipped sequence %s: %s", wid, exc)

    stats = {"n_graphs": len(structures), "n_skipped": skipped}
    logger.info(
        "Built %d graph structures in %.1fs  (skipped %d)",
        stats["n_graphs"],
        time.time() - t0,
        skipped,
    )
    return structures, stats


def _as_owned_tensor(array: np.ndarray, *, dtype: "torch.dtype") -> "torch.Tensor":
    """Copy *array* into a tensor that does not share storage with NumPy or siblings.

    ``torch.from_numpy`` on empty or sliced buffers can alias the same data_ptr
    under different dtypes, which ``torch.save`` rejects.
    """
    import torch

    return torch.tensor(np.ascontiguousarray(array), dtype=dtype)


def attach_cluster_embeddings(
    structures: list[dict[str, Any]],
    cluster_embeddings: dict[int, np.ndarray],
    *,
    use_edge_features: bool = True,
    include_node_positional_features: bool = True,
    include_edge_temporal_features: bool = True,
    include_edge_positional_features: bool = True,
    missing_embedding: Literal["zero", "fail"] = "zero",
) -> list:
    """Splice embeddings and selected feature groups onto cached structures.

    Position and time ablations remove columns from encoder input and decoder
    targets. Full cached structures remain reusable across Family A arms.
    """
    _require_torch_geometric()
    import torch
    from torch_geometric.data import Data

    normalized = {
        int(cid): np.asarray(vector, dtype=np.float32)
        for cid, vector in cluster_embeddings.items()
    }
    embed_dim = next(iter(normalized.values())).shape[0]
    if missing_embedding == "fail":
        for structure in structures:
            for cluster_id in structure["cluster_ids"]:
                if int(cluster_id) not in normalized:
                    raise MissingClusterEmbedding(int(cluster_id))

    from tqdm import tqdm

    all_data = []
    t0 = time.time()
    iterator = tqdm(
        structures,
        total=len(structures),
        desc="PyG splice",
        unit="seq",
        mininterval=2.0,
    )
    for structure in iterator:
        node_extra = np.asarray(structure["node_extra"], dtype=np.float32)
        if not include_node_positional_features:
            node_extra = node_extra[:, :4]
        node_feats = _node_features_from_structure(
            structure,
            normalized,
            embed_dim,
            missing_embedding,
            node_extra=node_extra,
        )
        edge_attr_np = structure["edge_attr"]
        if use_edge_features:
            columns = [0]
            if include_edge_temporal_features:
                columns.extend(range(1, 7))
            if include_edge_positional_features:
                columns.extend(range(7, 10))
            edge_attr_np = np.asarray(edge_attr_np[:, columns], dtype=np.float32)
        elif edge_attr_np.shape[0] == 0:
            edge_attr_np = np.zeros((0, 1), dtype=np.float32)
        else:
            edge_attr_np = np.asarray(edge_attr_np[:, :1], dtype=np.float32)

        id_col = structure["id_col"]
        kwargs = {id_col: structure["seq_id"], "num_nodes": int(structure["num_nodes"])}
        kwargs["event_cluster_ids"] = _as_owned_tensor(
            np.asarray(structure["event_cluster_ids"], dtype=np.int64), dtype=torch.long
        )
        all_data.append(
            Data(
                x=_as_owned_tensor(node_feats, dtype=torch.float32),
                edge_index=_as_owned_tensor(
                    np.asarray(structure["edge_index"], dtype=np.int64),
                    dtype=torch.long,
                ),
                edge_attr=_as_owned_tensor(edge_attr_np, dtype=torch.float32),
                y=torch.tensor([int(structure["y"])], dtype=torch.long),
                **kwargs,
            )
        )
    logger.info("Spliced embeddings onto %d graphs in %.1fs", len(all_data), time.time() - t0)
    return all_data


def save_graph_structures(
    path: str | Path,
    structures: list[dict[str, Any]],
    meta: dict[str, Any] | None = None,
) -> None:
    """Persist embedding-agnostic graph structures for Family A reuse."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "version": STRUCTURE_CACHE_VERSION,
        "structures": structures,
        "meta": meta or {},
    }
    with path.open("wb") as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    logger.info("Graph structures saved → %s  (%.1f MB)", path, path.stat().st_size / 1e6)


def load_graph_structures(path: str | Path) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    """Load a payload written by :func:`save_graph_structures`."""
    path = Path(path)
    with path.open("rb") as handle:
        payload = pickle.load(handle)
    if int(payload.get("version", -1)) != STRUCTURE_CACHE_VERSION:
        raise ValueError(
            f"Unsupported graph structure cache version {payload.get('version')!r} in {path}"
        )
    return list(payload["structures"]), dict(payload.get("meta") or {})


def build_pyg_dataset(
    sequences: dict,
    block_labels: dict,
    cluster_embeddings: dict[int, np.ndarray],
    *,
    use_edge_features: bool = True,
    dataset: str = "bgl",
    missing_embedding: Literal["zero", "fail"] = "zero",
    on_graph_error: Literal["skip", "fail"] = "skip",
    hdfs_feature_contract: HdfsFeatureContract = "stabilized_v2",
) -> list:
    """Convert sequences to a list of :class:`torch_geometric.data.Data` objects.

    Parameters
    ----------
    sequences : dict
        ``{sequence_id: DataFrame}`` produced by the sequencer.
    block_labels : dict
        ``{sequence_id: 0|1}`` anomaly labels.
    cluster_embeddings : dict
        ``{cluster_id: np.ndarray}`` embedding vectors.
    use_edge_features : bool
        When False, edge_attr carries only the log-scaled transition count.
    dataset : str
        ``"bgl"`` (uses ``unix_ts`` + ``window_id``) or
        ``"hdfs"`` (uses ``timestamp`` + ``block_id``).
    hdfs_feature_contract : str
        HDFS feature encoding declared by the model package. ``"notebook_raw_v1"``
        preserves the approved Colab baseline; ``"stabilized_v2"`` uses the
        current log-scaled feature construction.

    Returns
    -------
    list[torch_geometric.data.Data]
    """
    structures, _ = build_graph_structures(
        sequences,
        block_labels,
        dataset=dataset,
        on_graph_error=on_graph_error,
        hdfs_feature_contract=hdfs_feature_contract,
    )
    if not structures:
        return []
    return attach_cluster_embeddings(
        structures,
        cluster_embeddings,
        use_edge_features=use_edge_features,
        missing_embedding=missing_embedding,
    )


def split_dataset(
    all_data: list,
    seed: int = 42,
    *,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Stratified train / val / test split (default 70 / 15 / 15).

    Returns
    -------
    idx_train, idx_val, idx_test : np.ndarray
    """
    from sklearn.model_selection import train_test_split

    all_labels = np.array([d.y.item() for d in all_data])
    indices = np.arange(len(all_data))
    test_ratio = 1.0 - train_ratio - val_ratio

    idx_train, idx_temp = train_test_split(
        indices,
        test_size=(1.0 - train_ratio),
        random_state=seed,
        stratify=all_labels,
    )
    labels_temp = all_labels[idx_temp]
    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=test_ratio / (val_ratio + test_ratio),
        random_state=seed,
        stratify=labels_temp,
    )
    return idx_train, idx_val, idx_test


def split_label_indices(
    labels: np.ndarray,
    seed: int = 42,
    *,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Stratified 70/15/15 split from a label vector (no PyG graphs required)."""
    from sklearn.model_selection import train_test_split

    labels = np.asarray(labels)
    indices = np.arange(len(labels))
    test_ratio = 1.0 - train_ratio - val_ratio
    idx_train, idx_temp = train_test_split(
        indices,
        test_size=(1.0 - train_ratio),
        random_state=seed,
        stratify=labels,
    )
    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=test_ratio / (val_ratio + test_ratio),
        random_state=seed,
        stratify=labels[idx_temp],
    )
    return idx_train, idx_val, idx_test


def save_graph_dataset(
    all_data: list,
    idx_train: np.ndarray,
    idx_val: np.ndarray,
    idx_test: np.ndarray,
    path: str | Path,
    *,
    node_dim: int,
    edge_dim: int,
    embed_dim: int,
    dataset_meta: dict[str, Any] | None = None,
) -> None:
    """Persist the graph dataset bundle to ``path`` via ``torch.save``."""
    import torch

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    idx_train = np.ascontiguousarray(idx_train, dtype=np.int64).copy()
    idx_val = np.ascontiguousarray(idx_val, dtype=np.int64).copy()
    idx_test = np.ascontiguousarray(idx_test, dtype=np.int64).copy()
    payload: dict[str, Any] = {
        "data_list": all_data,
        "idx_train": idx_train,
        "idx_val": idx_val,
        "idx_test": idx_test,
        "node_dim": node_dim,
        "edge_dim": edge_dim,
        "embed_dim": embed_dim,
    }
    if dataset_meta:
        payload["dataset_meta"] = dataset_meta
        for key in (
            "feature_contract",
            "llm_enrichment_enabled",
            "enrichment_model_size",
            "use_edge_features",
            "embedding_flags",
            "graph_identity",
            "split_lock_id",
        ):
            if key in dataset_meta:
                payload[key] = dataset_meta[key]
    torch.save(payload, path)
    logger.info("Graph dataset saved → %s  (%.1f MB)", path, path.stat().st_size / 1e6)
    _save_graph_splits(
        path,
        all_data,
        idx_train,
        idx_val,
        idx_test,
        node_dim=node_dim,
        edge_dim=edge_dim,
        embed_dim=embed_dim,
        dataset_meta=dataset_meta,
    )


def graph_split_dir(path: str | Path) -> Path:
    """Directory of per-split shards next to ``graph_dataset.pt``."""
    path = Path(path)
    name = path.name
    if name.endswith(".gz"):
        name = name[:-3]
    stem = Path(name).stem
    return path.parent / f"{stem}_splits"


def _save_graph_splits(
    path: Path,
    all_data: list,
    idx_train: np.ndarray,
    idx_val: np.ndarray,
    idx_test: np.ndarray,
    *,
    node_dim: int,
    edge_dim: int,
    embed_dim: int,
    dataset_meta: dict[str, Any] | None,
) -> None:
    """Write train/val/test lists so training need not ``torch.load`` every graph."""
    import torch

    split_dir = graph_split_dir(path)
    split_dir.mkdir(parents=True, exist_ok=True)
    meta = {
        "node_dim": node_dim,
        "edge_dim": edge_dim,
        "embed_dim": embed_dim,
        "n_train": int(len(idx_train)),
        "n_val": int(len(idx_val)),
        "n_test": int(len(idx_test)),
        "n_total": int(len(all_data)),
    }
    if dataset_meta:
        meta["dataset_meta"] = dataset_meta
    torch.save([all_data[int(i)] for i in idx_train], split_dir / "train.pt")
    torch.save([all_data[int(i)] for i in idx_val], split_dir / "val.pt")
    torch.save([all_data[int(i)] for i in idx_test], split_dir / "test.pt")
    (split_dir / "meta.json").write_text(json.dumps(meta, indent=2, default=str))
    logger.info("Graph splits saved → %s", split_dir)


class GraphDirectoryDataset:
    """Lazy ``__getitem__`` over one graph per ``*.pt`` file.

    Unique-sequence HDFS (~17.6k) already avoids loading 575k blocks. Use this
    only when training the full block set: write shards with
    :func:`save_graph_directory`, then wrap each split directory. Metrics are
    unchanged if the graphs themselves are identical.
    """

    def __init__(self, directory: str | Path) -> None:
        self.directory = Path(directory)
        self.paths = sorted(self.directory.glob("*.pt"))
        if not self.paths:
            raise FileNotFoundError(f"No *.pt graphs in {self.directory}")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int):
        import torch

        return torch.load(self.paths[index], weights_only=False, map_location="cpu")


def save_graph_directory(graphs: list, directory: str | Path) -> Path:
    """Write one ``{index:06d}.pt`` per graph for :class:`GraphDirectoryDataset`."""
    import torch

    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for index, graph in enumerate(graphs):
        torch.save(graph, directory / f"{index:06d}.pt")
    return directory


def load_graph_splits(path: str | Path) -> tuple[list, list, list, dict[str, Any]]:
    """Load train/val/test graphs without materialising the full list when shards exist.

    Falls back to a single ``torch.load`` of ``data_list`` for older bundles.
    Unique-sequence campaigns load three split lists (~17.6k total). A 575k
    all-block run should use :class:`GraphDirectoryDataset` instead of one
    ``torch.save`` list.
    """
    import torch

    path = Path(path)
    split_dir = graph_split_dir(path)
    train_path = split_dir / "train.pt"
    if train_path.exists() and (split_dir / "val.pt").exists() and (split_dir / "test.pt").exists():
        train_graphs = torch.load(train_path, weights_only=False, map_location="cpu")
        val_graphs = torch.load(split_dir / "val.pt", weights_only=False, map_location="cpu")
        test_graphs = torch.load(split_dir / "test.pt", weights_only=False, map_location="cpu")
        meta: dict[str, Any] = {}
        meta_path = split_dir / "meta.json"
        if meta_path.exists():
            meta = json.loads(meta_path.read_text())
        sidecar = path.with_name("dataset_meta.json")
        if sidecar.exists():
            meta.setdefault("dataset_meta", json.loads(sidecar.read_text()))
        return list(train_graphs), list(val_graphs), list(test_graphs), meta

    bundle = torch.load(path, weights_only=False, map_location="cpu")
    all_data = bundle["data_list"]
    train_graphs = [all_data[int(i)] for i in bundle["idx_train"]]
    val_graphs = [all_data[int(i)] for i in bundle["idx_val"]]
    test_graphs = [all_data[int(i)] for i in bundle["idx_test"]]
    meta = {
        "node_dim": bundle.get("node_dim"),
        "edge_dim": bundle.get("edge_dim"),
        "embed_dim": bundle.get("embed_dim"),
        "dataset_meta": bundle.get("dataset_meta") or {},
    }
    return train_graphs, val_graphs, test_graphs, meta


# ── Internal helpers ──────────────────────────────────────────────────────────


def _is_numeric(x: object) -> bool:
    """Return True only for finite non-NaN/Inf numeric values."""
    try:
        v = float(cast(Any, x))
        return np.isfinite(v)
    except (ValueError, TypeError):
        return False


def _seq_to_structure(
    seq: pd.DataFrame,
    label: int,
    *,
    dataset: str,
    hdfs_feature_contract: HdfsFeatureContract = "stabilized_v2",
) -> dict[str, Any]:
    """Collapsed topology, 9-d node extras, and 10-d edges for one sequence."""
    ts_col = "unix_ts" if dataset.lower() == "bgl" else "timestamp"
    id_col = "window_id" if dataset.lower() == "bgl" else "block_id"

    seq_id = seq[id_col].iloc[0]
    cids = seq["cluster_id"].tolist()
    params = seq["parameters"].tolist() if "parameters" in seq.columns else [[] for _ in cids]
    ts = seq[ts_col].tolist() if ts_col in seq.columns else [None] * len(cids)
    n = len(cids)

    node_params: dict = defaultdict(list)
    node_count = Counter(cids)
    node_positions: dict = defaultdict(list)

    for i, (cid, p) in enumerate(zip(cids, params)):
        node_params[cid].extend(p if isinstance(p, list) else [])
        node_positions[cid].append(i / max(n - 1, 1))

    unique_cids = [int(cid) for cid in dict.fromkeys(cids)]
    cid_to_idx = {cid: idx for idx, cid in enumerate(unique_cids)}
    num_nodes = len(unique_cids)
    raw_hdfs = dataset.lower() == "hdfs" and hdfs_feature_contract == "notebook_raw_v1"

    node_extra = np.zeros((num_nodes, NODE_EXTRA_DIM), dtype=np.float32)
    for idx, cid in enumerate(unique_cids):
        nums = [float(x) for x in node_params[cid] if _is_numeric(x)]
        pos = np.array(node_positions[cid])
        if raw_hdfs:
            node_extra[idx, 0] = float(node_count[cid])
            node_extra[idx, 1] = float(len(node_params[cid]))
            node_extra[idx, 2] = float(np.mean(nums)) if nums else 0.0
            node_extra[idx, 3] = float(np.max(nums)) if nums else 0.0
        else:
            node_extra[idx, 0] = float(np.log1p(node_count[cid]))
            node_extra[idx, 1] = float(np.log1p(len(node_params[cid])))
            node_extra[idx, 2] = float(np.log1p(abs(np.mean(nums)))) if nums else 0.0
            node_extra[idx, 3] = float(np.log1p(abs(np.max(nums)))) if nums else 0.0
        node_extra[idx, 4] = float(pos.min())
        node_extra[idx, 5] = float(pos.max())
        node_extra[idx, 6] = float(pos.mean())
        node_extra[idx, 7] = float(pos.std()) if len(pos) > 1 else 0.0
        node_extra[idx, 8] = float(pos.max() - pos.min())

    node_extra = np.nan_to_num(node_extra, nan=0.0, posinf=0.0, neginf=0.0)

    edge_deltas: dict = defaultdict(list)
    edge_src_pos: dict = defaultdict(list)
    edge_dst_pos: dict = defaultdict(list)

    for i in range(n - 1):
        src, dst = cids[i], cids[i + 1]
        src_norm = i / max(n - 1, 1)
        dst_norm = (i + 1) / max(n - 1, 1)
        edge_src_pos[(src, dst)].append(src_norm)
        edge_dst_pos[(src, dst)].append(dst_norm)

        t_src, t_dst = ts[i], ts[i + 1]
        if t_src is not None and t_dst is not None:
            try:
                if raw_hdfs:
                    delta = (t_dst - t_src).total_seconds()
                else:
                    delta = float(t_dst) - float(t_src)
                edge_deltas[(src, dst)].append(delta)
            except (AttributeError, TypeError, ValueError):
                if (src, dst) not in edge_deltas:
                    edge_deltas[(src, dst)]
        elif (src, dst) not in edge_deltas:
            edge_deltas[(src, dst)]

    src_list, dst_list, edge_feats_list = [], [], []
    for (src, dst) in edge_src_pos:
        deltas = edge_deltas.get((src, dst), [])
        s_pos = np.array(edge_src_pos[(src, dst)])
        d_pos = np.array(edge_dst_pos[(src, dst)])
        ef = np.zeros(STRUCTURE_EDGE_DIM, dtype=np.float32)
        if raw_hdfs:
            ef[0] = float(len(s_pos))
            if deltas:
                arr = np.array(deltas, dtype=np.float64)
                ef[1] = float(arr.min())
                ef[2] = float(np.percentile(arr, 25))
                ef[3] = float(np.median(arr))
                ef[4] = float(np.percentile(arr, 75))
                ef[5] = float(arr.max())
                ef[6] = float(arr.std())
            else:
                ef[1:7] = [-1, -1, -1, -1, -1, 0]
        else:
            ef[0] = float(np.log1p(len(s_pos)))
            if deltas:
                arr = np.clip(np.array(deltas, dtype=np.float64), 0.0, None)
                ef[1] = float(np.log1p(arr.min()))
                ef[2] = float(np.log1p(np.percentile(arr, 25)))
                ef[3] = float(np.log1p(np.median(arr)))
                ef[4] = float(np.log1p(np.percentile(arr, 75)))
                ef[5] = float(np.log1p(arr.max()))
                ef[6] = float(np.log1p(arr.std()))
        ef[7] = float(s_pos.mean())
        ef[8] = float(d_pos.mean())
        ef[9] = float((d_pos - s_pos).mean())
        src_list.append(cid_to_idx[int(src)])
        dst_list.append(cid_to_idx[int(dst)])
        edge_feats_list.append(ef)

    if edge_feats_list:
        edge_index = np.asarray([src_list, dst_list], dtype=np.int64)
        edge_attr = np.nan_to_num(
            np.asarray(edge_feats_list, dtype=np.float32),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
    else:
        edge_index = np.zeros((2, 0), dtype=np.int64)
        edge_attr = np.zeros((0, STRUCTURE_EDGE_DIM), dtype=np.float32)

    return {
        "event_cluster_ids": np.asarray(cids, dtype=np.int64),
        "cluster_ids": np.asarray(unique_cids, dtype=np.int64),
        "node_extra": node_extra,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "y": int(label),
        "id_col": id_col,
        "seq_id": seq_id,
        "num_nodes": num_nodes,
    }


def _node_features_from_structure(
    structure: dict[str, Any],
    cluster_embeddings: dict[int, np.ndarray],
    embed_dim: int,
    missing_embedding: Literal["zero", "fail"],
    *,
    node_extra: np.ndarray | None = None,
) -> np.ndarray:
    cluster_ids = [int(cid) for cid in structure["cluster_ids"]]
    num_nodes = len(cluster_ids)
    extras = np.asarray(
        structure["node_extra"] if node_extra is None else node_extra,
        dtype=np.float32,
    )
    if extras.shape[0] != num_nodes:
        raise ValueError("node_extra must contain one row per graph node.")
    node_feats = np.zeros((num_nodes, embed_dim + extras.shape[1]), dtype=np.float32)
    for idx, cid in enumerate(cluster_ids):
        if cid not in cluster_embeddings:
            if missing_embedding == "fail":
                raise MissingClusterEmbedding(cid)
            emb = np.zeros(embed_dim, dtype=np.float32)
        else:
            emb = np.asarray(cluster_embeddings[cid], dtype=np.float32)
        node_feats[idx, :embed_dim] = emb
    node_feats[:, embed_dim:] = extras
    return np.nan_to_num(node_feats, nan=0.0, posinf=0.0, neginf=0.0)


def _seq_to_pyg(
    seq: pd.DataFrame,
    label: int,
    cluster_embeddings: dict,
    *,
    embed_dim: int,
    node_dim: int,
    edge_dim: int,
    use_edge_features: bool,
    dataset: str,
    missing_embedding: Literal["zero", "fail"] = "zero",
    hdfs_feature_contract: HdfsFeatureContract = "stabilized_v2",
):
    """Convert a single sequence DataFrame to a PyG Data object."""
    del node_dim, edge_dim  # derived from embeddings + cached 10-d structure
    structure = _seq_to_structure(
        seq,
        label,
        dataset=dataset,
        hdfs_feature_contract=hdfs_feature_contract,
    )
    return attach_cluster_embeddings(
        [structure],
        cluster_embeddings,
        use_edge_features=use_edge_features,
        missing_embedding=missing_embedding,
    )[0]



def gzip_file(source: str | Path, destination: str | Path) -> Path:
    source = Path(source)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with source.open("rb") as incoming, gzip.open(destination, "wb", compresslevel=6) as outgoing:
        shutil.copyfileobj(incoming, outgoing, length=16 * 1024 * 1024)
    return destination


def split_lock_id(idx_train, idx_val, idx_test) -> str:
    payload = np.concatenate(
        [np.asarray(idx_train), np.asarray(idx_val), np.asarray(idx_test)]
    )
    return hashlib.sha256(payload.astype(np.int64).tobytes()).hexdigest()[:16]


def save_split_lock(
    path: str | Path,
    idx_train,
    idx_val,
    idx_test,
    *,
    sequence_ids: list[str] | None = None,
    seed: int | None = None,
    protocol: str | None = None,
) -> str:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    lock_id = split_lock_id(idx_train, idx_val, idx_test)
    payload = {
        "idx_train": np.asarray(idx_train, dtype=np.int64),
        "idx_val": np.asarray(idx_val, dtype=np.int64),
        "idx_test": np.asarray(idx_test, dtype=np.int64),
        "n_total": np.int64(len(idx_train) + len(idx_val) + len(idx_test)),
        "split_lock_id": np.asarray(lock_id),
    }
    if seed is not None:
        payload["seed"] = np.int64(seed)
    if protocol is not None:
        payload["protocol"] = np.asarray(protocol)
    if sequence_ids is not None:
        payload["sequence_ids"] = np.asarray(sequence_ids)
    np.savez_compressed(path, **payload)
    return lock_id


def load_split_lock(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    with np.load(path, allow_pickle=True) as payload:
        lock = {
            "idx_train": np.asarray(payload["idx_train"], dtype=np.int64),
            "idx_val": np.asarray(payload["idx_val"], dtype=np.int64),
            "idx_test": np.asarray(payload["idx_test"], dtype=np.int64),
            "n_total": int(payload["n_total"]) if "n_total" in payload.files else None,
            "split_lock_id": str(payload["split_lock_id"]) if "split_lock_id" in payload.files else None,
        }
        if "sequence_ids" in payload.files:
            lock["sequence_ids"] = [str(item) for item in payload["sequence_ids"].tolist()]
        if "seed" in payload.files:
            lock["seed"] = int(payload["seed"])
        if "protocol" in payload.files:
            lock["protocol"] = str(payload["protocol"])
    if lock["split_lock_id"] is None:
        lock["split_lock_id"] = split_lock_id(lock["idx_train"], lock["idx_val"], lock["idx_test"])
    if lock["n_total"] is None:
        lock["n_total"] = len(lock["idx_train"]) + len(lock["idx_val"]) + len(lock["idx_test"])
    return lock


def write_campaign_manifest(campaign_dir, *, campaign_id, dataset, family, graphs, split_lock, extra=None):
    campaign_dir = Path(campaign_dir)
    campaign_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "campaign_id": campaign_id,
        "dataset": dataset,
        "family": family,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "split_lock": str(split_lock) if split_lock else None,
        "graphs": graphs,
    }
    if extra:
        payload.update(dict(extra))
    path = campaign_dir / "manifest.json"
    path.write_text(json.dumps(payload, indent=2, default=str))
    return path


def _to_binary_label(value: Any) -> int:
    if isinstance(value, str):
        return int(value.strip().lower() in {"1", "true", "anomaly", "anomalous"})
    return int(bool(value))


def hdfs_label_mapping(labels_path: Path) -> dict[str, int]:
    labels_frame = pd.read_csv(labels_path)
    columns = {column.lower(): column for column in labels_frame.columns}
    id_column = next((columns[name] for name in ("blockid", "block_id") if name in columns), None)
    label_column = next((columns[name] for name in ("label", "anomaly", "is_anomaly") if name in columns), None)
    if id_column is None or label_column is None:
        raise ValueError(f"Expected BlockId and Label columns in {labels_path}; found {list(labels_frame.columns)}")
    return {
        str(row[id_column]): _to_binary_label(row[label_column])
        for _, row in labels_frame.iterrows()
    }


ARM_SPECS = {
    "tfidf_only": {
        "llm": False, "tfidf": True, "sbert": False, "sbert_text": "embedding_text",
        "node_pos": True, "edge_pos": True, "edge_temp": True, "use_edges": True,
    },
    "sbert_raw": {
        "llm": False, "tfidf": False, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": True, "edge_pos": True, "edge_temp": True, "use_edges": True,
    },
    "sbert_llm": {
        "llm": True, "tfidf": False, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": True, "edge_pos": True, "edge_temp": True, "use_edges": True,
    },
    "hybrid_raw": {
        "llm": False, "tfidf": True, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": True, "edge_pos": True, "edge_temp": True, "use_edges": True,
    },
    "hybrid_llm": {
        "llm": True, "tfidf": True, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": True, "edge_pos": True, "edge_temp": True, "use_edges": True,
    },
    "no_positional_features": {
        "llm": True, "tfidf": True, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": False, "edge_pos": False, "edge_temp": True, "use_edges": True,
    },
    "no_temporal_features": {
        "llm": True, "tfidf": True, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": True, "edge_pos": True, "edge_temp": False, "use_edges": True,
    },
    "no_temporal_or_positional_features": {
        "llm": True, "tfidf": True, "sbert": True, "sbert_text": "embedding_text",
        "node_pos": False, "edge_pos": False, "edge_temp": False, "use_edges": True,
    },
}

PREPARE_STATE: dict[str, Any] = {}
_EMBED_CACHE: dict[tuple, tuple] = {}
_LOCK_REF: tuple | None = None


def load_azure_credentials() -> None:
    missing = []
    userdata_get = None
    if IN_COLAB:
        from google.colab import userdata
        userdata_get = userdata.get
    for key in AZURE_SECRET_KEYS:
        if os.environ.get(key):
            continue
        value = None
        if userdata_get is not None:
            try:
                value = userdata_get(key)
            except Exception:
                value = None
        if value:
            os.environ[key] = str(value)
        else:
            missing.append(key)
    if missing:
        raise EnvironmentError(
            "Azure credentials missing: " + ", ".join(missing)
            + ". Add them as Colab secrets (🔑) with notebook access."
        )


def _raw_paths() -> tuple[Path, Path]:
    raw = WORKSPACE_ROOT / "data" / "raw" / "hdfs" / "HDFS_full.log"
    labels = WORKSPACE_ROOT / "data" / "raw" / "hdfs" / "anomaly_label.csv"
    if not raw.exists() or not labels.exists():
        raise FileNotFoundError(
            f"Need {raw} and {labels}. Upload them to Drive "
            f"{DRIVE_ARTIFACT_ROOT / 'data' / 'raw' / 'hdfs'}/ or place them next to the checkout."
        )
    return raw, labels


def _maybe_smoke_sequences(sequences: dict, labels: dict) -> tuple[dict, dict]:
    cap = int(SMOKE_GRAPH_CAP)
    if not SMOKE or len(sequences) <= cap:
        return sequences, labels
    from sklearn.model_selection import train_test_split

    ids = list(sequences)
    y = [int(labels[item]) for item in ids]
    keep, _ = train_test_split(
        ids,
        train_size=cap,
        random_state=SPLIT_SEED,
        stratify=y,
    )
    sequences = {key: sequences[key] for key in keep}
    labels = {key: labels[key] for key in keep}
    print(f"[SMOKE] capped prepare to {len(sequences)} blocks")
    return sequences, labels


def prepare_shared() -> dict[str, Any]:
    """Parse → enrich → sequence → structures → stratified split lock."""
    if PREPARE_STATE.get("ready"):
        print("[PREPARE] shared artefacts already in memory")
        return PREPARE_STATE
    work = Path(CAMPAIGN_DIR)
    work.mkdir(parents=True, exist_ok=True)
    staging = work / "_prepare"
    staging.mkdir(parents=True, exist_ok=True)
    stage_drive_prepare_artifacts()
    drive_lock = DRIVE_ARTIFACT_ROOT / "campaigns" / CAMPAIGN_ID / "split_lock.npz"
    if IN_COLAB and drive_lock.exists() and not (work / "split_lock.npz").exists():
        shutil.copy2(drive_lock, work / "split_lock.npz")
        print("[DRIVE] reuse split_lock.npz")
    drain_ini = staging / "drain.ini"
    drain_ini.write_text(DRAIN_INI_TEXT)
    raw_path, labels_path = _raw_paths()

    templates_path = staging / "templates.json"
    sequences_path = staging / "sequences.parquet"
    structures_path = staging / "graph_structure.pkl"
    lock_path = work / "split_lock.npz"

    if not templates_path.exists():
        print(f"[PREPARE] Drain fit_on=all → {raw_path}")
        parser = DrainParser(config_path=str(drain_ini), persistence_path=str(staging / "drain_parser.bin"))
        parser.fit_file(str(raw_path))
        frame = parser.annotate_file(str(raw_path), unmatched="skip")
        parser.export_templates(str(templates_path))
        parser.save()
        annotated_path = staging / "annotated.parquet"
        frame.to_parquet(annotated_path, index=False)
        push_prepare_artifacts(reason="drain")
    else:
        print(f"[PREPARE] reusing templates {templates_path}")
        frame = None
        push_prepare_artifacts(reason="drain-reuse")

    templates = json.loads(templates_path.read_text())
    provenance_path = staging / "enrichment_provenance.json"
    needs_llm = any(spec["llm"] for spec in ARM_SPECS.values())
    if needs_llm and not all(isinstance(item.get("enriched_large"), dict) for item in templates if int(item.get("cluster_id", 0)) >= 0):
        load_azure_credentials()
        print(f"[PREPARE] Deepseek enrichment of {len(templates)} templates")
        enrich_templates(templates, "hdfs", model_size="large", enrichment_profile="grounded")
        templates_path.write_text(json.dumps(templates, indent=2))
        provenance = enrichment_provenance(
            templates,
            dataset="hdfs",
            enabled=True,
            deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO"),
        )
        provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True))
    elif needs_llm:
        print("[PREPARE] reusing frozen enriched_large fields")
        if not provenance_path.exists():
            provenance = enrichment_provenance(
                templates,
                dataset="hdfs",
                enabled=True,
                deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_DEEPSEEK_V4_PRO") or "cached",
            )
            provenance_path.write_text(json.dumps(provenance, indent=2, sort_keys=True))
    if needs_llm:
        require_complete_enrichment(templates, field="enriched_large")
        require_valid_enrichment_provenance(
            templates,
            json.loads(provenance_path.read_text()),
            dataset="hdfs",
        )
        push_prepare_artifacts(reason="enrichment")

    if not sequences_path.exists():
        if frame is None:
            frame = pd.read_parquet(staging / "annotated.parquet")
        sequences = build_sequences(frame, "hdfs")
        flat = pd.concat(sequences.values(), ignore_index=True)
        save_sequences(flat, sequences_path)
        push_prepare_artifacts(reason="sequences")
    _, sequences = load_sequences(sequences_path, "hdfs")
    mapping = hdfs_label_mapping(labels_path)
    missing = [seq_id for seq_id in sequences if str(seq_id) not in mapping]
    if missing:
        preview = ", ".join(str(item) for item in missing[:5])
        raise KeyError(
            f"{len(missing)} HDFS sequence(s) are missing from {labels_path} (e.g. {preview})."
        )
    labels = {seq_id: mapping[str(seq_id)] for seq_id in sequences}
    sequences, labels = _maybe_smoke_sequences(sequences, labels)

    if not structures_path.exists():
        print(f"[PREPARE] building {len(sequences)} collapsed structures")
        structures, stats = build_graph_structures(
            sequences,
            labels,
            dataset="hdfs",
            hdfs_feature_contract="notebook_raw_v1",
            on_graph_error="fail",
        )
        save_graph_structures(
            structures_path,
            structures,
            {"n_raw": len(sequences), "n_unique": len(sequences), **stats, "dataset": "hdfs"},
        )
        push_prepare_artifacts(reason="structures")
    else:
        structures, _ = load_graph_structures(structures_path)
        print(f"[PREPARE] reusing {len(structures)} structures")

    sequence_ids = [str(item["seq_id"]) for item in structures]
    if lock_path.exists() and not SMOKE:
        lock = load_split_lock(lock_path)
        idx_train, idx_val, idx_test = lock["idx_train"], lock["idx_val"], lock["idx_test"]
        lock_id = lock["split_lock_id"]
        if lock.get("protocol") not in {None, "stratified"}:
            raise ValueError(f"Existing split lock protocol is {lock.get('protocol')!r}, expected stratified")
        print(f"[PREPARE] reusing split_lock {lock_id}")
    else:
        ys = np.array([int(item["y"]) for item in structures])
        idx_train, idx_val, idx_test = split_label_indices(ys, seed=SPLIT_SEED)
        lock_id = save_split_lock(
            lock_path,
            idx_train,
            idx_val,
            idx_test,
            sequence_ids=sequence_ids,
            seed=SPLIT_SEED,
            protocol="stratified",
        )
        print(f"[PREPARE] wrote stratified split_lock {lock_id}  n={len(idx_train)}/{len(idx_val)}/{len(idx_test)}")
    push_to_drive(lock_path)

    templates_data, _, cluster_to_enriched = load_enriched_templates(templates_path, preferred_size="large")
    PREPARE_STATE.update(
        {
            "ready": True,
            "templates_data": templates_data,
            "cluster_to_enriched": cluster_to_enriched,
            "structures": structures,
            "idx_train": np.asarray(idx_train, dtype=np.int64),
            "idx_val": np.asarray(idx_val, dtype=np.int64),
            "idx_test": np.asarray(idx_test, dtype=np.int64),
            "lock_id": lock_id,
            "lock_path": lock_path,
            "work": work,
            "staging": staging,
            "provenance_path": provenance_path if provenance_path.exists() else None,
        }
    )
    return PREPARE_STATE


def _embeddings_for(spec: dict[str, Any]):
    key = (spec["llm"], spec["tfidf"], spec["sbert"], spec["sbert_text"])
    if key in _EMBED_CACHE:
        return _EMBED_CACHE[key]
    enriched = PREPARE_STATE["cluster_to_enriched"] if spec["llm"] else {}
    embeddings, cluster_ids, _ = compute_embeddings(
        PREPARE_STATE["templates_data"],
        enriched,
        tfidf_enabled=bool(spec["tfidf"]),
        sbert_enabled=bool(spec["sbert"]),
        tfidf_fit_texts=None,
        sbert_text=str(spec["sbert_text"]),
    )
    cluster_embeddings = {
        int(cluster_id): embedding
        for cluster_id, embedding in zip(cluster_ids, embeddings, strict=True)
    }
    _EMBED_CACHE[key] = (embeddings, cluster_embeddings)
    return _EMBED_CACHE[key]


def prepare_arm(arm: str) -> Path:
    if arm not in ARM_SPECS:
        raise KeyError(arm)
    spec = ARM_SPECS[arm]
    bundle_dir = Path(CAMPAIGN_DIR) / "graphs" / arm
    compressed = bundle_dir / "graph_dataset.pt.gz"
    meta_path = bundle_dir / "dataset_meta.json"
    if compressed.exists() and meta_path.exists() and not SMOKE:
        print(f"[PREPARE] {arm} already present → {compressed}")
        push_campaign_arm(arm)
        return compressed
    state = prepare_shared()
    embeddings, cluster_embeddings = _embeddings_for(spec)
    data_list = attach_cluster_embeddings(
        state["structures"],
        cluster_embeddings,
        use_edge_features=bool(spec["use_edges"]),
        include_node_positional_features=bool(spec["node_pos"]),
        include_edge_temporal_features=bool(spec["edge_temp"]),
        include_edge_positional_features=bool(spec["edge_pos"]),
        missing_embedding="fail" if spec["sbert"] or spec["tfidf"] else "zero",
    )
    node_dim = int(data_list[0].x.size(-1))
    edge_dim = int(data_list[0].edge_attr.size(-1)) if data_list[0].edge_attr is not None else 1
    embed_dim = int(embeddings.shape[1])
    tfidf_dim, sbert_dim = embedding_block_dims(
        embed_dim, tfidf_enabled=bool(spec["tfidf"]), sbert_enabled=bool(spec["sbert"])
    )
    identity = {
        "llm_enrichment_enabled": bool(spec["llm"]),
        "enrichment_model_size": "large" if spec["llm"] else None,
        "tfidf_enabled": bool(spec["tfidf"]),
        "sbert_enabled": bool(spec["sbert"]),
        "use_edge_features": bool(spec["use_edges"]),
        "feature_contract": "notebook_raw_v1",
        "unique_sequences": False,
        "fit_on": "all",
        "split_protocol": "stratified",
        "sbert_text": spec["sbert_text"],
        "node_positional_features": bool(spec["node_pos"]),
        "edge_temporal_features": bool(spec["edge_temp"]),
        "edge_positional_features": bool(spec["edge_pos"]),
    }
    dataset_meta = {
        "dataset": "hdfs",
        "n_total": len(data_list),
        "n_train": int(len(state["idx_train"])),
        "n_val": int(len(state["idx_val"])),
        "n_test": int(len(state["idx_test"])),
        "node_dim": node_dim,
        "edge_dim": edge_dim,
        "embed_dim": embed_dim,
        "tfidf_dim": tfidf_dim,
        "sbert_dim": sbert_dim,
        "embedding_flags": {
            "tfidf_enabled": bool(spec["tfidf"]),
            "sbert_enabled": bool(spec["sbert"]),
            "sbert_text": spec["sbert_text"],
        },
        "use_edge_features": bool(spec["use_edges"]),
        "llm_enrichment_enabled": bool(spec["llm"]),
        "enrichment_model_size": "large" if spec["llm"] else None,
        "feature_contract": "notebook_raw_v1",
        "graph_identity": identity,
        "split_lock_id": state["lock_id"],
        "unique_sequences": False,
        "split_protocol": "stratified",
        "fit_on": "all",
        "oov_graph_rate": 0.0,
    }
    bundle_dir.mkdir(parents=True, exist_ok=True)
    raw_pt = bundle_dir / "graph_dataset.pt"
    save_graph_dataset(
        data_list,
        state["idx_train"],
        state["idx_val"],
        state["idx_test"],
        raw_pt,
        node_dim=node_dim,
        edge_dim=edge_dim,
        embed_dim=embed_dim,
        dataset_meta=dataset_meta,
    )
    gzip_file(raw_pt, compressed)
    meta_path.write_text(json.dumps(dataset_meta, indent=2, default=str))
    raw_pt.unlink(missing_ok=True)
    _update_manifest_entry(arm, compressed, dataset_meta, identity)
    print(f"[PREPARE] {arm} → {compressed}  node_dim={node_dim} edge_dim={edge_dim}")
    push_campaign_arm(arm)
    return compressed


def _update_manifest_entry(arm: str, compressed: Path, dataset_meta: dict, identity: dict) -> None:
    campaign_dir = Path(CAMPAIGN_DIR)
    manifest_path = campaign_dir / "manifest.json"
    previous = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    graphs = {
        str(item.get("name")): item
        for item in previous.get("graphs", [])
        if isinstance(item, dict) and item.get("name")
    }
    graphs[arm] = {
        "name": arm,
        "graph_dataset": str(compressed.relative_to(campaign_dir)),
        "dataset_meta": f"graphs/{arm}/dataset_meta.json",
        "identity": identity,
        "node_dim": dataset_meta.get("node_dim"),
    }
    write_campaign_manifest(
        campaign_dir,
        campaign_id=CAMPAIGN_ID,
        dataset="hdfs",
        family="representation",
        graphs=[graphs[name] for name in sorted(graphs)],
        split_lock=campaign_dir / "split_lock.npz",
        extra={
            "seeds": list(SEEDS),
            "fit_on": "all",
            "split_protocol": "stratified",
            "split_lock_id": dataset_meta.get("split_lock_id"),
        },
    )


def run_prepare_arm(arm: str) -> None:
    if not RUN_PREPARE:
        print(f"[PREPARE] skipped for {arm} (RUN_PREPARE=False)")
        return
    prepare_arm(arm)


## Inlined AttributeAwareGAE + train / eval / report

Same clean-train loop as the BGL Full Monty notebook (val-F1 checkpoint, frozen test threshold). Learning rate is the HDFS value `0.01`. Outputs go to `outputs/hdfs/{campaign}_{arm}_seed{seed}/`. After each seed, Colab copies that folder (including `figures/confusion_matrix.png`, `learning_curve.png`, `test_pr_curve.png`, and `test_score_distribution.png` with the threshold) to Drive.


In [ ]:
"""Attribute-Aware Graph Autoencoder (AttributeAwareGAE).

Architecture extracted from ``6_GAE_Training_BGL_fixed.ipynb`` and corrected
so structure reconstruction matches a *directed*, *per-graph* adjacency:

Encoder
    * ``raw_node_norm`` — BatchNorm1d on input node features.
    * ``node_proj`` + ``edge_proj`` — linear projections to ``hidden_dim``.
    * ``encoder_conv`` — GINEConv with configurable aggregation.

Decoder (multi-task)
    1. Structure — directed concat-MLP (default) or inner product. Trained
       with BCE on observed edges and *in-graph* non-edges (never cross-graph
       pairs from a PyG mini-batch).
    2. Node feature reconstruction — 2-layer MLP from latent Z onto the
       reconstruction columns (full ``x`` or TF-IDF+extras when
       ``node_reconstruct=without_sbert``).
    3. Edge attribute reconstruction — 2-layer MLP from ⟨Z_i ∥ Z_j⟩.

``raw_edge_norm`` is intentionally absent: BGL's ``log1p(td_std)`` is ~0 for
most edges and in-model BatchNorm would explode. Edges are pre-normalised
on the training split with std clamped ≥ 0.1.
"""

from __future__ import annotations

_LOCK_REF = globals().get('_LOCK_REF')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv
from torch_geometric.utils import negative_sampling, scatter

FULL_STRUCTURE_MAX_NODES = 256
NODE_EXTRA_DIM = 9


def node_recon_index_tensor(
    node_dim: int,
    *,
    sbert_dim: int,
    extra_dim: int = NODE_EXTRA_DIM,
) -> torch.Tensor:
    """Column indices for the node decoder when skipping the SBERT block."""
    if sbert_dim <= 0:
        return torch.arange(node_dim, dtype=torch.long)
    embed_dim = node_dim - extra_dim
    if embed_dim < sbert_dim:
        raise ValueError(
            f"sbert_dim={sbert_dim} exceeds embedding width {embed_dim} "
            f"(node_dim={node_dim}, extra_dim={extra_dim})."
        )
    tfidf_dim = embed_dim - sbert_dim
    lexical = torch.arange(tfidf_dim, dtype=torch.long)
    extras = torch.arange(embed_dim, node_dim, dtype=torch.long)
    return torch.cat([lexical, extras])


class AttributeAwareGAE(nn.Module):
    """Multi-task Graph Autoencoder with a GINEConv encoder.

    Parameters
    ----------
    structure_decoder : str
        ``"mlp"`` (directed concat-MLP, default) or ``"inner_product"``
        (symmetric ⟨z_i, z_j⟩, Family B ablation).
    """

    def __init__(
        self,
        node_dim: int,
        edge_dim: int,
        hidden_dim: int = 128,
        latent_dim: int = 64,
        gine_aggregation: str = "sum",
        node_transformation: str = "mlp",
        structure_decoder: str = "mlp",
        node_recon_index: torch.Tensor | None = None,
        tfidf_dim: int = 0,
        sbert_dim: int = 0,
        fusion_mode: str = "concat",
        modality_projection_dim: int = 64,
        node_loss_mode: str = "global",
        node_block_weights: dict[str, float] | None = None,
    ) -> None:
        super().__init__()
        if structure_decoder not in {"mlp", "inner_product"}:
            raise ValueError(
                f"structure_decoder must be 'mlp' or 'inner_product', got {structure_decoder!r}"
            )
        self.structure_decoder_kind = structure_decoder
        self.latent_dim = latent_dim
        self.node_dim = int(node_dim)
        self.tfidf_dim = int(tfidf_dim)
        self.sbert_dim = int(sbert_dim)
        self.extra_dim = int(node_dim - self.tfidf_dim - self.sbert_dim)
        if self.extra_dim < 0:
            raise ValueError("tfidf_dim + sbert_dim cannot exceed node_dim.")
        if fusion_mode not in {"concat", "projected_gated"}:
            raise ValueError("fusion_mode must be 'concat' or 'projected_gated'.")
        if node_loss_mode not in {"global", "block_balanced"}:
            raise ValueError("node_loss_mode must be 'global' or 'block_balanced'.")
        self.fusion_mode = fusion_mode
        self.node_loss_mode = node_loss_mode
        self.node_block_weights = {
            "tfidf": 1.0,
            "sbert": 1.0,
            "extras": 1.0,
            **(node_block_weights or {}),
        }
        if node_recon_index is None:
            recon_index = torch.arange(node_dim, dtype=torch.long)
        else:
            recon_index = torch.as_tensor(node_recon_index, dtype=torch.long).reshape(-1)
        if recon_index.numel() == 0:
            raise ValueError("node_recon_index must contain at least one column.")
        # Not a state-dict buffer: old packages must keep the historical key set
        # when recon_dim == node_dim. The index is saved on the training checkpoint.
        self.node_recon_index = recon_index
        recon_dim = int(recon_index.numel())

        self.raw_node_norm = nn.BatchNorm1d(node_dim, affine=False)

        self.modality_projectors = nn.ModuleDict()
        self.modality_gate_logits = None
        node_projection_input = node_dim
        if fusion_mode == "projected_gated":
            blocks = self._input_block_ranges()
            if len(blocks) < 2:
                raise ValueError("projected_gated fusion requires at least two non-empty modalities.")
            for name, (start, stop) in blocks.items():
                self.modality_projectors[name] = nn.Sequential(
                    nn.Linear(stop - start, modality_projection_dim),
                    nn.LayerNorm(modality_projection_dim),
                    nn.ReLU(),
                )
            self.modality_gate_logits = nn.Parameter(torch.zeros(len(blocks)))
            node_projection_input = modality_projection_dim * len(blocks)

        self.node_proj = nn.Linear(node_projection_input, hidden_dim)
        self.edge_proj = nn.Linear(edge_dim, hidden_dim)

        if node_transformation == "mlp":
            nn_module: nn.Module = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, latent_dim),
            )
        else:
            nn_module = nn.Linear(hidden_dim, latent_dim)

        self.encoder_conv = GINEConv(nn_module, edge_dim=hidden_dim, aggr=gine_aggregation)

        self.structure_decoder = None
        if structure_decoder == "mlp":
            self.structure_decoder = nn.Sequential(
                nn.Linear(latent_dim * 2, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, 1),
            )

        self.node_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, recon_dim),
        )
        self.edge_decoder = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_dim),
        )

    def node_reconstruction_target(self, x_norm: torch.Tensor) -> torch.Tensor:
        """Select the decoder target columns from BatchNorm-scaled node features."""
        index = self.node_recon_index
        if index.device != x_norm.device:
            index = index.to(device=x_norm.device)
            self.node_recon_index = index
        return x_norm.index_select(1, index)

    def _input_block_ranges(self) -> dict[str, tuple[int, int]]:
        ranges: dict[str, tuple[int, int]] = {}
        cursor = 0
        if self.tfidf_dim:
            ranges["tfidf"] = (cursor, cursor + self.tfidf_dim)
            cursor += self.tfidf_dim
        if self.sbert_dim:
            ranges["sbert"] = (cursor, cursor + self.sbert_dim)
            cursor += self.sbert_dim
        if self.extra_dim:
            ranges["extras"] = (cursor, cursor + self.extra_dim)
        return ranges

    def project_node_inputs(self, x_norm: torch.Tensor) -> torch.Tensor:
        """Legacy concat or gated, equally sized modality projections."""
        if self.fusion_mode == "concat":
            return x_norm
        assert self.modality_gate_logits is not None
        gates = torch.softmax(self.modality_gate_logits, dim=0)
        projected = []
        for gate, (name, (start, stop)) in zip(
            gates, self._input_block_ranges().items(), strict=True
        ):
            projected.append(gate * self.modality_projectors[name](x_norm[:, start:stop]))
        return torch.cat(projected, dim=1)

    def node_block_errors(
        self, x_rec: torch.Tensor, target: torch.Tensor
    ) -> dict[str, torch.Tensor]:
        """Per-node MSE for each reconstructed modality block."""
        squared = F.mse_loss(x_rec, target, reduction="none")
        original = self.node_recon_index.to(squared.device)
        result: dict[str, torch.Tensor] = {}
        for name, (start, stop) in self._input_block_ranges().items():
            positions = torch.nonzero(
                (original >= start) & (original < stop), as_tuple=False
            ).reshape(-1)
            if positions.numel():
                result[name] = squared.index_select(1, positions).mean(dim=1)
        return result

    def node_reconstruction_loss(
        self, x_rec: torch.Tensor, target: torch.Tensor
    ) -> torch.Tensor:
        if self.node_loss_mode == "global":
            return F.mse_loss(x_rec, target)
        blocks = self.node_block_errors(x_rec, target)
        weighted = [
            float(self.node_block_weights[name]) * values.mean()
            for name, values in blocks.items()
            if float(self.node_block_weights[name]) > 0
        ]
        weight_sum = sum(
            float(self.node_block_weights[name])
            for name in blocks
            if float(self.node_block_weights[name]) > 0
        )
        if not weighted or weight_sum <= 0:
            raise ValueError("At least one reconstructed node block must have positive weight.")
        return torch.stack(weighted).sum() / weight_sum

    def node_anomaly_error(self, x_rec: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Per-node counterpart of the configured training objective."""
        if self.node_loss_mode == "global":
            return F.mse_loss(x_rec, target, reduction="none").mean(dim=1)
        blocks = self.node_block_errors(x_rec, target)
        weighted = [
            float(self.node_block_weights[name]) * values
            for name, values in blocks.items()
            if float(self.node_block_weights[name]) > 0
        ]
        weight_sum = sum(
            float(self.node_block_weights[name])
            for name in blocks
            if float(self.node_block_weights[name]) > 0
        )
        return torch.stack(weighted).sum(dim=0) / weight_sum

    def standardize_inputs(
        self,
        x: torch.Tensor,
        edge_attr: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Apply node BN; pass edge features through unchanged (pre-normalised)."""
        x_norm = self.raw_node_norm(x)
        if edge_attr is not None and edge_attr.numel() > 0 and edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(1)
        return x_norm, edge_attr

    def encode(
        self,
        x_norm: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr_norm: torch.Tensor | None,
    ) -> torch.Tensor:
        x_h = self.node_proj(self.project_node_inputs(x_norm))
        edge_h = self.edge_proj(edge_attr_norm) if edge_attr_norm is not None else None
        return self.encoder_conv(x_h, edge_index, edge_h)

    def decode_structure(self, z: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """Directed (MLP) or symmetric (inner-product) logits for edge pairs."""
        src, dst = edge_index
        if self.structure_decoder is None:
            return (z[src] * z[dst]).sum(dim=1)
        return self.structure_decoder(torch.cat([z[src], z[dst]], dim=-1)).squeeze(-1)

    def decode_node_features(self, z: torch.Tensor) -> torch.Tensor:
        return self.node_decoder(z)

    def decode_edge_attributes(self, z: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        src, dst = edge_index
        return self.edge_decoder(torch.cat([z[src], z[dst]], dim=-1))

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: torch.Tensor | None,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor | None]:
        """Return ``(z, x_norm, edge_attr_norm)`` for use in loss computation."""
        x_norm, edge_attr_norm = self.standardize_inputs(x, edge_attr)
        z = self.encode(x_norm, edge_index, edge_attr_norm)
        return z, x_norm, edge_attr_norm


def per_graph_negative_sampling(edge_index: torch.Tensor, ptr: torch.Tensor) -> torch.Tensor:
    """Sample one non-edge per observed edge, restricted to that graph's nodes.

    Mini-batch ``negative_sampling(..., num_nodes=batch.num_nodes)`` treats the
    disjoint union as one graph and yields trivial cross-graph negatives.
    """
    if edge_index.size(1) == 0 or ptr.numel() < 2:
        return edge_index.new_zeros((2, 0))
    src, dst = edge_index[0], edge_index[1]
    chunks: list[torch.Tensor] = []
    n_graphs = int(ptr.numel() - 1)
    for graph in range(n_graphs):
        lo = int(ptr[graph].item())
        hi = int(ptr[graph + 1].item())
        n_nodes = hi - lo
        mask = (src >= lo) & (src < hi)
        pos = edge_index[:, mask]
        if pos.size(1) == 0 or n_nodes <= 0:
            continue
        local = pos - lo
        neg_local = negative_sampling(
            local,
            num_nodes=n_nodes,
            num_neg_samples=pos.size(1),
        )
        if neg_local.numel() == 0:
            continue
        chunks.append(neg_local + lo)
    if not chunks:
        return edge_index.new_zeros((2, 0))
    return torch.cat(chunks, dim=1)


def complete_directed_index(num_nodes: int, device: torch.device) -> torch.Tensor:
    """All directed pairs including self-loops (collapsed graphs may loop)."""
    src = torch.arange(num_nodes, device=device).repeat_interleave(num_nodes)
    dst = torch.arange(num_nodes, device=device).repeat(num_nodes)
    return torch.stack([src, dst], dim=0)


def graph_ptr(batch, num_graphs: int, device: torch.device) -> torch.Tensor:
    """Node-offset pointer tensor, reconstructed from ``batch.batch`` if needed."""
    ptr = getattr(batch, "ptr", None)
    if ptr is not None:
        return ptr
    counts = torch.bincount(batch.batch, minlength=num_graphs)
    out = torch.zeros(num_graphs + 1, dtype=torch.long, device=device)
    out[1:] = torch.cumsum(counts, dim=0)
    return out


def _structure_terms(
    model: AttributeAwareGAE,
    z: torch.Tensor,
    pos_index: torch.Tensor,
    neg_index: torch.Tensor | None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (mean pos BCE, mean neg BCE) as logits; zeros when a set is empty."""
    device = z.device
    pos_loss = torch.tensor(0.0, device=device)
    neg_loss = torch.tensor(0.0, device=device)
    if pos_index.size(1) > 0:
        pos_logits = model.decode_structure(z, pos_index)
        pos_loss = F.binary_cross_entropy_with_logits(
            pos_logits, torch.ones_like(pos_logits)
        )
    if neg_index is not None and neg_index.size(1) > 0:
        neg_logits = model.decode_structure(z, neg_index)
        neg_loss = F.binary_cross_entropy_with_logits(
            neg_logits, torch.zeros_like(neg_logits)
        )
    return pos_loss, neg_loss


def _structure_error_vector(
    model: AttributeAwareGAE,
    z: torch.Tensor,
    edge_index: torch.Tensor,
    ptr: torch.Tensor,
    *,
    include_non_edges: bool,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    """Per-graph structure error: pos BCE (+ in-graph non-edge BCE when requested)."""
    device = z.device
    num_graphs = int(ptr.numel() - 1)
    scores = torch.zeros(num_graphs, device=device)
    src = edge_index[0]
    for graph in range(num_graphs):
        lo = int(ptr[graph].item())
        hi = int(ptr[graph + 1].item())
        n_nodes = hi - lo
        mask = (src >= lo) & (src < hi) if edge_index.size(1) else None
        pos = edge_index[:, mask] if mask is not None else edge_index.new_zeros((2, 0))
        neg = None
        if include_non_edges and n_nodes > 0:
            neg = _in_graph_non_edges(pos, lo, n_nodes, device, generator)
        pos_loss, neg_loss = _structure_terms(model, z, pos, neg)
        scores[graph] = pos_loss + (neg_loss if include_non_edges else torch.tensor(0.0, device=device))
    return scores


def _in_graph_non_edges(
    pos: torch.Tensor,
    lo: int,
    n_nodes: int,
    device: torch.device,
    generator: torch.Generator | None,
) -> torch.Tensor:
    """Non-edges inside one graph: full digraph when small, else sampled."""
    del generator  # sampling uses PyG's RNG; seed the process for campaigns
    if n_nodes <= 0:
        return pos.new_zeros((2, 0))
    if n_nodes <= FULL_STRUCTURE_MAX_NODES:
        complete = complete_directed_index(n_nodes, device)
        adj = torch.zeros((n_nodes, n_nodes), dtype=torch.bool, device=device)
        if pos.size(1):
            adj[pos[0] - lo, pos[1] - lo] = True
        keep = ~adj[complete[0], complete[1]]
        return complete[:, keep] + lo
    local = pos - lo if pos.size(1) else pos.new_zeros((2, 0))
    n_pos = max(int(pos.size(1)), 1)
    neg_local = negative_sampling(
        local,
        num_nodes=n_nodes,
        num_neg_samples=n_pos,
        force_undirected=False,
    )
    if neg_local.numel() == 0:
        return pos.new_zeros((2, 0))
    return neg_local + lo


def train_epoch(
    model: AttributeAwareGAE,
    loader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    *,
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
) -> tuple[float, float, float, float]:
    """Run one full training epoch with per-graph structure negatives."""
    model.train()
    total_loss = total_str = total_node = total_edge = 0.0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
        num_graphs = batch.num_graphs if hasattr(batch, "num_graphs") else 1
        ptr = graph_ptr(batch, num_graphs, device)

        loss_str = torch.tensor(0.0, device=device)
        if batch.edge_index.size(1) > 0:
            pos_logits = model.decode_structure(z, batch.edge_index)
            neg_edge = per_graph_negative_sampling(batch.edge_index, ptr)
            pos_loss = F.binary_cross_entropy_with_logits(
                pos_logits, torch.ones_like(pos_logits)
            )
            if neg_edge.size(1) > 0:
                neg_logits = model.decode_structure(z, neg_edge)
                neg_loss = F.binary_cross_entropy_with_logits(
                    neg_logits, torch.zeros_like(neg_logits)
                )
            else:
                neg_loss = torch.tensor(0.0, device=device)
            loss_str = pos_loss + neg_loss

        x_rec = model.decode_node_features(z)
        loss_node = model.node_reconstruction_loss(
            x_rec, model.node_reconstruction_target(x_norm)
        )

        loss_edge = torch.tensor(0.0, device=device)
        if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
            edge_rec = model.decode_edge_attributes(z, batch.edge_index)
            loss_edge = F.mse_loss(edge_rec, edge_attr_norm)

        loss = alpha * loss_str + beta * loss_node + gamma * loss_edge
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        n = batch.num_graphs
        total_loss += loss.item() * n
        total_str += loss_str.item() * n
        total_node += loss_node.item() * n
        total_edge += loss_edge.item() * n

    ng = len(loader.dataset)
    if ng == 0:
        return 0.0, 0.0, 0.0, 0.0
    return total_loss / ng, total_str / ng, total_node / ng, total_edge / ng


@torch.no_grad()
def compute_anomaly_scores(
    model: AttributeAwareGAE,
    loader,
    device: torch.device,
    *,
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
    return_components: bool = False,
    include_structure_non_edges: bool = True,
) -> tuple:
    """Compute per-graph anomaly scores (weighted reconstruction error).

    Structure error matches training: BCE on observed edges plus in-graph
    non-edges (full directed adjacency on small collapsed graphs). Combined
    scores stay ``α·str + β·node + γ·edge``.
    """
    model.eval()
    all_scores, all_labels = [], []
    all_structure, all_node, all_edge = [], [], []
    all_tfidf, all_sbert, all_extras = [], [], []
    graph_ids: list[str] = []

    for batch in loader:
        batch = batch.to(device)
        z, x_norm, edge_attr_norm = model(batch.x, batch.edge_index, batch.edge_attr)
        num_graphs = batch.num_graphs if hasattr(batch, "num_graphs") else 1
        ptr = graph_ptr(batch, num_graphs, device)

        g_str = _structure_error_vector(
            model,
            z,
            batch.edge_index,
            ptr,
            include_non_edges=include_structure_non_edges,
        )

        x_rec = model.decode_node_features(z)
        target = model.node_reconstruction_target(x_norm)
        node_errors = model.node_anomaly_error(x_rec, target)
        g_node = scatter(
            node_errors, batch.batch, dim=0, reduce="mean", dim_size=num_graphs
        )

        g_edge = torch.zeros(num_graphs, device=device)
        if edge_attr_norm is not None and edge_attr_norm.size(0) > 0:
            ea_errors = F.mse_loss(
                model.decode_edge_attributes(z, batch.edge_index),
                edge_attr_norm,
                reduction="none",
            ).mean(dim=1)
            edge_batch = batch.batch[batch.edge_index[0]]
            g_edge = scatter(
                ea_errors, edge_batch, dim=0, reduce="mean", dim_size=num_graphs
            )

        g_str = torch.nan_to_num(g_str, 0.0)
        g_edge = torch.nan_to_num(g_edge, 0.0)

        total = alpha * g_str + beta * g_node + gamma * g_edge
        all_scores.append(total.cpu())
        all_labels.append(batch.y.cpu())
        if return_components:
            all_structure.append(g_str.cpu())
            all_node.append(g_node.cpu())
            all_edge.append(g_edge.cpu())
            block_errors = model.node_block_errors(x_rec, target)
            for name, destination in (
                ("tfidf", all_tfidf),
                ("sbert", all_sbert),
                ("extras", all_extras),
            ):
                values = block_errors.get(name)
                if values is None:
                    destination.append(torch.zeros(num_graphs))
                else:
                    destination.append(
                        scatter(
                            values, batch.batch, dim=0, reduce="mean", dim_size=num_graphs
                        ).cpu()
                    )
            graph_ids.extend(_batch_graph_ids(batch, num_graphs))

    scores = torch.cat(all_scores).numpy()
    labels = torch.cat(all_labels).numpy()
    if not return_components:
        return scores, labels
    zeros = scores * 0.0
    components = {
        "structure": torch.cat(all_structure).numpy() if all_structure else zeros,
        "node": torch.cat(all_node).numpy() if all_node else zeros,
        "edge": torch.cat(all_edge).numpy() if all_edge else zeros,
        "tfidf": torch.cat(all_tfidf).numpy() if all_tfidf else zeros,
        "sbert": torch.cat(all_sbert).numpy() if all_sbert else zeros,
        "extras": torch.cat(all_extras).numpy() if all_extras else zeros,
    }
    return scores, labels, components, graph_ids


def _batch_graph_ids(batch, num_graphs: int) -> list[str]:
    """Return per-graph identifiers from a PyG batch when available."""
    for attribute in ("block_id", "window_id", "sequence_id", "graph_id"):
        if hasattr(batch, attribute):
            raw = getattr(batch, attribute)
            if raw is None:
                continue
            if hasattr(raw, "tolist"):
                values = raw.tolist()
            elif isinstance(raw, (list, tuple)):
                values = list(raw)
            else:
                values = [raw]
            if len(values) == num_graphs:
                return [str(value) for value in values]
    return [str(index) for index in range(num_graphs)]


# ---------------------------------------------------------------------------
# Campaign helpers inlined from run_ablation.py, dataset.py, ablation.py,
# classical_baseline.py, and utils.py. No src.modules imports.
# ---------------------------------------------------------------------------

# Inlined helpers from run_ablation.py, src/modules/ablation.py,
# src/modules/dataset.py, src/modules/classical_baseline.py, src/modules/utils.py.
# This fragment is concatenated into Ablation_Full_Monty.ipynb and then deleted.


import copy
import gzip
import json
import random
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:  # Colab runtimes sometimes omit PyYAML
    yaml = None


BGL_SPLIT_STRIDE = 1 << 40
NODE_EXTRA_DIM = 9
SBERT_EMBED_DIM = 384

TRAINING = {
    "train_mode": "clean",
    "test_run": False,
    "test_samples": 5000,
    "hidden_dim": 128,
    "latent_dim": 64,
    "batch_size": 256,
    "epochs": 25,
    "learning_rate": 0.01,
    "alpha": 1.0,
    "beta": 1.0,
    "gamma": 1.0,
    "pre_normalize_edges": True,
    "minimum_edge_std": 0.1,
}

GAE_ARCH = {
    "gine_aggregation": "sum",
    "node_transformation": "mlp",
    "structure_decoder": "mlp",
    "node_reconstruct": "all",
    "fusion_mode": "concat",
    "modality_projection_dim": 64,
    "node_loss_mode": "global",
    "node_block_weights": {"tfidf": 1.0, "sbert": 1.0, "extras": 1.0},
}

EXPECTED_ARMS = (
    "hybrid_llm",
    "hybrid_raw",
    "tfidf_only",
    "sbert_raw",
    "sbert_llm",
    "no_positional_features",
    "no_temporal_features",
    "no_temporal_or_positional_features",
)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _safe_name(value: str) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_.-]+", "-", value.strip())
    if not cleaned:
        raise ValueError("Run names must contain at least one letter or number.")
    return cleaned.strip(".-")


def _rounded(value: float) -> float:
    return round(float(value), 6)


def graph_split_dir(path: str | Path) -> Path:
    path = Path(path)
    name = path.name
    if name.endswith(".gz"):
        name = name[:-3]
    stem = Path(name).stem
    return path.parent / f"{stem}_splits"


def gunzip_file(source: str | Path, destination: str | Path | None = None) -> Path:
    source = Path(source)
    if destination is None:
        if source.suffix != ".gz":
            return source
        destination = source.with_suffix("")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(source, "rb") as incoming, destination.open("wb") as outgoing:
        shutil.copyfileobj(incoming, outgoing, length=16 * 1024 * 1024)
    return destination


def load_bundle_meta(graph_path: str | Path) -> dict[str, Any] | None:
    graph_path = Path(graph_path)
    sidecar = graph_path.with_name("dataset_meta.json")
    if not sidecar.exists() and graph_path.suffix == ".gz":
        sidecar = graph_path.with_suffix("").with_name("dataset_meta.json")
    if sidecar.exists():
        return json.loads(sidecar.read_text())
    parent_meta = graph_path.parent / "dataset_meta.json"
    if parent_meta.exists():
        return json.loads(parent_meta.read_text())
    try:
        bundle = torch.load(graph_path, map_location="cpu", weights_only=False)
    except Exception:
        return None
    if not isinstance(bundle, dict):
        return None
    meta = bundle.get("dataset_meta")
    if isinstance(meta, dict):
        return meta
    extracted = {
        key: bundle[key]
        for key in (
            "node_dim",
            "edge_dim",
            "embed_dim",
            "feature_contract",
            "llm_enrichment_enabled",
            "enrichment_model_size",
            "use_edge_features",
            "embedding_flags",
            "graph_identity",
            "split_lock_id",
        )
        if key in bundle
    }
    return extracted or None


def load_graph_splits(path: str | Path) -> tuple[list, list, list, dict[str, Any]]:
    path = Path(path)
    split_dir = graph_split_dir(path)
    train_path = split_dir / "train.pt"
    if train_path.exists() and (split_dir / "val.pt").exists() and (split_dir / "test.pt").exists():
        train_graphs = torch.load(train_path, weights_only=False, map_location="cpu")
        val_graphs = torch.load(split_dir / "val.pt", weights_only=False, map_location="cpu")
        test_graphs = torch.load(split_dir / "test.pt", weights_only=False, map_location="cpu")
        meta: dict[str, Any] = {}
        meta_path = split_dir / "meta.json"
        if meta_path.exists():
            meta = json.loads(meta_path.read_text())
        sidecar = path.with_name("dataset_meta.json")
        if sidecar.exists():
            meta.setdefault("dataset_meta", json.loads(sidecar.read_text()))
        return list(train_graphs), list(val_graphs), list(test_graphs), meta

    bundle = torch.load(path, weights_only=False, map_location="cpu")
    all_data = bundle["data_list"]
    train_graphs = [all_data[int(i)] for i in bundle["idx_train"]]
    val_graphs = [all_data[int(i)] for i in bundle["idx_val"]]
    test_graphs = [all_data[int(i)] for i in bundle["idx_test"]]
    meta = {
        "node_dim": bundle.get("node_dim"),
        "edge_dim": bundle.get("edge_dim"),
        "embed_dim": bundle.get("embed_dim"),
        "dataset_meta": bundle.get("dataset_meta") or {},
    }
    return train_graphs, val_graphs, test_graphs, meta


def sequence_ids_from_graphs(data_list: list) -> list[str]:
    ids: list[str] = []
    for index, graph in enumerate(data_list):
        value = None
        for attribute in ("block_id", "window_id", "sequence_id", "graph_id"):
            if hasattr(graph, attribute):
                raw = getattr(graph, attribute)
                value = raw.item() if hasattr(raw, "item") else raw
                break
        ids.append(str(index if value is None else value))
    return ids


def embedding_block_dims(
    embed_dim: int,
    *,
    tfidf_enabled: bool,
    sbert_enabled: bool,
    sbert_width: int = SBERT_EMBED_DIM,
) -> tuple[int, int]:
    if tfidf_enabled and sbert_enabled:
        if embed_dim <= sbert_width:
            raise ValueError(
                f"Hybrid embed_dim={embed_dim} is too small for MiniLM width {sbert_width}."
            )
        return int(embed_dim - sbert_width), int(sbert_width)
    if sbert_enabled:
        return 0, int(embed_dim)
    return int(embed_dim), 0


def sbert_dim_from_meta(meta: Mapping[str, Any] | None, node_dim: int | None = None) -> int:
    if not meta:
        return 0
    if meta.get("sbert_dim") is not None:
        return int(meta["sbert_dim"])
    flags = meta.get("embedding_flags") or {}
    identity = meta.get("graph_identity") or {}
    sbert_enabled = bool(flags.get("sbert_enabled", identity.get("sbert_enabled", False)))
    tfidf_enabled = bool(flags.get("tfidf_enabled", identity.get("tfidf_enabled", True)))
    embed_dim = int(meta.get("embed_dim") or 0)
    if embed_dim <= 0:
        width = int(node_dim or meta.get("node_dim") or 0)
        embed_dim = max(width - NODE_EXTRA_DIM, 0)
    if embed_dim <= 0 or not sbert_enabled:
        return 0
    _, sbert_dim = embedding_block_dims(
        embed_dim, tfidf_enabled=tfidf_enabled, sbert_enabled=True
    )
    return sbert_dim


def flatten_split_meta(
    split_meta: Mapping[str, Any] | None,
    bundle_meta: Mapping[str, Any] | None,
) -> dict[str, Any]:
    merged: dict[str, Any] = {}
    if isinstance(split_meta, Mapping):
        nested = split_meta.get("dataset_meta")
        if isinstance(nested, Mapping):
            merged.update(dict(nested))
        for key, value in split_meta.items():
            if key != "dataset_meta" and value is not None:
                merged.setdefault(key, value)
    if isinstance(bundle_meta, Mapping):
        for key, value in bundle_meta.items():
            merged.setdefault(key, value)
    return merged


def window_feature_matrices(
    train_graphs: list[Any],
    *splits: list[Any],
    oov_cluster_id: int = -1,
) -> tuple[np.ndarray, ...]:
    vocabulary = sorted(
        {
            int(cluster_id)
            for graph in train_graphs
            for cluster_id in _cluster_ids(graph)
            if int(cluster_id) >= 0
        }
    )
    if not vocabulary:
        raise ValueError("Isolation Forest baseline found no known training templates.")
    index = {cluster_id: position for position, cluster_id in enumerate(vocabulary)}

    def transform(graphs: list[Any]) -> np.ndarray:
        matrix = np.zeros((len(graphs), len(vocabulary) + 2), dtype=np.float32)
        for row, graph in enumerate(graphs):
            cluster_ids = _cluster_ids(graph)
            n_events = len(cluster_ids)
            if not n_events:
                continue
            for cluster_id in cluster_ids:
                if cluster_id in index:
                    matrix[row, index[cluster_id]] += 1.0
            matrix[row, -2] = sum(cluster_id == oov_cluster_id for cluster_id in cluster_ids) / n_events
            matrix[row, -1] = float(np.log1p(n_events))
        return matrix

    return tuple(transform(graphs) for graphs in (train_graphs, *splits))


def _cluster_ids(graph: Any) -> list[int]:
    value = getattr(graph, "event_cluster_ids", None)
    if value is None:
        raise ValueError(
            "Graph bundle lacks event_cluster_ids required by Isolation Forest. "
            "Rebuild the BGL campaign graphs with the current pipeline."
        )
    if hasattr(value, "detach"):
        value = value.detach().cpu().tolist()
    elif hasattr(value, "tolist"):
        value = value.tolist()
    return [int(cluster_id) for cluster_id in value]


def _copy_graph_splits(source_graph: Path, dest_graph: Path) -> None:
    source_dir = source_graph.parent / "graph_dataset_splits"
    if not source_dir.exists():
        source_dir = graph_split_dir(source_graph)
    if not source_dir.exists():
        return
    destination = graph_split_dir(dest_graph)
    if destination.resolve() == source_dir.resolve():
        return
    shutil.copytree(source_dir, destination, dirs_exist_ok=True)


def _normalise_edge_attributes(
    train_graphs: list,
    val_graphs: list,
    test_graphs: list,
    *,
    enabled: bool,
    minimum_std: float,
) -> tuple[list[float] | None, list[float] | None]:
    if not enabled:
        return None, None
    training_edges = [
        graph.edge_attr.float()
        for graph in train_graphs
        if getattr(graph, "edge_attr", None) is not None and graph.edge_attr.numel() > 0
    ]
    if not training_edges:
        return None, None
    stacked = torch.cat(training_edges, dim=0)
    mean = stacked.mean(dim=0)
    std = stacked.std(dim=0).clamp_min(minimum_std)
    for graph in [*train_graphs, *val_graphs, *test_graphs]:
        edge_attr = getattr(graph, "edge_attr", None)
        if edge_attr is not None and edge_attr.numel() > 0:
            graph.edge_attr = (edge_attr.float() - mean) / std
    return mean.tolist(), std.tolist()


def _loss_history(history: list[dict[str, float | int]]) -> dict[str, list[float]]:
    return {
        "total": [float(entry["train_total_loss"]) for entry in history],
        "structure": [float(entry["train_structure_loss"]) for entry in history],
        "node": [float(entry["train_node_loss"]) for entry in history],
        "edge": [float(entry["train_edge_loss"]) for entry in history],
    }


def _threshold_and_metrics(
    labels: np.ndarray, scores: np.ndarray, *, threshold: float | None = None
) -> tuple[float, dict[str, float]]:
    from sklearn.metrics import precision_recall_curve

    if threshold is None:
        precision, recall, thresholds = precision_recall_curve(labels, scores)
        if len(thresholds) == 0:
            threshold = float(np.median(scores))
        else:
            f1_values = (2 * precision[:-1] * recall[:-1]) / (
                precision[:-1] + recall[:-1] + 1e-12
            )
            threshold = float(thresholds[int(np.nanargmax(f1_values))])
    return threshold, _score_metrics(labels, scores, threshold)


def _score_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, float]:
    from sklearn.metrics import average_precision_score, f1_score, roc_auc_score

    predictions = (scores > threshold).astype(int)
    has_both_classes = len(np.unique(labels)) == 2
    return {
        "f1": _rounded(f1_score(labels, predictions, zero_division=0)),
        "pr_auc": _rounded(average_precision_score(labels, scores)) if has_both_classes else 0.0,
        "roc_auc": _rounded(roc_auc_score(labels, scores)) if has_both_classes else 0.0,
    }


def _node_recon_index_for_bundle(
    *,
    node_dim: int,
    node_reconstruct: str,
    split_meta: dict[str, Any],
    bundle_meta: dict[str, Any] | None,
) -> Any:
    if node_reconstruct != "without_sbert":
        return torch.arange(node_dim, dtype=torch.long)
    meta = flatten_split_meta(split_meta, bundle_meta)
    sbert_dim = sbert_dim_from_meta(meta, node_dim=node_dim)
    return node_recon_index_tensor(
        node_dim,
        sbert_dim=sbert_dim,
        extra_dim=int(meta.get("node_extra_dim", 9)),
    )


def _dump_config(path: Path, config: Mapping[str, Any]) -> None:
    serialisable = {key: value for key, value in config.items() if key != "__pipeline__"}
    if yaml is not None:
        path.write_text(yaml.safe_dump(serialisable, sort_keys=False))
    else:
        path.write_text(json.dumps(serialisable, indent=2, default=str))


def _component_aucs(labels: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    from sklearn.metrics import average_precision_score, roc_auc_score

    has_both = len(np.unique(labels)) == 2
    return {
        "pr_auc": float(average_precision_score(labels, scores)) if has_both else 0.0,
        "roc_auc": float(roc_auc_score(labels, scores)) if has_both else 0.0,
    }


def component_metrics(
    labels: np.ndarray,
    *,
    combined: np.ndarray,
    structure: np.ndarray,
    node: np.ndarray,
    edge: np.ndarray,
    predictions: np.ndarray,
    alpha: float,
    beta: float,
    gamma: float,
    node_blocks: Mapping[str, np.ndarray] | None = None,
) -> dict[str, Any]:
    metrics: dict[str, Any] = {
        "combined": _component_aucs(labels, combined),
        "structure": _component_aucs(labels, structure),
        "node": _component_aucs(labels, node),
        "edge": _component_aucs(labels, edge),
    }
    for name, values in (node_blocks or {}).items():
        array = np.asarray(values)
        metrics[name] = {
            **_component_aucs(labels, array),
            "mean_normal": float(array[labels == 0].mean()) if (labels == 0).any() else 0.0,
            "mean_anomaly": float(array[labels == 1].mean()) if (labels == 1).any() else 0.0,
        }
    weighted = np.column_stack([alpha * structure, beta * node, gamma * edge])
    totals = weighted.sum(axis=1, keepdims=True)
    shares = np.divide(
        weighted,
        np.maximum(totals, 1e-12),
        out=np.zeros_like(weighted, dtype=float),
        where=totals > 0,
    )
    true_positives = (labels == 1) & (predictions == 1)
    names = ("structure", "node", "edge")
    if true_positives.any():
        mean_share = shares[true_positives].mean(axis=0)
        dominant = np.argmax(shares[true_positives], axis=1)
        metrics["true_positive_share"] = {
            name: float(mean_share[index]) for index, name in enumerate(names)
        }
        metrics["true_positive_dominance_count"] = {
            name: int((dominant == index).sum()) for index, name in enumerate(names)
        }
        metrics["n_true_positives"] = int(true_positives.sum())
    else:
        metrics["true_positive_share"] = {"structure": 0.0, "node": 0.0, "edge": 0.0}
        metrics["true_positive_dominance_count"] = {"structure": 0, "node": 0, "edge": 0}
        metrics["n_true_positives"] = 0
    return metrics


def write_eval_pack(
    output_dir: str | Path,
    *,
    metrics: Mapping[str, Any],
    history_epochs: list[dict[str, Any]],
    labels: np.ndarray,
    scores: np.ndarray,
    structure: np.ndarray,
    node: np.ndarray,
    edge: np.ndarray,
    threshold: float,
    alpha: float,
    beta: float,
    gamma: float,
    node_blocks: Mapping[str, np.ndarray] | None = None,
    graph_ids: list[str] | None = None,
    config: Mapping[str, Any] | None = None,
    campaign_meta: Mapping[str, Any] | None = None,
) -> dict[str, Path]:
    output_dir = Path(output_dir)
    figures = output_dir / "figures"
    scores_dir = output_dir / "scores"
    figures.mkdir(parents=True, exist_ok=True)
    scores_dir.mkdir(parents=True, exist_ok=True)

    predictions = (np.asarray(scores) > threshold).astype(int)
    labels = np.asarray(labels)
    structure = np.asarray(structure)
    node = np.asarray(node)
    edge = np.asarray(edge)
    components = component_metrics(
        labels,
        combined=np.asarray(scores),
        structure=structure,
        node=node,
        edge=edge,
        predictions=predictions,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
        node_blocks=node_blocks,
    )
    component_path = output_dir / "component_metrics.json"
    component_path.write_text(json.dumps(components, indent=2))

    history_path = output_dir / "history.json"
    history_path.write_text(json.dumps({"epochs": history_epochs}, indent=2))

    config_path = output_dir / "config.yaml"
    if config is not None:
        _dump_config(config_path, config)

    score_columns: dict[str, Any] = {
        "graph_id": graph_ids if graph_ids is not None else list(range(len(labels))),
        "label": labels.astype(int),
        "prediction": predictions,
        "score": np.asarray(scores),
        "structure": structure,
        "node": node,
        "edge": edge,
        "weighted_structure": alpha * structure,
        "weighted_node": beta * node,
        "weighted_edge": gamma * edge,
    }
    for name, values in (node_blocks or {}).items():
        score_columns[name] = np.asarray(values)
    score_frame = pd.DataFrame(score_columns)
    scores_csv = scores_dir / "test_component_scores.csv"
    score_frame.to_csv(scores_csv, index=False)

    manifest_path = output_dir / "manifest.json"
    if campaign_meta is not None:
        manifest_path.write_text(json.dumps(dict(campaign_meta), indent=2, default=str))
    else:
        manifest_path.write_text("{}")

    written: dict[str, Path] = {
        "component_metrics": component_path,
        "history": history_path,
        "config": config_path,
        "scores_csv": scores_csv,
        "manifest": manifest_path,
        "metrics": output_dir / "metrics.json",
    }
    (output_dir / "metrics.json").write_text(json.dumps(dict(metrics), indent=2, default=str))
    written.update(
        _write_eval_figures(
            figures,
            history_epochs=history_epochs,
            labels=labels,
            scores=np.asarray(scores),
            predictions=predictions,
            threshold=threshold,
            structure=structure,
            node=node,
            edge=edge,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
        )
    )
    return written


def _write_eval_figures(
    figures: Path,
    *,
    history_epochs: list[dict[str, Any]],
    labels: np.ndarray,
    scores: np.ndarray,
    predictions: np.ndarray,
    threshold: float,
    structure: np.ndarray,
    node: np.ndarray,
    edge: np.ndarray,
    alpha: float,
    beta: float,
    gamma: float,
) -> dict[str, Path]:
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from sklearn.metrics import (
            ConfusionMatrixDisplay,
            average_precision_score,
            confusion_matrix,
            precision_recall_curve,
            roc_auc_score,
            roc_curve,
        )
    except ImportError:
        return {}

    written: dict[str, Path] = {}
    if history_epochs:
        epochs = [int(entry["epoch"]) for entry in history_epochs]
        figure, axes = plt.subplots(1, 3, figsize=(18, 4.5))
        axes[0].plot(epochs, [entry["train_total_loss"] for entry in history_epochs], color="black", linewidth=2)
        axes[0].set(title="Total training loss", xlabel="Epoch", ylabel="Loss")
        for key, color, label in (
            ("train_structure_loss", "#4C72B0", "structure"),
            ("train_node_loss", "#55A868", "node"),
            ("train_edge_loss", "#C44E52", "edge"),
        ):
            axes[1].plot(epochs, [entry[key] for entry in history_epochs], linewidth=2, color=color, label=label)
        axes[1].set(title="Reconstruction components", xlabel="Epoch", ylabel="Loss")
        axes[1].legend()
        if "val_f1" in history_epochs[0]:
            axes[2].plot(epochs, [entry["val_f1"] for entry in history_epochs], linewidth=2, color="#8172B2", label="Val F1")
            axes[2].plot(epochs, [entry["val_pr_auc"] for entry in history_epochs], linewidth=1.5, color="#DD8452", label="Val PR-AUC")
            axes[2].plot(epochs, [entry["val_roc_auc"] for entry in history_epochs], linewidth=1.5, color="#937860", label="Val ROC-AUC")
            axes[2].set(title="Validation metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1.05))
            axes[2].legend()
        for axis in axes:
            axis.grid(alpha=0.25)
        figure.tight_layout()
        path = figures / "training_history.png"
        figure.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        written["training_history"] = path
        learning_path = figures / "learning_curve.png"
        shutil.copy2(path, learning_path)
        written["learning_curve"] = learning_path

    figure, axes = plt.subplots(1, 1, figsize=(7, 4.5))
    for class_id, color, title in ((0, "#4C72B0", "Normal"), (1, "#C44E52", "Anomaly")):
        subset = scores[labels == class_id]
        if len(subset):
            axes.hist(subset, density=True, bins=40, alpha=0.45, color=color, label=f"{title} (n={len(subset)})")
    axes.axvline(
        threshold,
        color="black",
        linestyle="--",
        linewidth=2,
        label=f"threshold = {float(threshold):.4f}",
    )
    axes.set(
        title="Normal vs anomaly score distribution",
        xlabel="Anomaly score (weighted reconstruction error)",
        ylabel="Density",
    )
    axes.legend()
    axes.grid(alpha=0.25)
    figure.tight_layout()
    dist_path = figures / "test_score_distribution.png"
    figure.savefig(dist_path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    written["score_distribution"] = dist_path

    figure, axis = plt.subplots(figsize=(5, 4.5))
    ConfusionMatrixDisplay(
        confusion_matrix(labels, predictions, labels=[0, 1]),
        display_labels=["Normal", "Anomaly"],
    ).plot(ax=axis, colorbar=False, cmap="Blues")
    axis.set_title("Held-out confusion matrix")
    figure.tight_layout()
    cm_path = figures / "confusion_matrix.png"
    figure.savefig(cm_path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    written["confusion_matrix"] = cm_path

    if len(np.unique(labels)) == 2:
        figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        precision, recall, _ = precision_recall_curve(labels, scores)
        axes[0].plot(recall, precision, linewidth=2, label=f"PR-AUC = {average_precision_score(labels, scores):.4f}")
        axes[0].axhline(float(labels.mean()), color="gray", linestyle="--", label="positive-class rate")
        axes[0].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
        axes[0].legend()
        false_positive_rate, true_positive_rate, _ = roc_curve(labels, scores)
        axes[1].plot(false_positive_rate, true_positive_rate, linewidth=2, label=f"ROC-AUC = {roc_auc_score(labels, scores):.4f}")
        axes[1].plot([0, 1], [0, 1], "--", color="gray", label="random")
        axes[1].set(xlim=(0, 1), ylim=(0, 1.05), xlabel="False positive rate", ylabel="True positive rate", title="ROC curve")
        axes[1].legend()
        figure.tight_layout()
        pr_path = figures / "test_pr_roc.png"
        figure.savefig(pr_path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        written["pr_roc"] = pr_path

        pr_only, axis = plt.subplots(figsize=(6.5, 4.5))
        axis.plot(recall, precision, linewidth=2, label=f"PR-AUC = {average_precision_score(labels, scores):.4f}")
        axis.axhline(float(labels.mean()), color="gray", linestyle="--", label="positive-class rate")
        axis.set(xlim=(0, 1), ylim=(0, 1.05), xlabel="Recall", ylabel="Precision", title="Precision–recall curve")
        axis.legend()
        axis.grid(alpha=0.25)
        pr_only.tight_layout()
        pr_only_path = figures / "test_pr_curve.png"
        pr_only.savefig(pr_only_path, dpi=160, bbox_inches="tight")
        plt.close(pr_only)
        written["pr_curve"] = pr_only_path

    weighted = pd.DataFrame(
        {
            "Structure": alpha * structure,
            "Node": beta * node,
            "Edge": gamma * edge,
            "label": labels,
            "prediction": predictions,
        }
    )
    true_positives = weighted[(weighted.label == 1) & (weighted.prediction == 1)]
    if len(true_positives):
        contribution = true_positives[["Structure", "Node", "Edge"]].div(
            true_positives[["Structure", "Node", "Edge"]].sum(axis=1), axis=0
        ).fillna(0)
        figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        average_contribution = contribution.mean().mul(100)
        axes[0].pie(average_contribution, labels=list(average_contribution.index), autopct="%1.1f%%")
        axes[0].set_title(f"Average weighted contribution — true positives (n={len(true_positives)})")
        dominant = contribution.idxmax(axis=1).value_counts().reindex(["Structure", "Node", "Edge"], fill_value=0)
        axes[1].bar(dominant.index, dominant.values, color=["#4C72B0", "#55A868", "#C44E52"])
        axes[1].set(title="Dominant component per true positive", ylabel="Graphs")
        axes[1].grid(axis="y", alpha=0.25)
        figure.tight_layout()
        contrib_path = figures / "component_contribution.png"
        figure.savefig(contrib_path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        written["component_contribution"] = contrib_path
    return written


def resolve_campaign_baseline_name(
    records: list[Mapping[str, Any]] | pd.DataFrame | None = None,
    *,
    dataset: str,
    baseline_name: str | None = None,
) -> str:
    if baseline_name:
        return str(baseline_name)
    names: set[str] = set()
    if isinstance(records, pd.DataFrame):
        if "name" in records.columns:
            names = {str(item) for item in records["name"].tolist()}
    elif records:
        names = {str(item.get("name")) for item in records if item.get("name")}
    if "hybrid_llm" in names:
        return "hybrid_llm"
    if "baseline_full" in names:
        return "baseline_full"
    return "hybrid_llm" if str(dataset).lower() == "bgl" else "baseline_full"


def write_campaign_report(
    workspace: str | Path,
    *,
    dataset: str,
    campaign_id: str,
    baseline_name: str | None = None,
) -> Path:
    workspace = Path(workspace)
    output_root = workspace / "outputs" / dataset
    campaign_dir = output_root / "campaigns" / campaign_id
    campaign_dir.mkdir(parents=True, exist_ok=True)

    records: list[dict[str, Any]] = []
    prefix = f"{campaign_id}_"
    for metrics_path in sorted(output_root.glob("*/metrics.json")):
        run_dir = metrics_path.parent
        if run_dir.name in {"campaigns", "notebook_reports"} or run_dir.name.endswith("_ablation_matrix"):
            continue
        if not run_dir.name.startswith(prefix) and run_dir.name != campaign_id:
            manifest_path = run_dir / "manifest.json"
            if not manifest_path.exists():
                continue
            manifest = json.loads(manifest_path.read_text())
            if manifest.get("campaign_id") != campaign_id:
                continue
        metrics = json.loads(metrics_path.read_text())
        manifest = {}
        if (run_dir / "manifest.json").exists():
            manifest = json.loads((run_dir / "manifest.json").read_text())
        components = {}
        if (run_dir / "component_metrics.json").exists():
            components = json.loads((run_dir / "component_metrics.json").read_text())
        history = {}
        if (run_dir / "history.json").exists():
            history = json.loads((run_dir / "history.json").read_text())
        name = manifest.get("experiment_name") or run_dir.name.removeprefix(prefix)
        records.append(
            {
                "name": name,
                "run_id": run_dir.name,
                "status": "OK",
                "dataset": dataset,
                "campaign_id": campaign_id,
                "family": manifest.get("family"),
                "seed": manifest.get("seed", metrics.get("seed")),
                "split_lock_id": manifest.get("split_lock_id"),
                "split_protocol": manifest.get("split_protocol"),
                "oov_graph_rate": manifest.get("oov_graph_rate"),
                "oov_line_rate": manifest.get("oov_line_rate"),
                "run_dir": str(run_dir),
                "test_f1": metrics.get("test_f1"),
                "test_pr_auc": metrics.get("test_pr_auc"),
                "test_roc_auc": metrics.get("test_roc_auc"),
                "val_f1": metrics.get("val_f1"),
                "val_pr_auc": metrics.get("val_pr_auc"),
                "val_roc_auc": metrics.get("val_roc_auc"),
                "test_precision": metrics.get("test_precision"),
                "test_recall": metrics.get("test_recall"),
                "best_threshold": metrics.get("best_threshold"),
                "component_metrics": components,
                "history": _history_from_epochs(history.get("epochs") or metrics.get("history")),
            }
        )

    if not records:
        raise FileNotFoundError(
            f"No completed runs found for campaign {campaign_id!r} under {output_root}"
        )

    baseline_name = resolve_campaign_baseline_name(
        records, dataset=dataset, baseline_name=baseline_name
    )
    frame = pd.DataFrame(records)
    for column in ("test_f1", "test_pr_auc", "test_roc_auc"):
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.sort_values("test_pr_auc", ascending=False, na_position="last")
    csv_path = campaign_dir / "leaderboard.csv"
    json_path = campaign_dir / "leaderboard.json"
    frame.drop(columns=["component_metrics", "history"], errors="ignore").to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(records, indent=2, default=str))

    summary_frame, paired_frame = _seeded_campaign_statistics(frame, baseline_name)
    summary_frame.to_csv(campaign_dir / "summary_by_arm.csv", index=False)
    paired_frame.to_csv(campaign_dir / "paired_bootstrap_ci.csv", index=False)
    time_block_frame = _bgl_time_block_bootstrap(records, baseline_name=baseline_name)
    if not time_block_frame.empty:
        time_block_frame.to_csv(campaign_dir / "paired_time_block_bootstrap.csv", index=False)

    delta_path = campaign_dir / "delta_vs_baseline.csv"
    if not paired_frame.empty:
        paired_frame.to_csv(delta_path, index=False)

    _write_campaign_figures(campaign_dir, records, baseline_name=baseline_name)
    readme = campaign_dir / "README.md"
    readme.write_text(_campaign_readme(campaign_id, dataset, frame, baseline_name))
    return json_path


def _bgl_time_block_bootstrap(
    records: list[dict[str, Any]], *, baseline_name: str, samples: int = 2_000
) -> pd.DataFrame:
    from sklearn.metrics import average_precision_score

    baseline_records = [item for item in records if item.get("name") == baseline_name]
    rows: list[dict[str, Any]] = []
    for base in baseline_records:
        for arm in records:
            if arm.get("name") == baseline_name or arm.get("seed") != base.get("seed"):
                continue
            base_path = Path(str(base["run_dir"])) / "scores" / "test_component_scores.csv"
            arm_path = Path(str(arm["run_dir"])) / "scores" / "test_component_scores.csv"
            if not base_path.exists() or not arm_path.exists():
                continue
            merged = pd.read_csv(base_path)[["graph_id", "label", "score"]].merge(
                pd.read_csv(arm_path)[["graph_id", "label", "score"]],
                on=["graph_id", "label"], suffixes=("_baseline", "_arm"),
            )
            if merged.empty or merged["label"].nunique() != 2:
                continue
            try:
                graph_ids = pd.to_numeric(merged["graph_id"], errors="raise").astype(np.int64)
            except (ValueError, TypeError):
                continue
            unix_start = graph_ids % BGL_SPLIT_STRIDE
            for days in (1, 7):
                blocks = (unix_start // (days * 86_400)).to_numpy()
                unique = np.unique(blocks)
                if len(unique) < 2:
                    continue
                rng = np.random.default_rng(20_260 + int(base["seed"]))
                deltas: list[float] = []
                for _ in range(samples):
                    picked = rng.choice(unique, size=len(unique), replace=True)
                    indices = np.concatenate([np.flatnonzero(blocks == item) for item in picked])
                    labels = merged["label"].to_numpy()[indices]
                    if len(np.unique(labels)) != 2:
                        continue
                    deltas.append(float(
                        average_precision_score(labels, merged["score_arm"].to_numpy()[indices])
                        - average_precision_score(labels, merged["score_baseline"].to_numpy()[indices])
                    ))
                if deltas:
                    rows.append({
                        "baseline": baseline_name, "name": arm["name"], "seed": arm["seed"],
                        "block_days": days, "n_blocks": len(unique), "n_bootstrap": len(deltas),
                        "ap_delta": float(average_precision_score(merged["label"], merged["score_arm"])
                                          - average_precision_score(merged["label"], merged["score_baseline"])),
                        "ci95_low": float(np.quantile(deltas, 0.025)),
                        "ci95_high": float(np.quantile(deltas, 0.975)),
                        "bootstrap_unit": "paired_time_block",
                    })
    return pd.DataFrame(rows)


def _seeded_campaign_statistics(
    frame: pd.DataFrame,
    baseline_name: str,
    *,
    bootstrap_samples: int = 10_000,
    bootstrap_seed: int = 2026,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    metrics = ("test_f1", "test_pr_auc", "test_roc_auc")
    summary_rows: list[dict[str, Any]] = []
    for name, group in frame.groupby("name", sort=True):
        row: dict[str, Any] = {"name": name, "n_seeds": int(group["seed"].nunique())}
        for metric in metrics:
            values = pd.to_numeric(group[metric], errors="coerce").dropna()
            row[f"{metric}_mean"] = float(values.mean()) if len(values) else np.nan
            row[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        summary_rows.append(row)

    paired_rows: list[dict[str, Any]] = []
    baseline = frame[frame["name"] == baseline_name]
    if not baseline.empty and baseline["seed"].notna().all():
        rng = np.random.default_rng(bootstrap_seed)
        pair_keys = ["seed"]
        if "split_lock_id" in frame.columns and baseline["split_lock_id"].notna().all():
            pair_keys.append("split_lock_id")
        for name, group in frame.groupby("name", sort=True):
            merged = baseline[[*pair_keys, *metrics]].merge(
                group[[*pair_keys, *metrics]], on=pair_keys, suffixes=("_baseline", "_arm")
            )
            for metric in metrics:
                delta = (
                    pd.to_numeric(merged[f"{metric}_arm"], errors="coerce")
                    - pd.to_numeric(merged[f"{metric}_baseline"], errors="coerce")
                ).dropna().to_numpy(dtype=float)
                if not len(delta):
                    continue
                sampled = rng.choice(delta, size=(bootstrap_samples, len(delta)), replace=True).mean(axis=1)
                paired_rows.append(
                    {
                        "name": name,
                        "metric": metric,
                        "n_paired_seeds": int(len(delta)),
                        "mean_delta": float(delta.mean()),
                        "ci95_low": float(np.quantile(sampled, 0.025)),
                        "ci95_high": float(np.quantile(sampled, 0.975)),
                        "bootstrap_unit": "training_seed",
                        "pairing_keys": "+".join(pair_keys),
                    }
                )
    return pd.DataFrame(summary_rows), pd.DataFrame(paired_rows)


def _history_from_epochs(history: Any) -> dict[str, list[float]]:
    if isinstance(history, dict) and "total" in history:
        return history
    if isinstance(history, list):
        return {
            "total": [float(entry.get("train_total_loss", 0.0)) for entry in history],
            "structure": [float(entry.get("train_structure_loss", 0.0)) for entry in history],
            "node": [float(entry.get("train_node_loss", 0.0)) for entry in history],
            "edge": [float(entry.get("train_edge_loss", 0.0)) for entry in history],
        }
    return {}


def _write_campaign_figures(
    campaign_dir: Path,
    records: list[dict[str, Any]],
    *,
    baseline_name: str,
) -> None:
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        import matplotlib.ticker as mticker
    except ImportError:
        return

    ok = [record for record in records if record.get("status") == "OK"]
    if not ok:
        return
    names = [record["name"] for record in ok]
    x = np.arange(len(names))
    figure, axes = plt.subplots(1, 3, figsize=(15, 5))
    for axis, metric, label in zip(
        axes,
        ("test_f1", "test_pr_auc", "test_roc_auc"),
        ("Test F1", "Test PR-AUC", "Test ROC-AUC"),
    ):
        values = [float(record.get(metric) or 0.0) for record in ok]
        bars = axis.barh(x, values, color=plt.cm.tab10.colors[: len(names)], edgecolor="white", height=0.6)
        if values:
            bars[int(np.argmax(values))].set_edgecolor("#111")
            bars[int(np.argmax(values))].set_linewidth(2)
        for bar, value in zip(bars, values):
            axis.text(value + 0.002, bar.get_y() + bar.get_height() / 2, f"{value:.4f}", va="center", fontsize=8)
        axis.set_yticks(x)
        axis.set_yticklabels(names, fontsize=9)
        axis.set_xlabel(label)
        axis.invert_yaxis()
        axis.grid(axis="x", alpha=0.3)
        axis.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    figure.suptitle("Ablation comparison", fontsize=14, fontweight="bold")
    figure.tight_layout()
    figure.savefig(campaign_dir / "ablation_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(figure)

    histories = {record["name"]: record.get("history") or {} for record in ok if record.get("history")}
    if histories:
        n_exp = len(histories)
        figure, axes = plt.subplots(1, n_exp, figsize=(5 * n_exp, 4), squeeze=False)
        for axis, (name, hist) in zip(axes[0], histories.items()):
            epochs = range(1, len(hist.get("total") or []) + 1)
            axis.plot(epochs, hist.get("total") or [], "ko-", lw=2, ms=4, label="Total")
            axis.plot(epochs, hist.get("structure") or [], "bo--", lw=1.4, ms=3, label="Structure")
            axis.plot(epochs, hist.get("node") or [], "go-.", lw=1.4, ms=3, label="Node")
            axis.plot(epochs, hist.get("edge") or [], "ro:", lw=1.4, ms=3, label="Edge")
            axis.set_title(name, fontsize=9)
            axis.set_xlabel("Epoch")
            axis.set_ylabel("Loss")
            axis.legend(fontsize=7)
            axis.grid(alpha=0.3)
        figure.suptitle("Training loss curves", fontsize=13, fontweight="bold")
        figure.tight_layout()
        figure.savefig(campaign_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
        plt.close(figure)

    component_rows = []
    for record in ok:
        combined = (record.get("component_metrics") or {}).get("combined") or {}
        node = (record.get("component_metrics") or {}).get("node") or {}
        edge = (record.get("component_metrics") or {}).get("edge") or {}
        structure = (record.get("component_metrics") or {}).get("structure") or {}
        if combined or node:
            component_rows.append(
                {
                    "name": record["name"],
                    "combined": combined.get("roc_auc", 0.0),
                    "structure": structure.get("roc_auc", 0.0),
                    "node": node.get("roc_auc", 0.0),
                    "edge": edge.get("roc_auc", 0.0),
                }
            )
    if component_rows:
        figure, axis = plt.subplots(figsize=(10, 5))
        index = np.arange(len(component_rows))
        width = 0.2
        for offset, key, color in (
            (-1.5, "combined", "black"),
            (-0.5, "structure", "#4C72B0"),
            (0.5, "node", "#55A868"),
            (1.5, "edge", "#C44E52"),
        ):
            axis.bar(index + offset * width, [row[key] for row in component_rows], width, label=key, color=color)
        axis.set_xticks(index)
        axis.set_xticklabels([row["name"] for row in component_rows], rotation=30, ha="right")
        axis.set_ylabel("ROC-AUC")
        axis.set_ylim(0, 1.05)
        axis.legend()
        axis.set_title("Component ROC-AUC vs combined")
        axis.grid(axis="y", alpha=0.3)
        figure.tight_layout()
        figure.savefig(campaign_dir / "component_comparison.png", dpi=150, bbox_inches="tight")
        plt.close(figure)


def _campaign_readme(
    campaign_id: str,
    dataset: str,
    frame: pd.DataFrame,
    baseline_name: str,
) -> str:
    generated = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    lines = [
        f"# {dataset.upper()} ablation campaign `{campaign_id}`",
        "",
        f"Generated {generated}. Primary ranking metric: **test PR-AUC**.",
        "",
        "PR-AUC is threshold-free; chance is the positive-class rate, not 0.5. "
        "Test F1 uses the validation-F1 threshold (one operating point). "
        "ROC-AUC is secondary (chance = 0.5) and can look strong under class imbalance.",
        "",
        "| name | test_f1 | test_pr_auc | test_roc_auc | val_f1 |",
        "|---|---:|---:|---:|---:|",
    ]
    for _, row in frame.iterrows():
        marker = " **(baseline)**" if row["name"] == baseline_name else ""
        lines.append(
            f"| {row['name']}{marker} | {row.get('test_f1'):.4f} | {row.get('test_pr_auc'):.4f} "
            f"| {row.get('test_roc_auc'):.4f} | {row.get('val_f1'):.4f} |"
            if pd.notna(row.get("test_f1"))
            else f"| {row['name']}{marker} |  |  |  |  |"
        )
    lines.extend(
        [
            "",
            "Artifacts per run live under `outputs/{dataset}/{run_id}/` "
            "(metrics, figures, scores, checkpoint). This folder is the campaign rollup.",
            "",
        ]
    )
    return "\n".join(lines)


def resolved_training() -> dict[str, Any]:
    training = copy.deepcopy(TRAINING)
    if SMOKE:
        training["test_run"] = True
        training["epochs"] = 1
    return training


def assert_arm_meta(arm: str, meta: Mapping[str, Any] | None) -> None:
    global _LOCK_REF
    if not meta:
        raise FileNotFoundError(f"Missing dataset_meta.json for arm {arm!r}.")
    identity = meta.get("graph_identity") or {}
    split_lock = meta.get("split_lock_id")
    protocol = identity.get("split_protocol") or meta.get("split_protocol")
    fit_on = identity.get("fit_on") or meta.get("fit_on")
    n_train = int(meta.get("n_train") or 0)
    n_val = int(meta.get("n_val") or 0)
    n_test = int(meta.get("n_test") or 0)
    errors = []
    if protocol != "stratified":
        errors.append(f"split_protocol={protocol!r} (expected 'stratified')")
    if fit_on != "all":
        errors.append(f"fit_on={fit_on!r} (expected 'all')")
    if EXPECTED_SPLIT_LOCK_ID and split_lock != EXPECTED_SPLIT_LOCK_ID:
        errors.append(f"split_lock_id={split_lock!r} (expected {EXPECTED_SPLIT_LOCK_ID!r})")
    marker = (split_lock, n_train, n_val, n_test)
    if _LOCK_REF is None:
        _LOCK_REF = marker
    elif marker != _LOCK_REF:
        errors.append(f"split counts/lock {marker} disagree with first arm {_LOCK_REF}")
    if errors:
        raise ValueError(f"Arm {arm!r} is not compatible with this notebook: " + "; ".join(errors))


def stage_arm_graph(arm: str) -> Path:
    campaign_dir = Path(CAMPAIGN_DIR)
    source_gz = campaign_dir / "graphs" / arm / "graph_dataset.pt.gz"
    source_pt = campaign_dir / "graphs" / arm / "graph_dataset.pt"
    source_meta = campaign_dir / "graphs" / arm / "dataset_meta.json"
    if not source_gz.exists() and not source_pt.exists():
        raise FileNotFoundError(
            f"Prepared graph for {arm!r} not found under {campaign_dir / 'graphs' / arm}. "
            f"Upload campaigns/{CAMPAIGN_ID} to Drive or run prepare cells first."
        )
    dest = (
        Path(WORKSPACE_ROOT)
        / "data"
        / "processed"
        / DATASET
        / f"{_safe_name(CAMPAIGN_ID)}_{_safe_name(arm)}_graph_dataset.pt"
    )
    dest.parent.mkdir(parents=True, exist_ok=True)
    if source_gz.exists():
        if not dest.exists() or dest.stat().st_mtime < source_gz.stat().st_mtime:
            gunzip_file(source_gz, dest)
        _copy_graph_splits(source_gz, dest)
    else:
        if not dest.exists() or dest.stat().st_mtime < source_pt.stat().st_mtime:
            shutil.copy2(source_pt, dest)
        _copy_graph_splits(source_pt, dest)
    if source_meta.exists():
        shutil.copy2(source_meta, dest.with_name("dataset_meta.json"))
    meta = load_bundle_meta(dest)
    assert_arm_meta(arm, meta)
    return dest


def _experiment_config(arm: str, seed: int, family: str = "representation") -> dict[str, Any]:
    return {
        "experiment": {
            "name": arm,
            "dataset": DATASET,
            "seed": seed,
            "campaign_id": CAMPAIGN_ID,
            "family": family,
            "run_id": f"{_safe_name(CAMPAIGN_ID)}_{_safe_name(arm)}_seed{seed}",
        },
        "training": resolved_training(),
        "ablation": {
            "graph": {
                "gine_aggregation": GAE_ARCH["gine_aggregation"],
                "node_transformation": GAE_ARCH["node_transformation"],
                "structure_decoder": GAE_ARCH["structure_decoder"],
            },
            "fusion": {
                "node_reconstruct": GAE_ARCH["node_reconstruct"],
                "mode": GAE_ARCH["fusion_mode"],
                "modality_projection_dim": GAE_ARCH["modality_projection_dim"],
                "node_loss": GAE_ARCH["node_loss_mode"],
                "node_block_weights": GAE_ARCH["node_block_weights"],
            },
        },
    }


def train_graph_bundle(
    graph_path: Path,
    *,
    training: dict[str, Any],
    seed: int,
    bundle_meta: dict[str, Any] | None = None,
) -> tuple[dict[str, Any], dict[str, Any], dict[str, Any]]:
    from sklearn.metrics import confusion_matrix, precision_score, recall_score
    from torch.optim import Adam
    from torch_geometric.loader import DataLoader

    seed_everything(seed)
    device = get_device()
    train_graphs, val_graphs, test_graphs, split_meta = load_graph_splits(graph_path)
    meta = flatten_split_meta(split_meta, bundle_meta)
    node_dim = int(meta.get("node_dim") or split_meta.get("node_dim") or 0)
    edge_dim = int(meta.get("edge_dim") or split_meta.get("edge_dim") or 0)
    if not node_dim or not edge_dim:
        bundle = torch.load(graph_path, weights_only=False, map_location="cpu")
        node_dim = int(bundle["node_dim"])
        edge_dim = int(bundle["edge_dim"])
    if training["train_mode"] == "clean":
        train_graphs = [graph for graph in train_graphs if graph.y.item() == 0]
    if not train_graphs:
        raise ValueError("No training graphs remain after applying train_mode.")

    if training["test_run"]:
        sample_count = int(training["test_samples"])
        train_graphs = train_graphs[:sample_count]
        val_graphs = val_graphs[:sample_count]
        test_graphs = test_graphs[:sample_count]

    edge_mean, edge_std = _normalise_edge_attributes(
        train_graphs,
        val_graphs,
        test_graphs,
        enabled=bool(training["pre_normalize_edges"]),
        minimum_std=float(training["minimum_edge_std"]),
    )
    batch_size = int(training["batch_size"])
    train_loader = DataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_graphs, batch_size=batch_size)
    test_loader = DataLoader(test_graphs, batch_size=batch_size)

    model = AttributeAwareGAE(
        node_dim=int(node_dim),
        edge_dim=int(edge_dim),
        hidden_dim=int(training["hidden_dim"]),
        latent_dim=int(training["latent_dim"]),
        gine_aggregation=str(GAE_ARCH["gine_aggregation"]),
        node_transformation=str(GAE_ARCH["node_transformation"]),
        structure_decoder=str(GAE_ARCH["structure_decoder"]),
        node_recon_index=_node_recon_index_for_bundle(
            node_dim=int(node_dim),
            node_reconstruct=str(GAE_ARCH["node_reconstruct"]),
            split_meta=split_meta,
            bundle_meta=bundle_meta,
        ),
        tfidf_dim=int(meta.get("tfidf_dim") or 0),
        sbert_dim=int(meta.get("sbert_dim") or 0),
        fusion_mode=str(GAE_ARCH["fusion_mode"]),
        modality_projection_dim=int(GAE_ARCH["modality_projection_dim"]),
        node_loss_mode=str(GAE_ARCH["node_loss_mode"]),
        node_block_weights=dict(GAE_ARCH["node_block_weights"]),
    ).to(device)
    optimizer = Adam(model.parameters(), lr=float(training["learning_rate"]))
    loss_args = {
        "alpha": float(training["alpha"]),
        "beta": float(training["beta"]),
        "gamma": float(training["gamma"]),
    }
    history: list[dict[str, float | int]] = []
    best_state: dict[str, Any] | None = None
    best_val_f1 = -1.0
    best_threshold = 0.5

    for epoch in range(1, int(training["epochs"]) + 1):
        total, structure, node, edge = train_epoch(
            model, train_loader, optimizer, device, **loss_args
        )
        val_scores, val_labels = compute_anomaly_scores(model, val_loader, device, **loss_args)
        threshold, val_metrics = _threshold_and_metrics(val_labels, val_scores)
        history.append(
            {
                "epoch": epoch,
                "train_total_loss": total,
                "train_structure_loss": structure,
                "train_node_loss": node,
                "train_edge_loss": edge,
                "val_f1": val_metrics["f1"],
                "val_pr_auc": val_metrics["pr_auc"],
                "val_roc_auc": val_metrics["roc_auc"],
            }
        )
        print(
            f"epoch {epoch:02d}/{int(training['epochs']):02d}  "
            f"loss={total:.4f}  val_f1={val_metrics['f1']:.4f}  "
            f"val_pr_auc={val_metrics['pr_auc']:.4f}  device={device}"
        )
        if val_metrics["f1"] >= best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_threshold = threshold
            best_state = copy.deepcopy(model.state_dict())

    if best_state is None:
        raise RuntimeError("Training did not produce a validation checkpoint.")
    model.load_state_dict(best_state)
    val_scores, val_labels = compute_anomaly_scores(model, val_loader, device, **loss_args)
    _, val_metrics = _threshold_and_metrics(val_labels, val_scores, threshold=best_threshold)
    test_scores, test_labels, test_components, test_ids = compute_anomaly_scores(
        model, test_loader, device, return_components=True, **loss_args
    )
    test_metrics = _score_metrics(test_labels, test_scores, best_threshold)
    test_predictions = (test_scores > best_threshold).astype(int)
    matrix = confusion_matrix(test_labels, test_predictions, labels=[0, 1]).tolist()
    test_metrics.update(
        {
            "precision": _rounded(precision_score(test_labels, test_predictions, zero_division=0)),
            "recall": _rounded(recall_score(test_labels, test_predictions, zero_division=0)),
            "confusion_matrix": matrix,
        }
    )
    metrics: dict[str, Any] = {
        "best_threshold": _rounded(best_threshold),
        "val_f1": val_metrics["f1"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_roc_auc": val_metrics["roc_auc"],
        "test_f1": test_metrics["f1"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_confusion_matrix": test_metrics["confusion_matrix"],
        "device": str(device),
        "n_train": len(train_graphs),
        "n_val": len(val_graphs),
        "n_test": len(test_graphs),
        "history": history,
        "history_epochs": history,
    }
    checkpoint = {
        "model_state_dict": best_state,
        "node_dim": int(node_dim),
        "edge_dim": int(edge_dim),
        "hidden_dim": int(training["hidden_dim"]),
        "latent_dim": int(training["latent_dim"]),
        "best_threshold": best_threshold,
        "training": training,
        "history": _loss_history(history),
        "edge_mean": edge_mean,
        "edge_std": edge_std,
        "gine_aggregation": GAE_ARCH["gine_aggregation"],
        "node_transformation": GAE_ARCH["node_transformation"],
        "structure_decoder": GAE_ARCH["structure_decoder"],
        "node_reconstruct": GAE_ARCH["node_reconstruct"],
        "node_recon_dim": int(model.node_recon_index.numel()),
        "node_recon_index": model.node_recon_index.detach().cpu().tolist(),
        "tfidf_dim": model.tfidf_dim,
        "sbert_dim": model.sbert_dim,
        "fusion_mode": GAE_ARCH["fusion_mode"],
        "modality_projection_dim": GAE_ARCH["modality_projection_dim"],
        "node_loss_mode": GAE_ARCH["node_loss_mode"],
        "node_block_weights": GAE_ARCH["node_block_weights"],
    }
    metrics["history"] = _loss_history(history)
    eval_payload = {
        "test_scores": test_scores,
        "test_labels": test_labels,
        "structure": test_components["structure"],
        "node": test_components["node"],
        "edge": test_components["edge"],
        "tfidf": test_components["tfidf"],
        "sbert": test_components["sbert"],
        "extras": test_components["extras"],
        "graph_ids": test_ids,
    }
    return metrics, checkpoint, eval_payload


def train_isolation_forest_bundle(graph_path: Path, *, seed: int) -> tuple[dict[str, Any], dict[str, Any]]:
    from sklearn.ensemble import IsolationForest
    from sklearn.metrics import confusion_matrix, precision_score, recall_score

    train_graphs, val_graphs, test_graphs, _ = load_graph_splits(graph_path)
    clean_train = [graph for graph in train_graphs if int(graph.y.item()) == 0]
    if not clean_train:
        raise ValueError("No normal training graphs remain for Isolation Forest.")
    x_train, x_val, x_test = window_feature_matrices(clean_train, val_graphs, test_graphs)
    model = IsolationForest(
        n_estimators=300,
        max_samples=min(256, len(x_train)),
        contamination="auto",
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(x_train)
    val_scores = -model.score_samples(x_val)
    val_labels = np.asarray([int(graph.y.item()) for graph in val_graphs])
    threshold, val_metrics = _threshold_and_metrics(val_labels, val_scores)
    test_scores = -model.score_samples(x_test)
    test_labels = np.asarray([int(graph.y.item()) for graph in test_graphs])
    test_metrics = _score_metrics(test_labels, test_scores, threshold)
    predictions = (test_scores > threshold).astype(int)
    test_metrics.update(
        {
            "precision": _rounded(precision_score(test_labels, predictions, zero_division=0)),
            "recall": _rounded(recall_score(test_labels, predictions, zero_division=0)),
            "confusion_matrix": confusion_matrix(test_labels, predictions, labels=[0, 1]).tolist(),
        }
    )
    metrics = {
        "best_threshold": _rounded(threshold),
        "val_f1": val_metrics["f1"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_roc_auc": val_metrics["roc_auc"],
        "test_f1": test_metrics["f1"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_confusion_matrix": test_metrics["confusion_matrix"],
        "n_train": len(clean_train),
        "n_val": len(val_graphs),
        "n_test": len(test_graphs),
        "baseline": "isolation_forest",
        "n_estimators": 300,
        "max_samples": min(256, len(x_train)),
    }
    payload = {
        "test_scores": test_scores,
        "test_labels": test_labels,
        "graph_ids": sequence_ids_from_graphs(test_graphs),
    }
    return metrics, payload


def run_gae_arm(arm: str) -> list[Path]:
    graph_path = stage_arm_graph(arm)
    bundle_meta = load_bundle_meta(graph_path) or {}
    training = resolved_training()
    written: list[Path] = []
    for seed in SEEDS:
        run_id = f"{_safe_name(CAMPAIGN_ID)}_{_safe_name(arm)}_seed{seed}"
        output_dir = Path(WORKSPACE_ROOT) / "outputs" / DATASET / run_id
        if (output_dir / "metrics.json").exists():
            print(f"[SKIP] {arm} seed={seed} already completed at {output_dir}")
            push_run_outputs(output_dir)
            written.append(output_dir)
            continue
        print(f"[TRAIN] {arm} seed={seed} graph={graph_path}")
        metrics, checkpoint, eval_payload = train_graph_bundle(
            graph_path, training=training, seed=seed, bundle_meta=bundle_meta
        )
        output_dir.mkdir(parents=True, exist_ok=True)
        torch.save(checkpoint, output_dir / "attribute_gae.pt")
        campaign_meta = {
            "campaign_id": CAMPAIGN_ID,
            "family": "representation",
            "experiment_name": arm,
            "run_id": run_id,
            "graph_identity": bundle_meta.get("graph_identity"),
            "split_lock_id": bundle_meta.get("split_lock_id"),
            "parent_graph": str(graph_path),
            "feature_contract": bundle_meta.get("feature_contract"),
            "seed": seed,
            "split_protocol": (bundle_meta.get("graph_identity") or {}).get("split_protocol", "stratified"),
            "oov_graph_rate": bundle_meta.get("oov_graph_rate"),
            "oov_line_rate": bundle_meta.get("oov_line_rate"),
        }
        write_eval_pack(
            output_dir,
            metrics={key: value for key, value in metrics.items() if key != "history_epochs"},
            history_epochs=metrics["history_epochs"],
            labels=eval_payload["test_labels"],
            scores=eval_payload["test_scores"],
            structure=eval_payload["structure"],
            node=eval_payload["node"],
            edge=eval_payload["edge"],
            node_blocks={name: eval_payload[name] for name in ("tfidf", "sbert", "extras")},
            threshold=float(metrics["best_threshold"]),
            alpha=float(training["alpha"]),
            beta=float(training["beta"]),
            gamma=float(training["gamma"]),
            graph_ids=eval_payload["graph_ids"],
            config=_experiment_config(arm, seed),
            campaign_meta=campaign_meta,
        )
        figures = output_dir / "figures"
        print(
            f"[DONE] {arm} seed={seed}  test_pr_auc={metrics['test_pr_auc']}  "
            f"test_f1={metrics['test_f1']}  test_roc_auc={metrics['test_roc_auc']} → {output_dir}"
        )
        print(
            "[PLOTS]",
            figures / "confusion_matrix.png",
            figures / "training_history.png",
            figures / "test_pr_curve.png",
            figures / "test_score_distribution.png",
        )
        push_run_outputs(output_dir)
        written.append(output_dir)
    return written


def run_isolation_forest() -> list[Path]:
    graph_path = stage_arm_graph("hybrid_llm")
    written: list[Path] = []
    for seed in SEEDS:
        run_id = f"{_safe_name(CAMPAIGN_ID)}_isolation_forest_seed{seed}"
        output_dir = Path(WORKSPACE_ROOT) / "outputs" / DATASET / run_id
        if (output_dir / "metrics.json").exists():
            print(f"[SKIP] isolation_forest seed={seed}")
            push_run_outputs(output_dir)
            written.append(output_dir)
            continue
        metrics, payload = train_isolation_forest_bundle(graph_path, seed=seed)
        zeros = np.zeros_like(payload["test_scores"], dtype=float)
        campaign_meta = {
            "campaign_id": CAMPAIGN_ID,
            "family": "baseline",
            "experiment_name": "isolation_forest",
            "run_id": run_id,
            "seed": seed,
            "split_lock_id": (load_bundle_meta(graph_path) or {}).get("split_lock_id"),
            "baseline_features": "train_template_counts,oov_share,log1p_window_length",
        }
        write_eval_pack(
            output_dir,
            metrics=metrics,
            history_epochs=[],
            labels=payload["test_labels"],
            scores=payload["test_scores"],
            structure=zeros,
            node=zeros,
            edge=zeros,
            threshold=float(metrics["best_threshold"]),
            alpha=1.0,
            beta=1.0,
            gamma=1.0,
            graph_ids=payload["graph_ids"],
            config=_experiment_config("isolation_forest", seed, family="baseline"),
            campaign_meta=campaign_meta,
        )
        print(
            f"[BASELINE] isolation_forest seed={seed}  "
            f"test_pr_auc={metrics['test_pr_auc']} → {output_dir}"
        )
        figures = output_dir / "figures"
        print(
            "[PLOTS]",
            figures / "confusion_matrix.png",
            figures / "test_pr_curve.png",
            figures / "test_score_distribution.png",
        )
        push_run_outputs(output_dir)
        written.append(output_dir)
    return written


## Family A arms

Each pair of cells is independently rerunnable: prepare writes `campaigns/hdfs-full-monty-v1/graphs/<arm>/`, train writes `outputs/hdfs/...`. Skip completed `metrics.json`.


### `hybrid_llm`

Baseline: TF-IDF + SBERT on LLM `embedding_text`.


In [ ]:
ARM = "hybrid_llm"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `hybrid_raw`

Same hybrid GAE; SBERT sees Drain template text (no LLM).


In [ ]:
ARM = "hybrid_raw"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `tfidf_only`

Lexical TF-IDF only.


In [ ]:
ARM = "tfidf_only"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `sbert_raw`

SBERT on Drain templates, no TF-IDF.


In [ ]:
ARM = "sbert_raw"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `sbert_llm`

SBERT on LLM paragraphs, no TF-IDF.


In [ ]:
ARM = "sbert_llm"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `no_positional_features`

hybrid_llm without node/edge positional stats.


In [ ]:
ARM = "no_positional_features"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `no_temporal_features`

hybrid_llm without edge time-delta stats.


In [ ]:
ARM = "no_temporal_features"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### `no_temporal_or_positional_features`

Primary PB3 contrast vs hybrid_llm.


In [ ]:
ARM = "no_temporal_or_positional_features"
print(f"Arm: {ARM}  prepare={RUN_PREPARE} train={RUN_TRAIN} smoke={SMOKE}")
run_prepare_arm(ARM)
if RUN_TRAIN:
    run_gae_arm(ARM)
else:
    print("[TRAIN] skipped")


### Isolation Forest

Bag-of-templates counts + OOV share + `log1p` event length on the same frozen `hybrid_llm` split.


In [ ]:
print("Isolation Forest on hybrid_llm graphs")
if RUN_PREPARE:
    run_prepare_arm("hybrid_llm")
if RUN_TRAIN:
    run_isolation_forest()
else:
    print("[TRAIN] skipped")


## Campaign comparison

Rank by test PR-AUC vs `hybrid_llm`. Chance is the HDFS positive-class rate (about 0.03), not 0.5. There is no BGL time-block bootstrap on this stratified split; paired seed CIs are the uncertainty estimate.


In [ ]:
import pandas as pd
from IPython.display import display

report_path = write_campaign_report(
    WORKSPACE_ROOT,
    dataset="hdfs",
    campaign_id=_safe_name(CAMPAIGN_ID),
    baseline_name="hybrid_llm",
)
campaign_out = Path(WORKSPACE_ROOT) / "outputs" / "hdfs" / "campaigns" / _safe_name(CAMPAIGN_ID)
print(f"Campaign report: {report_path}")

leaderboard = pd.read_csv(campaign_out / "leaderboard.csv")
summary = pd.read_csv(campaign_out / "summary_by_arm.csv")
paired = pd.read_csv(campaign_out / "paired_bootstrap_ci.csv")
print("Leaderboard (all seeds, sorted by test PR-AUC):")
display(leaderboard[["name", "seed", "test_pr_auc", "test_f1", "test_roc_auc", "val_f1"]])
print("Means ± std by arm:")
display(summary)
print("Paired seed-bootstrap deltas vs hybrid_llm:")
display(paired)

readme = campaign_out / "README.md"
if readme.exists():
    print(readme.read_text())
push_to_drive(campaign_out)


## Optional: copy leftover campaign + outputs to Drive (Colab)

Parser, enrichment, each arm, and each seed's figures are already pushed when they finish. Run this only as a catch-all if a copy failed mid-cell.


In [ ]:
if IN_COLAB:
    for name in ("campaigns", "outputs"):
        source = WORKSPACE_ROOT / name
        destination = DRIVE_ARTIFACT_ROOT / name
        if source.exists():
            destination.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=True)
            print(f"Copied {source} → {destination}")
        else:
            print(f"No {name}/ to copy.")
else:
    print(f"Local artefacts: {CAMPAIGN_DIR}")
    print(f"Local outputs:   {Path(WORKSPACE_ROOT) / 'outputs' / 'hdfs'}")
